# Automation of Systematic Mapping Studies - A Machine Learning Approach

Naale Josephat · MSc Data Science · University of Dar es Salaam

This notebook runs the whole study end to end. Every step below states what it does,
what it needs, and what it leaves behind on disk.

| Step | What it does |
|---|---|
| 1 | Imports and environment |
| 2 | Paths and settings |
| 3 | Text utilities |
| 4 | Reading the PDFs |
| 5 | Build the dataset |
| 6 | Dataset quality report |
| 7 | Document length and outliers |
| 8 | Turning the dataset into numbers |
| 9 | Grouping the papers with K-Means |
| 10 | Assigning the three mapping facets |
| 11 | Finding themes with BERTopic |
| 12 | Scoring the model |
| 13 | The validation dashboard |
| 14 | Benchmarking |
| 15 | The orchestrator |
| 16 | Entry points |
| 17 | Evaluation module |
| 18 | Figure generation |
| 19 | **Run the pipeline** |
| 20 | Hyperparameter tuning by grid search |
| 21 | **Save the model with pickle** |
| 22 | **Load the model back** |
| 23 | Evaluate and draw the dissertation figures |
| 24 | Classification reports |
| 25 | Cluster figures |
| 26 | Training history |
| 27 | Where everything ended up |
| 28 | Inspect one paper |

Run the steps in order the first time; the cells build on one another and executing one
alone on a fresh kernel will fail. Afterwards the definition steps plus the load step are
enough to reload a trained model without touching the PDFs again.

# Part A - Building the dataset

Steps 1 to 6 turn a folder of PDFs into one table. Run this phase once.

---

## Step 1 - Imports and environment

Pulls in every library the pipeline uses and records whether the two optional
pieces are present. Neither is required: without **tesseract** scanned PDFs are
skipped, and without **spaCy** nothing changes because the model is only probed here.

On Colab this also mounts Google Drive. Off Colab it quietly carries on with local paths.

In [1]:
import os
import re
import json
import joblib
import unicodedata
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, field
from collections import Counter
from datetime import datetime
import shutil

import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')          # non-interactive backend safe for Colab
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Google Drive (Colab only; skipped when running locally)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# PDF
import pymupdf

# OCR for scanned PDFs (no text layer). Optional: the pipeline still runs
# without it, it just can't recover text from image-only pages.
try:
    import pytesseract
    from PIL import Image
    OCR_AVAILABLE = True
except ImportError:
    OCR_AVAILABLE = False
    print("⚠️  pytesseract/Pillow not available – scanned PDFs will be skipped. "
          "Install with: !apt-get install -y tesseract-ocr && "
          "!pip install pytesseract pillow")

# ML
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.pairwise import cosine_similarity
import hdbscan
from umap import UMAP

# Coherence
try:
    from gensim.models import CoherenceModel
    from gensim.corpora.dictionary import Dictionary
    GENSIM_AVAILABLE = True
except ImportError:
    GENSIM_AVAILABLE = False
    print("⚠️  Gensim not available – topic coherence will be skipped.")

# spaCy (optional)
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    SPACY_AVAILABLE = True
except Exception:
    SPACY_AVAILABLE = False

# ---- names the later steps use at cell level -------------------------------
# The classes above import these inside their own methods, which keeps each
# class self-contained but leaves the names undefined for the notebook itself.
# Importing them here means any step can be re-run on its own.
import pickle
import matplotlib.ticker
from matplotlib.lines import Line2D
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples
from sklearn.preprocessing import normalize

# ---- one shared palette for every figure in this notebook -----------------
# A validated categorical set: the first three hues stay distinguishable to
# colour-blind readers even in a scatter plot, where any pair may sit together.
SURFACE, INK, INK_2 = "#fcfcfb", "#0b0b0b", "#52514e"
MUTED, GRID = "#8a8984", "#e6e5e1"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "font.size": 10.5, "text.color": INK,
    "axes.labelcolor": INK_2, "axes.edgecolor": GRID,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "grid.color": GRID, "grid.linewidth": 0.8,
})

## Step 2 - Paths and settings

Everything configurable lives here, so no later cell contains a hard-coded path.

- `PDF_FOLDER` — the folder of primary studies to read
- `OUTPUT_DIR` — where every table, figure and model file is written
- `MODEL_PKL` — the single pickle holding the trained model (step 15)
- `EMBEDDING_MODEL` — the sentence encoder; `all-mpnet-base-v2` is stronger but slower

In [2]:
# ---- paths -----------------------------------------------------------------
if IN_COLAB:
    BASE_DIR = "/content/drive/MyDrive/naale"
    PDF_FOLDER = f"{BASE_DIR}/pdf_files"
    OUTPUT_DIR = f"{BASE_DIR}/final files/output"
else:
    BASE_DIR   = "/home/naedatatz/Desktop/Notebook"
    PDF_FOLDER = os.path.join(BASE_DIR, "pdfs")
    OUTPUT_DIR = os.path.join(BASE_DIR, "output")

DATASET_CSV = os.path.join(OUTPUT_DIR, "dataset.csv")    # Part A writes this once

# Each modelling run gets its own folder so nothing is ever overwritten.
# Set NEW_RUN_FOLDER = False to write straight into output/ and replace the
# previous run instead.
# Carry the corpus text (full_text and the five body sections) into
# sms_results.csv as well as dataset.csv. Set False for a slim results table.
RESULTS_INCLUDE_FULLTEXT = True

# Exclude papers whose length the interquartile rule flags as extreme
# (a proceedings volume, or a failed extraction). The dataset on disk always
# keeps every row; this only affects what the model is trained on.
DROP_LENGTH_OUTLIERS = False

NEW_RUN_FOLDER = True
if NEW_RUN_FOLDER:
    RUN_DIR = os.path.join(OUTPUT_DIR, "runs",
                           f"run_{datetime.now():%Y%m%d_%H%M%S}")
else:
    RUN_DIR = OUTPUT_DIR

MODEL_DIR = os.path.join(RUN_DIR, "bertopic_model")   # BERTopic's own format
MODEL_PKL = os.path.join(RUN_DIR, "sms_model.pkl")    # the single-file pickle
FIGS_DIR  = os.path.join(RUN_DIR, "figs")

# ---- run settings ----------------------------------------------------------
EMBEDDING_MODEL  = "all-MiniLM-L6-v2"   # or "all-mpnet-base-v2" (768-d, slower)
TARGET_K         = None                 # None = let the silhouette choose k
MIN_CLUSTER_SIZE = None                 # None = auto-scale to the corpus size
OUTLIER_STRATEGY = "embeddings"         # reassign HDBSCAN outliers by similarity
OUTLIER_THRESHOLD = 0.4
RANDOM_STATE     = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)

# a stable "latest" pointer, so scripts and the write-up need not chase timestamps
if NEW_RUN_FOLDER:
    _latest = os.path.join(OUTPUT_DIR, "latest")
    try:
        if os.path.islink(_latest) or os.path.exists(_latest):
            os.remove(_latest)
        os.symlink(os.path.relpath(RUN_DIR, OUTPUT_DIR), _latest)
    except OSError:
        pass                      # symlinks unavailable (e.g. some Drive mounts)

print(f"PDFs   : {PDF_FOLDER}")
print(f"Dataset: {DATASET_CSV}")
print(f"Run    : {RUN_DIR}")
print(f"Encoder: {EMBEDDING_MODEL}")
print(f"OCR available : {OCR_AVAILABLE}   (needed only for scanned PDFs)")

PDFs   : /home/naedatatz/Desktop/Notebook/pdfs
Dataset: /home/naedatatz/Desktop/Notebook/output/dataset.csv
Run    : /home/naedatatz/Desktop/Notebook/output/runs/run_20260920_171507
Encoder: all-MiniLM-L6-v2
OCR available : True   (needed only for scanned PDFs)


## Step 3 - Text utilities

Three small helpers used throughout.

`clean_for_excel` strips characters Excel refuses to store. `clean_academic_text`
removes publisher boilerplate, repairs common space-loss artefacts and drops
stopwords, leaving the words that carry meaning. `create_output_directory` makes a
timestamped folder when no explicit output path is given.

In [3]:
# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def clean_for_excel(text: str) -> str:
    """Strip illegal XML characters for Excel export."""
    if not text or not isinstance(text, str):
        return ""
    cleaned = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)
    cleaned = cleaned.replace('\x00', '').replace('\uffff', '')
    return cleaned[:30000] + "... [truncated]" if len(cleaned) > 30000 else cleaned


# ----------------------------------------------------------------------------
# TEXT CLEANING — removes PDF boilerplate that otherwise forms "noise clusters"
# (download counts, copyright/permission blocks, DOIs/URLs/emails, affiliations,
#  page furniture, and space-loss artefacts like "inthisarticle"). Running
# heads / page numbers are now mostly gone before this ever runs (see
# RobustPDFExtractor), so this is a second line of defence, not the only one.
# ----------------------------------------------------------------------------
_BOILERPLATE_PATTERNS = [
    r'permission to make digital or hard copies[^.]*?\.',   # ACM permission block
    r'copies are not made or distributed[^.]*?\.',
    r'copyrights? (?:held|©|\(c\))[^.]*?\.',
    r'all rights reserved\.?',
    r'acm reference format[^.]*?\.',
    r'acm isbn[^\n]*',
    r'this work is licensed[^.]*?\.',
    r'downloads?\s*\(?\d[\d,]*\)?',                          # "downloads (1,234)"
    r'total downloads?\b',
    r'\b\d+\s*(?:weeks?|months?|year)\b',                    # "6 weeks", "12 months"
    r'https?://\S+', r'www\.\S+', r'doi:\s*\S+', r'10\.\d{4,}/\S+',  # urls / dois
    r'\S+@\S+\.\S+',                                         # emails
    r'\bfig(?:ure)?\.?\s*\d+\b', r'\btable\s*\d+\b',
    r'\bpage\s*\d+\b', r'\bpp?\.\s*\d+',
    r'\bvol\.?\s*\d+', r'\bno\.?\s*\d+', r'\bissn\s*\S+',
]
# short affiliation / metadata noise words to drop as whole tokens
_NOISE_WORDS = {
    'downloads', 'download', 'total', 'copies', 'copy', 'permission', 'acm', 'ieee',
    'springer', 'elsevier', 'doi', 'isbn', 'issn', 'copyright', 'reserved', 'licensed',
    'university', 'universities', 'institute', 'department', 'faculty', 'école', 'ecole',
    'email', 'author', 'authors', 'corresponding', 'eds', 'editor', 'editors',
    'brazil', 'sweden', 'germany', 'china', 'india', 'norway', 'finland', 'pucrs',
    'abstract', 'keywords', 'introduction', 'references', 'bibliography',
}


def clean_academic_text(text: str, remove_stopwords: bool = True) -> str:
    """Remove PDF boilerplate, space-loss artefacts and (by default) English
    stopwords, keeping the real content. Uppercase acronyms (UI, ML, IT) are
    preserved even when their lowercase form is a stopword."""
    if not text or not isinstance(text, str):
        return ""
    t = text
    # cut everything from a References/Bibliography heading onward
    t = re.split(r'\n\s*(?:references|bibliography|acknowledg(?:e?ments)?)\s*\n',
                 t, flags=re.IGNORECASE)[0]
    for pat in _BOILERPLATE_PATTERNS:
        t = re.sub(pat, ' ', t, flags=re.IGNORECASE)
    # fix common space-loss glue so it becomes stopwords instead of fake keywords
    _GLUE = [
        (r'\bin\s?this(paper|article|work|study|section|approach)\b', r'in this \1'),
        (r'\binaddition\b', 'in addition'), (r'\bofthe\b', 'of the'),
        (r'\btothe\b', 'to the'), (r'\bforthe\b', 'for the'), (r'\bonthe\b', 'on the'),
        (r'\bwepresent\b', 'we present'), (r'\bwepropose\b', 'we propose'),
        (r'\bthispaper\b', 'this paper'), (r'\bthisarticle\b', 'this article'),
        (r'\bhasbeen\b', 'has been'), (r'\bcanbe\b', 'can be'),
        (r'\bsuchas\b', 'such as'), (r'\bbasedon\b', 'based on'),
        (r'\baswell\b', 'as well'), (r'\bfromthe\b', 'from the'),
        (r'\bbehaviou?rdriven\b', 'behaviour driven'),
        (r'\bdrivendevelopment\b', 'driven development'),
        (r'\btestdriven\b', 'test driven'), (r'\buserstories\b', 'user stories'),
    ]
    for pat, rep in _GLUE:
        t = re.sub(pat, rep, t, flags=re.IGNORECASE)
    # any remaining lowercase run >18 chars with no space is an extraction artefact
    t = ' '.join('' if (len(w) > 18 and w.isalpha() and w.islower()) else w
                 for w in t.split())
    # drop standalone numbers, 1-char tokens and known noise words — but KEEP
    # short acronyms (UI, ML, AI, QA, 5G): 2-char tokens survive if they are
    # uppercase or contain a digit
    toks = []
    for w in t.split():
        lw = re.sub(r'[^A-Za-z0-9]', '', w)
        if not lw or len(lw) < 2:
            continue
        if len(lw) < 3 and not (lw.isupper() or any(ch.isdigit() for ch in lw)):
            continue
        if lw.isdigit():
            continue
        if lw.lower() in _NOISE_WORDS:
            continue
        # sklearn's stopword list quirkily contains a few content words
        # ('system', 'call', 'found'); keep the domain-relevant ones
        if (remove_stopwords and lw.lower() in ENGLISH_STOP_WORDS
                and not lw.isupper()
                and lw.lower() not in {'system', 'call', 'found'}):
            continue
        toks.append(w)
    return re.sub(r'\s+', ' ', ' '.join(toks)).strip()


def create_output_directory(base_path: str = "/content/drive/MyDrive/naale/final files/output") -> str:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = os.path.join(base_path, f"run_{timestamp}")
    os.makedirs(output_dir, exist_ok=True)
    print(f"📁 Output directory: {output_dir}")
    return output_dir

## Step 4 - Reading the PDFs

Section headings in a paper are marked by **typography**, not by wording, so this
extractor reads font size and weight rather than matching keywords. It flattens each
page into typed lines in reading order, which keeps two-column papers intact, strips
running heads and feet, scores each line for "is this a heading", and slices the body
into canonical sections.

One detail matters a great deal. In many publisher PDFs each word is its own span and
the spans carry no space characters, so joining them naively glues whole sentences
together. The join below inserts a space wherever the horizontal gap between two spans
is wide enough to be a real one. Without it roughly a third of this corpus arrives as
unreadable runs like `providethestartingpoint`, and the clustering then groups papers
by which publisher typeset them.

In [4]:
# ============================================================================
# FIX 1 – FONT-AWARE PDF EXTRACTOR (PyMuPDF)
# ----------------------------------------------------------------------------
# Regex-only section splitting fails on real papers: headings are recognised
# by TYPOGRAPHY (larger, bold, standing alone), not by wording. This reads
# font size and weight from the PDF, so "3 Study Design" is found even though
# no keyword matches, and a sentence containing the word "introduction" is
# not mistaken for a heading.
#
# Pipeline per document:
#   1. Linearise the PDF into typed lines (text + font size + bold + page),
#      in reading order — sort=True gives correct order in two-column papers.
#   2. Strip running heads/feet — any line repeating on >40% of pages.
#   3. Detect headings by a SCORE: size vs body size, bold, numbering,
#      shortness, no terminal period, title/upper case, known section wording.
#   4. Extract the title from page-1 typography, rejecting journal banners,
#      emails, dates and "Downloaded from …" lines.
#   5. Extract the abstract with 3 strategies, each REPORTED in the output
#      (abstract_strategy: heading > pre_intro > fallback) so weak rows can
#      be audited instead of silently guessed at.
#   6. Map detected headings onto canonical sections and slice the body.
# ============================================================================

_SECTION_PATTERNS: List[tuple] = [
    ("abstract",     r"abstract|summary"),
    ("introduction", r"introduction|motivation"),
    ("background",   r"background|related\s+work|literature\s+review|prior\s+work|"
                     r"state\s+of\s+the\s+art|foundations?|preliminar(?:y|ies)|"
                     r"fundamentals?|basics|terminology|"
                     r"theoretical\s+(?:background|foundations?)"),
    ("methodology",  r"(?:materials\s+and\s+)?methods?|methodology|research\s+method|"
                     r"study\s+design|experimental(?:\s+(?:setup|design))?|"
                     r"data\s+(?:and\s+methods|collection)|approach|our\s+approach|"
                     r"(?:the\s+)?proposed\s+(?:approach|method|solution|technique|"
                     r"framework|architecture|model)|solution|implementation|"
                     r"(?:system\s+)?(?:design|architecture)|"
                     r"(?:\w+\s+)?modell?ing|(?:\w+\s+)?specification|formali[sz]ation|"
                     r"(?:our|the)\s+(?:technique|framework|process|tool)|"
                     r"study\s+protocol|research\s+design"),
    ("results",      r"results\s+and\s+discussion|results?|findings|"
                     r"experiments?(?:\s+and\s+results)?|evaluation|"
                     r"empirical\s+(?:study|evaluation|results)|validation|"
                     r"case\s+stud(?:y|ies)|experimental\s+results"),
    ("discussion",   r"discussion|analysis|threats\s+to\s+validity|limitations|"
                     r"(?:internal|external|construct|conclusion)\s+validity|"
                     r"lessons\s+learn\w*|implications"),
    ("conclusion",   r"conclusions?(?:\s+and\s+(?:future\s+work|outlook))?|"
                     r"future\s+work|final\s+(?:remarks|considerations)|"
                     r"concluding\s+remarks|summary\s+and\s+(?:conclusions?|outlook)"),
    ("references",   r"references|bibliography|acknowledge?ments?|funding|"
                     r"declarations?|appendix"),
]

_NUM = r"(?:\d+(?:\.\d+)*|[IVXLC]+|[A-Z])[.)]?"
# A heading is often a compound: "Background and Related Work",
# "Discussion and Conclusion". The optional tail lets the line still match,
# and the earliest section in _SECTION_PATTERNS wins.
_TAIL = r"(?:\s*(?:and|&|/|,)\s+[A-Za-z][A-Za-z\s-]{1,34})?"
_HEADING_RX = [
    (name, re.compile(rf"^\s*(?:{_NUM}\s+)?(?:{pat}){_TAIL}\s*[:.]?\s*$",
                      re.IGNORECASE))
    for name, pat in _SECTION_PATTERNS
]
_NUMBERED_RX = re.compile(rf"^\s*{_NUM}\s+\S")

_JUNK_LINE_RX = re.compile(
    r"^\s*(?:table|figure|fig\.|listing|algorithm|eq\.?)\s*\d|"      # captions
    r"^\s*(?:given|when|then|scenario|feature)\b\s*(?:\[|$)|"        # Gherkin
    r"@|"                                         # emails
    r"https?://|www\.|doi[:\s]|10\.\d{4,}/|"      # links, DOIs
    r"^\s*(?:downloaded|licensed|authorized|permission|copyright|©|\(c\)\s)|"
    r"^\s*(?:proceedings|conference|journal|workshop|symposium|vol\.?\s*\d|"
    r"isbn|issn|acm\s+reference|arxiv:)|"
    r"all\s+rights\s+reserved|"
    r"^\s*(?:page\s*)?\d{1,3}\s*$",               # bare page numbers
    re.IGNORECASE)

_AFFILIATION_RX = re.compile(
    r"\b(?:universit|institut|department|dept\.|faculty|school\s+of|college|"
    r"laborator|research\s+cent|academy|corporation|inc\.|gmbh|ltd\.)\b",
    re.IGNORECASE)

_ABSTRACT_END_RX = re.compile(
    r"^\s*(?:keywords?|key\s+words|index\s+terms|ccs\s+concepts|"
    r"general\s+terms|categories\s+and\s+subject)\b", re.IGNORECASE)

_LIGATURES = {"ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff", "ﬃ": "ffi", "ﬄ": "ffl",
             "…": "...", "–": "-", "—": "-", "’": "'", "‘": "'",
             "“": '"', "”": '"'}


def _pdf_normalize(text: str) -> str:
    """Ligatures, de-hyphenation across line breaks, whitespace, control chars."""
    if not text:
        return ""
    for bad, good in _LIGATURES.items():
        text = text.replace(bad, good)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"-\s*\n\s*(?=[a-z])", "", text)     # inter-\nnational
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", text)
    text = re.sub(r"\s*\n\s*", " ", text)
    return re.sub(r"\s{2,}", " ", text).strip()


@dataclass
class _PdfLine:
    """One typeset line, with the typography needed for heading detection."""
    text: str
    size: float
    bold: bool
    page: int
    y: float
    x: float

    @property
    def clean(self) -> str:
        return self.text.strip()


@dataclass
class _ParsedPaper:
    title: str = ""
    title_source: str = "none"
    abstract: str = ""
    abstract_source: str = "none"
    sections: Dict[str, str] = field(default_factory=dict)
    n_pages: int = 0
    headings_found: List[str] = field(default_factory=list)
    full_text: str = ""
    error: str = ""
    ocr_pages: int = 0            # pages recovered via OCR
    native_pages: int = 0         # pages with a usable text layer already
    extraction_method: str = "native"   # native | ocr | hybrid


class RobustPDFExtractor:
    """
    Font-aware PyMuPDF extractor. Abstract extraction uses 3 strategies, tried
    in order, with the winner recorded per document as `abstract_strategy`:
      heading   – an explicit 'Abstract' heading (or IEEE 'Abstract—…' inline
                  form), terminated at Keywords/Index Terms or the next heading
      pre_intro – the span between the title block and the first detected
                  section heading, with title/author/affiliation lines removed
      fallback  – opening ~250 words of body text (always succeeds, flagged
                  as low quality so it can be audited rather than trusted)

    SCANNED PAGES: each page is checked for a usable text layer (>=
    min_chars_per_page non-space characters from PyMuPDF). Pages that fail
    this check are rasterised at ocr_dpi and OCR'd with pytesseract, PAGE BY
    PAGE — so a hybrid document (e.g. one scanned page in an otherwise
    digital paper) OCRs only the pages that need it. OCR word boxes stand in
    for font size (line height in points, scaled from pixels back through
    ocr_dpi), so heading detection and abstract extraction run unchanged on
    OCR'd text. Bold cannot be recovered from OCR, so OCR lines never get the
    heading-detection "bold" score point — numbering, size and known section
    wording still make headings findable. Each paper's `extraction_method`
    reports native / ocr / hybrid so scanned runs can be audited separately.
    """

    def __init__(self, running_head_ratio: float = 0.4,
                 max_heading_chars: int = 95,
                 fallback_words: int = 250,
                 min_abstract_chars: int = 120,
                 section_char_limit: int = 3000,
                 ocr_enabled: bool = True,
                 ocr_dpi: int = 300,
                 ocr_lang: str = 'eng',
                 min_chars_per_page: int = 25,
                 ocr_max_pages: Optional[int] = None):
        """
        ocr_enabled        : try OCR on pages with no usable text layer.
        ocr_dpi             : render resolution for OCR (300 is a good
                              accuracy/speed trade-off; raise for tiny text).
        ocr_lang            : tesseract language code(s), e.g. 'eng',
                              'eng+fra'. Needs the matching tesseract-ocr-<lang>
                              package installed.
        min_chars_per_page  : a page with fewer non-space characters than this
                              in its native text layer is treated as scanned.
        ocr_max_pages       : cap OCR to the first N pages needing it, per
                              document (None = no cap). OCR is much slower
                              than native extraction; use this to bound worst-
                              case time on a corpus with a few very long
                              scanned PDFs.
        """
        self.running_head_ratio = running_head_ratio
        self.max_heading_chars = max_heading_chars
        self.fallback_words = fallback_words
        self.min_abstract_chars = min_abstract_chars
        self.section_char_limit = section_char_limit
        self.ocr_enabled = ocr_enabled
        self.ocr_dpi = ocr_dpi
        self.ocr_lang = ocr_lang
        self.min_chars_per_page = min_chars_per_page
        self.ocr_max_pages = ocr_max_pages
        self._last_title = ""
        self._warned_no_ocr = False

    # ------------------------------------------------------------------ #
    # 1. PDF -> typed lines (native text layer, one page)
    # ------------------------------------------------------------------ #
    def _read_page_lines(self, page, pno: int) -> List[_PdfLine]:
        lines: List[_PdfLine] = []
        try:
            blocks = page.get_text("dict", sort=True)["blocks"]
        except Exception:
            blocks = page.get_text("dict")["blocks"]
        for block in blocks:
            if block.get("type") != 0:            # 0 = text, 1 = image
                continue
            for ln in block["lines"]:
                spans = [s for s in ln["spans"] if s["text"].strip()]
                if not spans:
                    continue
                # Spans carry no space characters in many publisher PDFs
                # (each word is its own span), so joining with "" glues the
                # whole line together. Insert a space wherever the horizontal
                # gap between two spans is wide enough to be a real space.
                _parts = []
                for _i, _s in enumerate(spans):
                    if _i:
                        _prev = spans[_i - 1]
                        _gap = _s["bbox"][0] - _prev["bbox"][2]
                        if (_gap > 0.20 * max(_s["size"], 1.0)
                                and not _parts[-1].endswith((" ", "-"))
                                and not _s["text"].startswith(" ")):
                            _parts.append(" ")
                    _parts.append(_s["text"])
                text = "".join(_parts)
                for bad, good in _LIGATURES.items():
                    text = text.replace(bad, good)
                biggest = max(spans, key=lambda s: s["size"])
                bold = any(("bold" in s["font"].lower()) or (s["flags"] & 2 ** 4)
                           for s in spans)
                lines.append(_PdfLine(text=text.strip(),
                                      size=round(biggest["size"], 1),
                                      bold=bold, page=pno,
                                      y=round(ln["bbox"][1], 1),
                                      x=round(ln["bbox"][0], 1)))
        return lines

    # ------------------------------------------------------------------ #
    # 1b. Scanned pages: rasterise + OCR, one page. OCR word/line boxes are
    # converted into the SAME _PdfLine shape as native text (line height in
    # points stands in for font size), so every downstream step — running-
    # head stripping, heading scoring, abstract strategies — is unaware
    # whether a given line came from the text layer or from OCR.
    # ------------------------------------------------------------------ #
    def _ocr_page_lines(self, page, pno: int) -> List[_PdfLine]:
        try:
            pix = page.get_pixmap(dpi=self.ocr_dpi, colorspace=pymupdf.csGRAY)
            img = Image.frombytes("L", (pix.width, pix.height), pix.samples)
            data = pytesseract.image_to_data(img, lang=self.ocr_lang,
                                             output_type=pytesseract.Output.DICT)
        except Exception as e:
            print(f"    ⚠️  OCR failed on page {pno + 1}: {str(e)[:80]}")
            return []

        scale = 72.0 / self.ocr_dpi     # pixels (at ocr_dpi) -> PDF points
        groups: Dict[tuple, dict] = {}
        for i in range(len(data['text'])):
            word = data['text'][i].strip()
            try:
                conf = float(data['conf'][i])
            except (TypeError, ValueError):
                conf = -1.0
            if not word or conf < 0:            # conf == -1 marks non-text boxes
                continue
            # tesseract's block/par/line numbering already reflects its own
            # layout/reading-order analysis, so grouping by it (and NOT
            # re-sorting by y) preserves reading order in multi-column scans.
            key = (data['block_num'][i], data['par_num'][i], data['line_num'][i])
            g = groups.setdefault(key, {'words': [], 'tops': [], 'bottoms': [], 'lefts': []})
            top, height, left = data['top'][i], data['height'][i], data['left'][i]
            g['words'].append(word)
            g['tops'].append(top)
            g['bottoms'].append(top + height)
            g['lefts'].append(left)

        lines: List[_PdfLine] = []
        for g in groups.values():
            text = _pdf_normalize(" ".join(g['words']))
            if not text:
                continue
            top, bottom, left = min(g['tops']), max(g['bottoms']), min(g['lefts'])
            size = max(1.0, (bottom - top) * scale)   # line height ~ font size
            lines.append(_PdfLine(text=text, size=round(size, 1), bold=False,
                                  page=pno, y=round(top * scale, 1),
                                  x=round(left * scale, 1)))
        return lines

    # ------------------------------------------------------------------ #
    # 1c. Document driver: native text where available, OCR per page where
    # it isn't. Handles fully scanned PDFs and hybrid ones (e.g. a single
    # scanned cover page in an otherwise digital paper) with the same code.
    # ------------------------------------------------------------------ #
    def _read_document_lines(self, doc) -> Tuple[List[_PdfLine], int, int]:
        lines: List[_PdfLine] = []
        ocr_pages = native_pages = 0
        for pno, page in enumerate(doc):
            native = self._read_page_lines(page, pno)
            text_len = sum(len(ln.clean.replace(" ", "")) for ln in native)
            if text_len >= self.min_chars_per_page:
                lines.extend(native)
                native_pages += 1
                continue

            # Page looks scanned/empty — try OCR if we're allowed to.
            if self.ocr_enabled and OCR_AVAILABLE and \
               (self.ocr_max_pages is None or ocr_pages < self.ocr_max_pages):
                ocr_lines = self._ocr_page_lines(page, pno)
                ocr_len = sum(len(ln.clean.replace(" ", "")) for ln in ocr_lines)
                if ocr_len > text_len:              # OCR actually found more
                    lines.extend(ocr_lines)
                    ocr_pages += 1
                    continue

            if self.ocr_enabled and not OCR_AVAILABLE and not self._warned_no_ocr:
                print("    ⚠️  page(s) look scanned but pytesseract/Pillow "
                      "aren't installed — those pages will be blank. "
                      "See the INSTALL comment at the top of this file.")
                self._warned_no_ocr = True

            lines.extend(native)          # keep whatever little native text exists
        return lines, ocr_pages, native_pages

    def _body_size(self, lines: List[_PdfLine]) -> float:
        """Modal font size weighted by characters — the size of running text."""
        weight: Counter = Counter()
        for ln in lines:
            weight[ln.size] += len(ln.clean)
        return weight.most_common(1)[0][0] if weight else 10.0

    def _strip_running_heads(self, lines: List[_PdfLine], n_pages: int) -> List[_PdfLine]:
        """Drop headers/footers: short lines repeating across many pages."""
        if n_pages < 3:
            return lines
        seen: Dict[str, set] = {}
        for ln in lines:
            key = re.sub(r"\d+", "#", ln.clean.lower())
            if 0 < len(key) <= 120:
                seen.setdefault(key, set()).add(ln.page)
        repeated = {k for k, pages in seen.items()
                    if len(pages) >= max(2, int(n_pages * self.running_head_ratio))}
        if not repeated:
            return lines
        return [ln for ln in lines
                if re.sub(r"\d+", "#", ln.clean.lower()) not in repeated]

    # ------------------------------------------------------------------ #
    # 2. Heading detection (typography + wording)
    # ------------------------------------------------------------------ #
    def _is_heading(self, ln: _PdfLine, body: float) -> bool:
        t = ln.clean
        if not t or len(t) > self.max_heading_chars:
            return False
        if _JUNK_LINE_RX.search(t):
            return False
        if t.endswith((".", ",", ";", ":")) and not _NUMBERED_RX.match(t):
            if not re.match(r"^\s*(?:\d+(?:\.\d+)*)[.)]\s*$", t):
                return False
        words = t.split()
        if len(words) > 12:
            return False

        score = 0
        if ln.size >= body + 0.6:
            score += 2
        if ln.bold:
             score += 2
        if _NUMBERED_RX.match(t):
            score += 2
        if t.isupper() and len(t) > 2:
            score += 1
        if len(words) <= 6:
            score += 1
        if any(rx.match(t) for _, rx in _HEADING_RX):
            score += 2
        return score >= 3

    def _canonical(self, text: str) -> Optional[str]:
        t = text.strip()
        for name, rx in _HEADING_RX:
            if rx.match(t):
                return name
        return None

    # ------------------------------------------------------------------ #
    # 3. Title
    # ------------------------------------------------------------------ #
    def _extract_title(self, lines: List[_PdfLine], doc) -> tuple:
        page1 = [ln for ln in lines if ln.page == 0]
        if page1:
            page_h = doc[0].rect.height
            cands = [ln for ln in page1
                     if ln.y < page_h * 0.55
                     and not _JUNK_LINE_RX.search(ln.clean)
                     and not _AFFILIATION_RX.search(ln.clean)
                     and len(ln.clean) > 3
                     and not re.match(r"^\s*abstract\b", ln.clean, re.I)]
            if cands:
                top = max(ln.size for ln in cands)
                block = [ln for ln in cands if ln.size >= top - 0.4]
                block.sort(key=lambda l: (l.y, l.x))
                title_lines, last_y = [], None
                for ln in block:
                    if last_y is not None and ln.y - last_y > ln.size * 2.5:
                        break
                    title_lines.append(ln.clean)
                    last_y = ln.y
                title = _pdf_normalize(" ".join(title_lines))
                title = re.sub(r"\s*[\*†‡§¶]+\s*$", "", title)
                if 12 <= len(title) <= 400 and len(title.split()) >= 3:
                    return title, "typography"

        meta = (doc.metadata or {}).get("title", "") or ""
        meta = _pdf_normalize(meta)
        if len(meta) > 12 and not meta.lower().endswith((".dvi", ".pdf", ".doc")):
            return meta, "metadata"

        for ln in lines[:15]:
            t = _pdf_normalize(ln.clean)
            if 20 < len(t) < 250 and not _JUNK_LINE_RX.search(t):
                return t, "first_line"
        return "", "none"

    # ------------------------------------------------------------------ #
    # 4. Abstract (3 strategies, each reported)
    # ------------------------------------------------------------------ #
    def _extract_abstract(self, lines: List[_PdfLine], heads: List[tuple],
                          body_size: float) -> tuple:
        # -- S1: an explicit Abstract heading, or an inline "Abstract—…" line --
        for i, ln in enumerate(lines):
            m = re.match(r"^\s*abstract\b[\s\-\u2013\u2014:.\u00b7\u2022|]*(.*)$",
                        ln.clean, re.IGNORECASE)
            if not m:
                continue
            chunks = [m.group(1)] if m.group(1).strip() else []
            for nxt in lines[i + 1:]:
                t = nxt.clean
                if _ABSTRACT_END_RX.match(t):
                    break
                if self._is_heading(nxt, body_size) and self._canonical(t) != "abstract":
                    break
                if _NUMBERED_RX.match(t) and len(t) < self.max_heading_chars:
                    break
                chunks.append(t)
                if sum(len(c.split()) for c in chunks) > 400:
                    break
            cand = _pdf_normalize("\n".join(chunks))
            # An explicit heading is strong evidence, so accept a shorter span
            # here than the pre-intro strategy is allowed to guess at.
            if len(cand) >= max(60, self.min_abstract_chars // 2):
                return cand, "heading"

        # -- S2: everything between the title block and the first real heading --
        first = next((idx for idx, name in heads if name != "abstract"), None)
        if first:
            title_words = set(re.findall(r"\w+", self._last_title.lower()))
            block, started = [], False
            for ln in lines[:first]:
                t = ln.clean
                if _JUNK_LINE_RX.search(t) or _AFFILIATION_RX.search(t):
                    continue
                t = re.sub(r"^\s*abstract\b[\s\-\u2013\u2014:.]*", "", t, flags=re.I)
                if not t.strip():
                    continue
                if not started:
                    words = set(re.findall(r"\w+", t.lower()))
                    overlap = len(words & title_words) / max(1, len(words))
                    if overlap > 0.6 or len(t) < 45:
                        continue                        # title/author front matter
                    started = True
                block.append(t)
            cand = _pdf_normalize("\n".join(block))
            if len(cand) >= self.min_abstract_chars:
                return cand[:4000], "pre_intro"

        # -- S3: opening of the body text, flagged as low quality --
        body = [ln.clean for ln in lines
                if not _JUNK_LINE_RX.search(ln.clean)
                and not _AFFILIATION_RX.search(ln.clean)]
        cand = _pdf_normalize("\n".join(body[3:]))
        words = cand.split()
        if words:
            return " ".join(words[:self.fallback_words]), "fallback"
        return "", "none"

    # ------------------------------------------------------------------ #
    # 5. Orchestration — public API kept identical to the old class
    # ------------------------------------------------------------------ #
    def extract_pdf(self, pdf_path: str) -> Dict[str, str]:
        """Same signature/keys as the original pdfplumber-based extractor,
        plus 'abstract_strategy' and 'title_source' for the quality columns."""
        out = {k: '' for k in
               ['title', 'abstract', 'introduction', 'background', 'methodology',
                'results', 'discussion', 'conclusion', 'references', 'full_text']}
        out['abstract_strategy'] = 'none'
        out['title_source'] = 'none'
        out['extraction_method'] = 'native'
        out['ocr_pages'] = 0

        try:
            doc = pymupdf.open(pdf_path)
        except Exception as e:
            print(f"  ⚠️  {os.path.basename(pdf_path)}: open failed ({str(e)[:60]})")
            return out

        with doc:
            if doc.needs_pass:
                print(f"  ⚠️  {os.path.basename(pdf_path)}: encrypted")
                return out

            paper = _ParsedPaper(n_pages=doc.page_count)
            lines, ocr_pages, native_pages = self._read_document_lines(doc)
            paper.ocr_pages, paper.native_pages = ocr_pages, native_pages
            paper.extraction_method = ('native' if ocr_pages == 0 else
                                       'ocr' if native_pages == 0 else 'hybrid')
            if not lines:
                reason = ("no text layer, and OCR is unavailable or found nothing"
                          if not OCR_AVAILABLE or not self.ocr_enabled
                          else "no text layer, and OCR found nothing usable")
                print(f"  ⚠️  {os.path.basename(pdf_path)}: {reason}")
                out['extraction_method'] = paper.extraction_method
                return out

            lines = self._strip_running_heads(lines, doc.page_count)
            body_size = self._body_size(lines)
            paper.title, paper.title_source = self._extract_title(lines, doc)
            self._last_title = paper.title

            heads, seen = [], set()
            for i, ln in enumerate(lines):
                if not self._is_heading(ln, body_size):
                    continue
                name = self._canonical(ln.clean)
                if name and name not in seen:
                    heads.append((i, name))
                    seen.add(name)
            paper.headings_found = [n for _, n in heads]

            paper.abstract, paper.abstract_source = self._extract_abstract(
                lines, heads, body_size)

            stop = next((i for i, n in heads if n == "references"), len(lines))
            for j, (start, name) in enumerate(heads):
                if name in ("abstract", "references"):
                    continue
                end = heads[j + 1][0] if j + 1 < len(heads) else stop
                end = min(end, stop)
                if end <= start:
                    continue
                text = _pdf_normalize("\n".join(l.clean for l in lines[start + 1:end]))
                if self.section_char_limit:
                    text = text[:self.section_char_limit]
                paper.sections[name] = text

            paper.full_text = _pdf_normalize("\n".join(l.clean for l in lines))

        out['title'] = paper.title
        out['abstract'] = paper.abstract
        out['full_text'] = paper.full_text
        # 'background' merges into introduction rather than being dropped —
        # the downstream pipeline has no separate column for it.
        for name, text in paper.sections.items():
            key = name
            if key in out:
                out[key] = (out[key] + " " + text).strip() if out[key] else text
        out['abstract_strategy'] = paper.abstract_source
        out['title_source'] = paper.title_source
        out['extraction_method'] = paper.extraction_method
        out['ocr_pages'] = paper.ocr_pages
        return out

## Step 5 - Build the dataset

This is the only step that touches the PDFs. It reads each one with the extractor
above and writes one row per paper to `dataset.csv`, including a **`full_text`
column** holding the complete document text.

Why this is a separate phase. Reading 126 PDFs is the slow part and its result never
changes unless the PDFs do. Once the dataset exists, every modelling decision below
can be re-run in seconds against the same text, and the dataset is a citable artefact
in its own right: it is the corpus, independent of any model.

| Column | What it holds |
|---|---|
| `paper_id` | short stable id, `P0001`, `P0002`, … |
| `source_file` | the original PDF file name |
| `title`, `year`, `keywords`, `abstract` | the header fields |
| `introduction`, `background`, `methodology`, `results`, `discussion`, `conclusion` | the body, sliced by heading |
| **`full_text`** | the entire document as one string |
| `word_count`, `n_sections` | size and how much structure was recovered |
| `abstract_strategy`, `title_source`, `extraction_method`, `ocr_pages` | how each field was obtained, for auditing |

Skip this step on later runs. If `dataset.csv` already exists it is reused unless you
pass `rebuild=True`.

In [5]:
SECTION_COLUMNS = ["introduction", "background", "methodology", "results",
                   "discussion", "conclusion"]

# ---- publication year and author keywords ---------------------------------
# `.` never matches a newline here, so it is used instead of [^\n] to keep
# each pattern inside a single line of the front matter.
_YEAR_RE = re.compile(r"(?:19[89]\d|20[0-2]\d)")
_YEAR_HINTS = [
    re.compile(r"(?:©|\(c\)|copyright)\s*,?\s*((?:19[89]\d|20[0-2]\d))", re.I),
    re.compile(r"published\s+(?:online\s+)?(?:in\s+)?.{0,30}?((?:19[89]\d|20[0-2]\d))", re.I),
    re.compile(r"\b(?:accepted|received|revised)\b.{0,40}?((?:19[89]\d|20[0-2]\d))", re.I),
    re.compile(r"\bdoi\b.{0,60}?((?:19[89]\d|20[0-2]\d))", re.I),
]
_KEYWORDS_RE = re.compile(
    r"(?:key\s*words?|index\s+terms|ccs\s+concepts)\s*[:—\-\.]?\s*(.{10,400}?)"
    r"(?:\n\s*\n|\b(?:1\s*\.?\s*)?introduction\b|$)",
    re.I | re.S)


def extract_year(full_text, head=4000):
    """Publication year, preferring an explicit copyright or published line.

    Falls back to the latest plausible year in the front matter, which is where
    the venue and copyright notice sit. Returns None rather than guessing when
    no year appears at all.
    """
    head_text = str(full_text)[:head]
    for rx in _YEAR_HINTS:
        m = rx.search(head_text)
        if m:
            return int(m.group(1))
    this_year = datetime.now().year
    found = [int(y) for y in _YEAR_RE.findall(head_text) if int(y) <= this_year]
    return max(found) if found else None


def extract_keywords(full_text, abstract="", max_terms=12):
    """Author keywords from a Keywords or Index Terms line, joined with ';'."""
    for source in (str(abstract), str(full_text)[:8000]):
        m = _KEYWORDS_RE.search(source)
        if not m:
            continue
        raw = re.sub(r"\s+", " ", m.group(1)).strip(" .;,")
        parts = re.split(r"[;,·•]|\s{2,}", raw)
        terms = [p.strip(" .-—") for p in parts]
        terms = [t for t in terms
                 if 2 < len(t) < 60 and not t.lower().startswith("introduction")]
        if terms:
            return "; ".join(terms[:max_terms])
    return ""


def build_dataset(pdf_folder=PDF_FOLDER, out_csv=DATASET_CSV, rebuild=False,
                  extractor=None):
    """Read every PDF once and write the corpus to a CSV, full text included."""
    if os.path.exists(out_csv) and not rebuild:
        existing = pd.read_csv(out_csv)
        print(f"\u2139\ufe0f  Reusing {out_csv} ({len(existing)} papers). "
              f"Pass rebuild=True to read the PDFs again.")
        return existing

    pdfs = sorted(f for f in os.listdir(pdf_folder) if f.lower().endswith(".pdf"))
    print(f"\U0001f4c1 Reading {len(pdfs)} PDFs from {pdf_folder}")
    extractor = extractor or RobustPDFExtractor()

    rows, failed = [], []
    for idx, name in enumerate(tqdm(pdfs, desc="Building dataset"), start=1):
        path = os.path.join(pdf_folder, name)
        try:
            s = extractor.extract_pdf(path)
        except Exception as e:                       # never lose the whole run
            failed.append((name, str(e)[:80]))
            continue
        if not s.get("full_text"):
            failed.append((name, "no text recovered"))
            continue
        row = {
            # a short stable id, as the proposal's schema uses; the file name
            # is kept in its own column so nothing is lost
            "paper_id":          f"P{idx:04d}",
            "source_file":       name,
            "title":             s.get("title", ""),
            "abstract":          s.get("abstract", ""),
            "year":              extract_year(s.get("full_text", "")),
            "keywords":          extract_keywords(s.get("full_text", ""),
                                                  s.get("abstract", "")),
            "full_text":         s.get("full_text", ""),
            "word_count":        len(s.get("full_text", "").split()),
            "abstract_strategy": s.get("abstract_strategy", "none"),
            "title_source":      s.get("title_source", "none"),
            "extraction_method": s.get("extraction_method", "native"),
            "ocr_pages":         s.get("ocr_pages", 0),
        }
        for col in SECTION_COLUMNS:
            row[col] = s.get(col, "")
        row["n_sections"] = sum(1 for c in SECTION_COLUMNS if row[c].strip())
        rows.append(row)

    df = pd.DataFrame(rows)
    order = (["paper_id", "source_file", "title", "year", "keywords", "abstract"]
             + SECTION_COLUMNS
             + ["full_text", "word_count", "n_sections", "abstract_strategy",
                "title_source", "extraction_method", "ocr_pages"])
    df = df[[c for c in order if c in df.columns]]
    df.to_csv(out_csv, index=False, encoding="utf-8")

    mb = os.path.getsize(out_csv) / 1e6
    print(f"\n\u2705 dataset \u2192 {out_csv}  ({len(df)} papers, {mb:.1f} MB)")
    if failed:
        print(f"\u26a0\ufe0f  {len(failed)} PDF(s) produced no text:")
        for name, why in failed:
            print(f"      {name[:60]:<62} {why}")
    return df


dataset = build_dataset(rebuild=True)
dataset.head(3)

📁 Reading 126 PDFs from /home/naedatatz/Desktop/Notebook/pdfs


Building dataset:  12%|██▋                    | 15/126 [00:02<00:22,  4.88it/s]

    ⚠️  OCR failed on page 15: tesseract is not installed or it's not in your PATH. See README file for more in
    ⚠️  OCR failed on page 48: tesseract is not installed or it's not in your PATH. See README file for more in
    ⚠️  OCR failed on page 132: tesseract is not installed or it's not in your PATH. See README file for more in
    ⚠️  OCR failed on page 187: tesseract is not installed or it's not in your PATH. See README file for more in
    ⚠️  OCR failed on page 265: tesseract is not installed or it's not in your PATH. See README file for more in


Building dataset:  16%|███▋                   | 20/126 [00:05<00:36,  2.91it/s]

    ⚠️  OCR failed on page 15: tesseract is not installed or it's not in your PATH. See README file for more in
    ⚠️  OCR failed on page 48: tesseract is not installed or it's not in your PATH. See README file for more in
    ⚠️  OCR failed on page 132: tesseract is not installed or it's not in your PATH. See README file for more in
    ⚠️  OCR failed on page 187: tesseract is not installed or it's not in your PATH. See README file for more in
    ⚠️  OCR failed on page 265: tesseract is not installed or it's not in your PATH. See README file for more in


...

Building dataset:  61%|██████████████         | 77/126 [00:19<00:13,  3.52it/s]

    ⚠️  OCR failed on page 1: tesseract is not installed or it's not in your PATH. See README file for more in


Building dataset: 100%|██████████████████████| 126/126 [00:27<00:00,  4.64it/s]



✅ dataset → /home/naedatatz/Desktop/Notebook/output/dataset.csv  (123 papers, 9.4 MB)
⚠️  3 PDF(s) produced no text:
      Empirical Findings on BDD Story Parsing to Support Consisten   no text recovered
      Extending behavior-driven development for assessing user int   no text recovered
      Feature-Trace- Generating Operational Profile and Supporting   no text recovered


,paper_id,source_file,title,year,keywords,abstract,introduction,background,methodology,results,discussion,conclusion,full_text,word_count,n_sections,abstract_strategy,title_source,extraction_method,ocr_pages
0,P0001,A Behavior-Based Ontology for Supporting Autom...,A Behavior-Based Ontology for Supporting Autom...,2017.0,Automated Requirements Assessment; Behavior-Dr...,Nowadays many software development frameworks ...,"In this paper, we introduce the ontological mo...",Assessing interactive systems is an activity t...,"Text Fields. Finally, the last one contains el...","Figure 10. Behavior ""chooseRefferingTo"". The o...",2) Ontology Support for Testing Web Final UIs:...,,2017 IEEE 11th International Conference on Sem...,5697,5,heading,typography,native,0
1,P0002,A Behavior-Driven Approach to Intent Specifica...,A Behavior-Driven Approach to Intent Specifica...,NaN,,One of the goals of Software-Defined Networkin...,"To inspire the design of such NBI, the Open Ne...",,,,,"J. Rexford, S. Shenker, and J. Turner, ""OpenFl...",A Behavior-Driven Approach to Intent Specifica...,5169,2,heading,typography,native,0
2,P0003,A Conceptual Metamodel to Bridging Requirement...,A Conceptual Metamodel to Bridging Requirement...,2019.0,,critical vulnerability of software projects wh...,"Therefore, by considering both the intrinsic r...",The SoPaMM metamodel borrows the MetaObject Fa...,,,,,PDF Download 3350768.3351300.pdf 18 December 2...,4082,2,heading,typography,native,0


## Step 6 - Dataset quality report

Before any modelling, check what the corpus actually contains. Everything downstream
inherits these numbers, so a weak abstract or a missing methodology section here shows
up later as a bad cluster or a wrong facet.

`abstract_strategy` is the one to watch. `heading` means the abstract was found under
its own heading and is trustworthy; `pre_intro` means it was taken from the text before
the introduction and is usually right; `fallback` means the extractor guessed, and those
rows are worth opening by hand.

In [6]:
def dataset_report(df):
    n = len(df)
    print(f"{n} papers, {df.word_count.sum():,} words total")
    print(f"  median length : {int(df.word_count.median()):,} words "
          f"(shortest {int(df.word_count.min()):,}, longest {int(df.word_count.max()):,})")
    print(f"  full_text      : {int((df.full_text.fillna('').str.len() > 0).sum())}/{n} non-empty")

    print("\n  abstract found by:")
    for k, v in df.abstract_strategy.value_counts().items():
        print(f"      {k:<12} {v:>4}  ({100*v/n:>4.1f}%)")
    print("  title found by:")
    for k, v in df.title_source.value_counts().items():
        print(f"      {k:<12} {v:>4}  ({100*v/n:>4.1f}%)")

    print("\n  body sections recovered:")
    for col in SECTION_COLUMNS:
        got = int((df[col].fillna("").str.strip() != "").sum())
        print(f"      {col:<14} {got:>4}/{n}  ({100*got/n:>4.1f}%)")
    print(f"  papers with all six  : {int((df.n_sections == len(SECTION_COLUMNS)).sum())}/{n}")

    weak = df[(df.abstract_strategy == "fallback") | (df.title.fillna("").str.len() < 10)]
    if len(weak):
        print(f"\n  \u26a0\ufe0f  {len(weak)} row(s) worth checking by hand:")
        for _, r in weak.head(8).iterrows():
            print(f"      {str(r.source_file)[:58]:<60} "
                  f"abstract={r.abstract_strategy}")
    return df.describe(include="all").T[["count", "unique", "top"]]


dataset_report(dataset)

123 papers, 1,242,356 words total
  median length : 5,169 words (shortest 785, longest 120,392)
  full_text      : 123/123 non-empty

  abstract found by:
      heading       105  (85.4%)
      pre_intro      16  (13.0%)
      fallback        2  ( 1.6%)
  title found by:
      typography    115  (93.5%)
      first_line      6  ( 4.9%)
      metadata        2  ( 1.6%)

  body sections recovered:
      introduction    110/123  (89.4%)
      background       73/123  (59.3%)
      methodology      56/123  (45.5%)
      results          58/123  (47.2%)
      discussion       42/123  (34.1%)
      conclusion       71/123  (57.7%)
  papers with all six  : 12/123

  ⚠️  2 row(s) worth checking by hand:
      Analysing Requirements Communication Using Use Case Specif   abstract=fallback
      Approach of integrating behaviour-driven development with    abstract=fallback


,count,unique,top
paper_id,123,123,P0001
source_file,123,123,A Behavior-Based Ontology for Supporting Autom...
title,123,116,Information and Software Technology
year,101.0,NaN,NaN
keywords,123,56,
abstract,123,121,User Story is a technique widely used in Agile...
introduction,123,109,
background,123,72,
methodology,123,57,
results,123,57,


## Step 7 - Document length and outliers

A model is only as good as the text it trains on, so this step checks the corpus for
documents whose length says something went wrong.

Two failure modes matter. A **very long** document is usually not one paper: it is a
whole proceedings volume, a thesis, or a PDF that swallowed its own back matter. It
contributes far more text than any single study should and drags the vocabulary and the
embeddings with it. A **very short** document usually means extraction failed part way,
leaving a title page and little else.

Outliers are flagged by the interquartile rule, the same one a boxplot draws: anything
beyond 1.5 IQR from the quartiles. The figure shows the distribution on both a linear and
a log scale, because paper lengths are heavily right-skewed and a linear axis alone hides
the bulk of the corpus.

Nothing is removed automatically. Set `DROP_LENGTH_OUTLIERS = True` in step 2 to exclude
the flagged papers from modelling; the dataset on disk always keeps every row.

In [7]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SURFACE, INK, INK_2 = "#fcfcfb", "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]


def length_outliers(df, out_dir=None, whisker=1.5):
    """Flag length outliers by the interquartile rule and plot the distribution."""
    out_dir = out_dir or FIGS_DIR
    os.makedirs(out_dir, exist_ok=True)
    w = df["word_count"].astype(float)
    q1, q3 = w.quantile(0.25), w.quantile(0.75)
    iqr = q3 - q1
    hi, lo = q3 + whisker * iqr, max(q1 - whisker * iqr, 0)

    flagged = df.assign(_w=w)
    flagged["length_outlier"] = (w > hi) | (w < lo)
    flagged["length_flag"] = np.where(w > hi, "too long",
                              np.where(w < lo, "too short", "ok"))

    n_hi = int((w > hi).sum()); n_lo = int((w < lo).sum())
    print(f"{len(df)} papers, {int(w.sum()):,} words")
    print(f"   median {int(w.median()):,}   mean {int(w.mean()):,}   "
          f"min {int(w.min()):,}   max {int(w.max()):,}")
    print(f"   Q1 {int(q1):,}   Q3 {int(q3):,}   fences [{int(lo):,}, {int(hi):,}]")
    print(f"   flagged: {n_hi} too long, {n_lo} too short")
    if n_hi:
        share = 100 * w[w > hi].sum() / w.sum()
        print(f"   the {n_hi} longest hold {share:.1f}% of all words in the corpus")
    for _, r in flagged[flagged.length_outlier].sort_values("_w", ascending=False).iterrows():
        print(f"      {r.length_flag:<9} {int(r._w):>7,} words   {str(r.source_file)[:52]}")

    # ---- figure ----------------------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(13, 8.6))
    fig.subplots_adjust(top=0.82, bottom=0.08, hspace=0.42, wspace=0.24)

    ax = axes[0, 0]
    ax.hist(w, bins=40, color=SERIES[0], edgecolor=SURFACE)
    ax.axvline(hi, color=SERIES[1], lw=2, ls="--")
    ax.annotate(f"upper fence {int(hi):,}", (hi, ax.get_ylim()[1] * 0.92),
                xytext=(8, 0), textcoords="offset points", fontsize=10,
                fontweight="bold", color=INK)
    ax.set_title("A. Word count, linear scale", loc="left", fontweight="bold")
    ax.set_xlabel("Words per paper"); ax.set_ylabel("Papers")

    ax = axes[0, 1]
    ax.hist(w, bins=np.logspace(np.log10(max(w.min(), 1)), np.log10(w.max()), 34),
            color=SERIES[2], edgecolor=SURFACE)
    ax.axvline(hi, color=SERIES[1], lw=2, ls="--")
    if n_lo or lo > 0:
        ax.axvline(max(lo, 1), color=SERIES[1], lw=2, ls=":")
    ax.set_xscale("log")
    ax.set_title("B. Word count, log scale", loc="left", fontweight="bold")
    ax.set_xlabel("Words per paper (log)"); ax.set_ylabel("Papers")

    ax = axes[1, 0]
    bp = ax.boxplot(w, vert=False, widths=0.5, patch_artist=True,
                    flierprops=dict(marker="o", markersize=7,
                                    markerfacecolor=SERIES[1],
                                    markeredgecolor=SURFACE))
    bp["boxes"][0].set_facecolor(SERIES[0]); bp["boxes"][0].set_alpha(0.65)
    for part in ("whiskers", "caps", "medians"):
        for ln in bp[part]:
            ln.set_color(INK_2)
    ax.set_title(f"C. Boxplot ({n_hi + n_lo} outliers marked)", loc="left",
                 fontweight="bold")
    ax.set_xlabel("Words per paper"); ax.set_yticks([])

    ax = axes[1, 1]
    order = np.sort(w)[::-1]
    ax.plot(np.arange(1, len(order) + 1), 100 * np.cumsum(order) / order.sum(),
            color=SERIES[0], lw=2.4)
    if n_hi:
        ax.axvline(n_hi, color=SERIES[1], lw=2, ls="--")
        ax.annotate(f"{n_hi} longest = "
                    f"{100 * order[:n_hi].sum() / order.sum():.0f}% of the corpus",
                    (n_hi, 50), xytext=(10, 0), textcoords="offset points",
                    fontsize=10, fontweight="bold", color=INK)
    ax.set_title("D. Concentration of text", loc="left", fontweight="bold")
    ax.set_xlabel("Papers, longest first"); ax.set_ylabel("Cumulative % of all words")

    fig.text(0.055, 0.965, "Document length and outliers", fontsize=17,
             fontweight="bold", color=INK, va="top")
    fig.text(0.055, 0.918,
             f"{len(df)} papers. {n_hi} run longer and {n_lo} shorter than the "
             f"interquartile rule allows.", fontsize=10.5, color=INK_2, va="top")
    fig.text(0.055, 0.890,
             "A very long document is usually a proceedings volume rather than one study; "
             "a very short one is usually a failed extraction.",
             fontsize=10.5, color=INK_2, va="top")

    fig.savefig(os.path.join(out_dir, "fig_length_outliers.png"), dpi=200)
    plt.show()

    cols = [c for c in ("paper_id", "source_file", "word_count", "n_sections",
                        "abstract_strategy", "length_flag") if c in flagged.columns]
    flagged.loc[flagged.length_outlier, cols].to_csv(
        os.path.join(out_dir, "length_outliers.csv"), index=False)
    return flagged.drop(columns="_w")


dataset = length_outliers(dataset)

if DROP_LENGTH_OUTLIERS:
    _before = len(dataset)
    dataset_model = dataset[~dataset.length_outlier].reset_index(drop=True)
    print(f"\n\u2702\ufe0f  excluded {_before - len(dataset_model)} outlier(s); "
          f"modelling on {len(dataset_model)} papers")
else:
    dataset_model = dataset
    print(f"\n\u2139\ufe0f  keeping all {len(dataset_model)} papers "
          f"(set DROP_LENGTH_OUTLIERS = True to exclude the flagged ones)")

123 papers, 1,242,356 words
   median 5,169   mean 10,100   min 785   max 120,392
   Q1 3,557   Q3 8,559   fences [0, 16,062]
   flagged: 10 too long, 0 too short
   the 10 longest hold 48.1% of all words in the corpus
      too long  120,392 words   Automated Acceptance Tests as Software Requirements-
      too long  120,392 words   Agile Processes in Software Engineering and Extreme 
      too long  116,264 words   Implementing behavior driven development in an open 
      too long  103,259 words   Characterising the Quality of Behaviour Driven Devel
      too long   48,186 words   Executable requirements in a safety-critical context
      too long   19,222 words   Using acceptance tests to predict files changed by p
      too long   18,966 words   Adapting Behavior Driven Development (BDD) for large
      too long   17,624 words   A Multi-Case Study of Agile Requirements Engineering
      too long   16,778 words   A User_Centered Behavioral Software Development Mode
      too long  

# Part B - Modelling

Part B takes the dataset and never opens a PDF again, so the whole phase re-runs in
under a minute while you tune it.

---

## Step 8 - Turning the dataset into numbers

Before anything can be clustered the text has to become vectors, and that choice matters
more than the clustering algorithm that follows. Four representations are built from the
same dataset text and compared head to head. These are the four that survived a wider
comparison; binary counts and raw unigram counts were dropped because they scored below
every one of them.

| Representation | What it is |
|---|---|
| Bag of words, 1-2 grams | raw counts of words and word pairs |
| TF-IDF | the same counts, weighted down for terms common across the corpus |
| TF-IDF then LSA | TF-IDF compressed to 100 dense dimensions by truncated SVD |
| Sentence-BERT | a transformer embedding of the whole passage |

**How they are judged.** No labels exist at this point in the pipeline, so all three
criteria are label-free. Silhouette measures separation. Davies-Bouldin measures
compactness against separation and is better when lower. C_v coherence asks whether the
words that characterise each cluster actually co-occur in the corpus, which is the one
that tracks whether a human could name the cluster.

The winner becomes `MODEL_VECTORS` and everything downstream uses it.

In [8]:
import os

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.preprocessing import normalize


def build_representations(texts, sbert_model=EMBEDDING_MODEL):
    """The four representations that beat the rest, all L2-normalised."""
    reps = {}
    bow = CountVectorizer(min_df=2, max_features=5000, stop_words="english",
                          ngram_range=(1, 2))
    reps["Bag of words (1-2 grams)"] = normalize(
        bow.fit_transform(texts).toarray().astype(float))

    tf = TfidfVectorizer(min_df=2, max_features=5000, stop_words="english")
    X_tf = tf.fit_transform(texts).toarray()
    reps["TF-IDF"] = normalize(X_tf)
    reps["TF-IDF + LSA 100"] = normalize(
        TruncatedSVD(100, random_state=RANDOM_STATE).fit_transform(X_tf))

    reps[f"Sentence-BERT ({sbert_model})"] = normalize(
        np.asarray(SentenceTransformer(sbert_model).encode(
            texts, show_progress_bar=False)))
    return reps


def cluster_coherence(texts, labels, topn=8):
    """C_v over each cluster's most distinctive terms; NaN if gensim is absent."""
    if not GENSIM_AVAILABLE:
        return float("nan")
    vec = CountVectorizer(min_df=2, stop_words="english", max_features=5000)
    M = vec.fit_transform(texts)
    terms = np.array(vec.get_feature_names_out())
    overall = np.asarray(M.mean(axis=0)).ravel()
    topics = []
    for c in sorted(set(labels)):
        rows = np.where(labels == c)[0]
        if len(rows) < 2:
            continue
        lift = np.asarray(M[rows].mean(axis=0)).ravel() - overall
        words = [w for w in terms[np.argsort(-lift)[:topn]] if " " not in w]
        if len(words) >= 3:
            topics.append(words)
    if len(topics) < 2:
        return float("nan")
    tok = [t.lower().split() for t in texts]
    d = Dictionary(tok)
    try:
        return float(CoherenceModel(topics=topics, texts=tok, dictionary=d,
                                    coherence="c_v").get_coherence())
    except Exception:
        return float("nan")


def compare_representations(texts, k=4, out_dir=None):
    out_dir = out_dir or RUN_DIR
    reps = build_representations(texts)
    rows = []
    for name, X in reps.items():
        lab = KMeans(k, random_state=RANDOM_STATE, n_init=10).fit_predict(X)
        rows.append({
            "representation": name, "dims": X.shape[1],
            "silhouette": round(float(silhouette_score(X, lab)), 4),
            "davies_bouldin": round(float(davies_bouldin_score(X, lab)), 4),
            "coherence_cv": round(cluster_coherence(texts, lab), 4),
        })
        print(f"   {name:<34} dims={X.shape[1]:>5}  "
              f"sil={rows[-1]['silhouette']:.4f}  "
              f"DB={rows[-1]['davies_bouldin']:.3f}  "
              f"C_v={rows[-1]['coherence_cv']:.4f}")

    tab = pd.DataFrame(rows)
    # rank: coherence first, separation second, compactness as the tie-break
    r = tab.copy()
    for col, better in (("coherence_cv", 1), ("silhouette", 1), ("davies_bouldin", -1)):
        v = r[col].astype(float)
        span = v.max() - v.min() + 1e-9
        r[col + "_n"] = ((v - v.min()) / span) if better > 0 else (1 - (v - v.min()) / span)
    tab["score"] = (0.45 * r.coherence_cv_n + 0.35 * r.silhouette_n
                    + 0.20 * r.davies_bouldin_n).round(4)
    tab = tab.sort_values("score", ascending=False).reset_index(drop=True)
    tab.to_csv(os.path.join(out_dir, "representation_comparison.csv"), index=False)

    best = tab.iloc[0]["representation"]
    print(f"\n   \U0001f3c6 best: {best}")
    return tab, reps, best


_model_texts = [clean_academic_text(t) or t for t in
                (dataset_model["title"].fillna("") + ". "
                 + dataset_model["abstract"].fillna("")).tolist()]

_rep_table, _reps, _best_rep = compare_representations(_model_texts)
MODEL_VECTORS = _reps[_best_rep]
print(f"   MODEL_VECTORS shape: {MODEL_VECTORS.shape}")
_rep_table

Loading weights: 100%|████████████████████| 103/103 [00:00<00:00, 13088.93it/s]


   Bag of words (1-2 grams)           dims= 2534  sil=0.0223  DB=4.525  C_v=0.4461
   TF-IDF                             dims= 1388  sil=0.0062  DB=6.397  C_v=0.4345
   TF-IDF + LSA 100                   dims=  100  sil=0.0087  DB=5.934  C_v=0.4220
   Sentence-BERT (all-MiniLM-L6-v2)   dims=  384  sil=0.0352  DB=3.386  C_v=0.3950

   🏆 best: Bag of words (1-2 grams)
   MODEL_VECTORS shape: (123, 2534)


,representation,dims,silhouette,davies_bouldin,coherence_cv,score
0,Bag of words (1-2 grams),2534,0.0223,4.5251,0.4461,0.7687
1,Sentence-BERT (all-MiniLM-L6-v2),384,0.0352,3.3862,0.3950,0.5500
2,TF-IDF,1388,0.0062,6.3972,0.4345,0.3478
3,TF-IDF + LSA 100,100,0.0087,5.9336,0.4220,0.2987


## Step 9 - Grouping the papers with K-Means

Fully unsupervised: no keyword rules and no labels. Documents are encoded with
sentence-BERT, `k` is chosen by maximising the silhouette across a sweep, and each
group is named by the terms that are far more frequent inside it than in the corpus
overall.

Two controls are worth knowing. `target_k` forces an exact number of groups, and
`min_clusters` sets a floor, for when the silhouette collapses to `k=2` on a corpus
that is topically uniform.

The fitted `KMeans` object is kept on the classifier so a reloaded model can place
new papers without refitting.

In [9]:
# ============================================================================
# K-MEANS CLUSTER EXTRACTOR  (FULLY UNSUPERVISED — no rules, no labels, no
#                             category prototypes)
# ----------------------------------------------------------------------------
# The corpus is grouped with NO injected knowledge whatsoever:
#   1. every document is embedded with Sentence-BERT (semantic), or with TF-IDF
#      when use_tfidf=True (the lexical baseline); vectors are L2-normalised;
#   2. K-Means partitions the corpus into k clusters, where k is chosen
#      DATA-DRIVENLY by maximising the silhouette score over a range of k;
#   3. each cluster is NAMED FROM THE DATA — by its own most distinctive TF-IDF
#      keywords (the same idea BERTopic uses for themes), not by any predefined
#      category description.
# The output is therefore anonymous, data-derived groups. It does NOT emit the
# named SMS facet categories (Solution Proposal, Tool, Case Study, …): assigning
# those names is a human step — use cluster_summary() to inspect each cluster's
# keywords and map clusters to categories yourself if you need the named facets.
# This is a batch operation: fit_predict is called once on the whole corpus.
# ============================================================================

class KMeansFacetClassifier:

    # tokens that are PDF-footer / citation / metadata noise, not content
    NOISE_TERMS = {'downloads', 'total downloads', 'total', 'eds', 'purpose',
                   'university', 'edu', 'copies', 'acm', 'ieee', 'doi', 'pp',
                   'vol', 'permission', 'reserved', 'copyright', 'springer',
                   'et al', 'fig', 'table', 'author', 'authors', 'email'}

    def __init__(self, embedding_model=None, use_tfidf: bool = False,
                 k_min: int = 2, k_max: int = 15, top_keywords: int = 5,
                 target_k: int = None, min_clusters: int = 2, random_state: int = 42):
        """
        embedding_model : a SentenceTransformer to reuse (avoids a second load).
        use_tfidf       : cluster on TF-IDF (lexical baseline) instead of SBERT.
        k_min, k_max    : silhouette is maximised over k in [k_min, k_max].
        target_k        : if set, use exactly this many clusters (overrides the
                          silhouette pick) — use it to force a diverse, granular
                          clustering when silhouette collapses to k=2 on a
                          homogeneous corpus.
        min_clusters    : lower floor for the silhouette search, so it cannot
                          collapse below this many clusters even if k=2 scores
                          highest (a milder diversity control than target_k).
        top_keywords    : how many DISTINCTIVE keywords to name each cluster.
        """
        self.embedding_model = embedding_model
        self.use_tfidf = use_tfidf
        self.k_min = k_min
        self.k_max = k_max
        self.target_k = target_k
        self.min_clusters = min_clusters
        self.top_keywords = top_keywords
        self.rs = random_state
        self.chosen_k_ = {}             # {'k':, 'silhouette':}
        self.silhouette_scores_ = {}    # {k: silhouette} for the diagram
        self.cluster_keywords_ = {}     # cluster_id -> "kw1, kw2, ..."
        self.labels_ = None             # per-document cluster id

    # ------------------------------------------------------------------ #
    def _get_embedder(self):
        if self.embedding_model is None:
            from sentence_transformers import SentenceTransformer
            self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        return self.embedding_model

    def _embed_docs(self, texts):
        from sklearn.preprocessing import normalize
        if self.use_tfidf:
            from sklearn.feature_extraction.text import TfidfVectorizer
            vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2,
                                  max_features=5000, stop_words='english')
            vecs = vec.fit_transform(texts).toarray()
        else:
            vecs = np.asarray(self._get_embedder().encode(list(texts), show_progress_bar=False))
        return normalize(vecs)

    # ------------------------------------------------------------------ #
    def _best_kmeans(self, X):
        """Record the silhouette for every k and choose the number of clusters.
        If target_k is set, use it. Otherwise maximise silhouette over
        [max(k_min, min_clusters), k_max]. Returns (k, silhouette, kmeans)."""
        from sklearn.cluster import KMeans
        from sklearn.metrics import silhouette_score
        n = X.shape[0]
        k_hi = max(2, min(self.k_max, n - 1))
        # if a target_k beyond k_max was requested, extend the sweep to reach it
        if self.target_k:
            if self.target_k > n - 1:
                print(f"      ⚠️ target_k={self.target_k} > n_docs-1; "
                      f"using silhouette choice instead")
            else:
                k_hi = max(k_hi, self.target_k)
        self.silhouette_scores_ = {}
        fitted = {}
        for k in range(2, k_hi + 1):          # sweep the full range for the diagram
            km = KMeans(n_clusters=k, random_state=self.rs, n_init=10).fit(X)
            if len(set(km.labels_)) < 2:
                continue
            try:
                self.silhouette_scores_[k] = float(silhouette_score(X, km.labels_))
                fitted[k] = km
            except Exception:
                continue
        if not fitted:
            km = KMeans(n_clusters=2, random_state=self.rs, n_init=10).fit(X)
            return (2, 0.0, km)
        if self.target_k and self.target_k in fitted:      # forced diverse k
            k = self.target_k
        else:                                              # silhouette pick (with floor)
            floor = max(2, self.k_min, self.min_clusters)
            cand = {k: s for k, s in self.silhouette_scores_.items() if k >= floor} \
                   or self.silhouette_scores_
            k = max(cand, key=cand.get)
        return (k, self.silhouette_scores_[k], fitted[k])

    # ------------------------------------------------------------------ #
    def _cluster_keywords(self, texts, labels):
        """Name each cluster by its most DISTINCTIVE keywords (c-TF-IDF style):
        terms scored by how much more frequent they are in the cluster than in
        the corpus overall, with numeric / footer / citation noise removed. This
        makes cluster names diverse instead of all showing the same corpus-common
        words (development, bdd, software, …)."""
        from sklearn.feature_extraction.text import TfidfVectorizer
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2,
                              max_features=5000, stop_words='english')
        M = vec.fit_transform(texts)
        terms = np.array(vec.get_feature_names_out())
        # keep only content terms: letters, length 3-18, not a number, not a
        # glued space-loss artefact, and not a footer/citation noise word
        def _ok(t):
            if not re.search('[a-z]', t) or len(t) < 2:
                return False
            if t.replace(' ', '').isdigit():
                return False
            if t in self.NOISE_TERMS:
                return False
            for w in t.split():                      # drop glued tokens & noise words
                if len(w) > 18 or w in _NOISE_WORDS:
                    return False
            return True
        keep = np.array([_ok(t) for t in terms])
        overall = np.asarray(M.mean(axis=0)).ravel()
        out = {}
        for c in sorted(set(labels)):
            cl = np.asarray(M[labels == c].mean(axis=0)).ravel()
            score = (cl - overall) * keep          # distinctiveness
            top = terms[score.argsort()[::-1][:self.top_keywords]] if terms.size else []
            out[int(c)] = ", ".join(top)
        return out

    # ------------------------------------------------------------------ #
    def plot_silhouette(self, out_path):
        """Save the silhouette-vs-k diagram recorded during clustering."""
        if not self.silhouette_scores_:
            return
        import os
        import matplotlib
        matplotlib.use('Agg'); import matplotlib.pyplot as plt
        os.makedirs(os.path.dirname(out_path) or '.', exist_ok=True)
        ks = sorted(self.silhouette_scores_)
        sc = [self.silhouette_scores_[k] for k in ks]
        chosen = self.chosen_k_.get('k')
        fig, ax = plt.subplots(figsize=(6.6, 3.4))
        ax.plot(ks, sc, '-o', color='#2E5A88', ms=4)
        if chosen in self.silhouette_scores_:
            ax.scatter([chosen], [self.silhouette_scores_[chosen]], color='#C0504D',
                       zorder=5, label=f'chosen k={chosen} ({self.silhouette_scores_[chosen]:.3f})')
            ax.legend(fontsize=8, frameon=False)
        ax.set_xlabel('Number of clusters (k)'); ax.set_ylabel('Silhouette score')
        ax.set_title('Silhouette score vs number of clusters', fontsize=10, weight='bold')
        ax.set_ylim(0, max(0.1, max(sc) * 1.15))
        for s in ('top', 'right'):
            ax.spines[s].set_visible(False)
        fig.tight_layout(); fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f"      · silhouette diagram → {out_path}")

    # ------------------------------------------------------------------ #
    def fit_predict(self, texts) -> Dict[str, list]:
        """Group the corpus (k chosen by silhouette) and name each group from its
        own keywords. Returns {'cluster_id': [...], 'cluster_label': [...]}
        aligned with `texts`. No SMS category names are produced."""
        texts = [str(t) if t else "" for t in texts]
        doc_vecs = self._embed_docs(texts)
        self.doc_vecs_ = doc_vecs            # stored for algorithm comparison
        k, sil, km = self._best_kmeans(doc_vecs)
        self.chosen_k_ = {'k': int(k), 'silhouette': round(float(sil), 4)}
        self.kmeans_ = km          # kept so a reloaded model can predict
        self.labels_ = km.labels_
        self.cluster_keywords_ = self._cluster_keywords(texts, km.labels_)
        print(f"      · silhouette chose k={k} clusters (score={sil:.3f})")
        for c in sorted(self.cluster_keywords_):
            n = int((km.labels_ == c).sum())
            print(f"        cluster {c:>2} (n={n:>3}): {self.cluster_keywords_[c]}")
        cluster_id = [int(c) for c in km.labels_]
        cluster_label = [f"Cluster {c}: {self.cluster_keywords_[c]}" for c in km.labels_]
        return {'cluster_id': cluster_id, 'cluster_label': cluster_label}

    # ------------------------------------------------------------------ #
    def cluster_summary(self):
        """Return a DataFrame (cluster_id, size, keywords) for manual naming."""
        if self.labels_ is None:
            return pd.DataFrame()
        rows = [{'cluster_id': c,
                 'size': int((self.labels_ == c).sum()),
                 'keywords': self.cluster_keywords_[c]}
                for c in sorted(self.cluster_keywords_)]
        return pd.DataFrame(rows).sort_values('size', ascending=False).reset_index(drop=True)

## Step 10 - Assigning the three mapping facets

A systematic map classifies every study along three facets. This step assigns them
without a single hand-written keyword rule and without any labelled training data.

| Facet | Taxonomy | Judged on |
|---|---|---|
| Research type | Wieringa et al. (2006) | title and abstract |
| Contribution type | Petersen et al. (2008) | title, abstract, introduction |
| Evaluation method | Petersen et al. (2008) | abstract, methodology, results |

**How it works.** Each category is described in three natural-language anchor
sentences drawn from its published definition. Both the anchors and the papers are
embedded with the same sentence-BERT encoder used for clustering, and a paper scores
against a category by its best-matching anchor.

**Why the scores are standardised.** Raw cosine similarity is biased: some anchors sit
generically closer to academic prose than others, so one category swallows the corpus.
Unstandardised, "Tool" took 48% of the papers and "Solution Proposal" took one. Scoring
each category in units of its own standard deviation across the corpus removes that
bias. Measured against the published distribution the error falls from 0.133 to 0.044
for contribution type, and every category gets used.

**These labels are model-assigned, not verified.** Treat them as a first pass that makes
the map drawable, then correct a sample in `gold_labels_template.csv` and pass it back as
`gold_csv` in the evaluation step for a true F1. Each facet also carries a confidence and a margin,
and rows where the top two categories are within 0.25 of a standard deviation are flagged
in `<facet>_needs_review` — those are the ones to check first.

In [10]:
# --- the published taxonomies, one anchor set per category ------------------
RESEARCH_TYPE = {                                    # Wieringa et al. (2006)
    "Solution Proposal": [
        "We propose a novel solution to the problem and argue for its benefits.",
        "A new approach is presented and demonstrated with a small worked example.",
        "This paper introduces a technique, illustrated by a proof of concept.",
    ],
    "Validation Research": [
        "The technique is novel and has not yet been implemented in practice.",
        "We validate the proposed approach through controlled experiments in the laboratory.",
        "The method is evaluated using simulation and prototyping before industrial adoption.",
    ],
    "Evaluation Research": [
        "The technique is implemented in practice and evaluated in a real industrial setting.",
        "We report an empirical evaluation of the approach as used by practitioners in industry.",
        "A field study investigates how the method performs in a real organisation.",
    ],
    "Experience Papers": [
        "We report lessons learned from our own experience of applying the method in practice.",
        "This experience report describes what happened when the approach was adopted in a company.",
        "The paper recounts practical experience and the problems encountered in a real project.",
    ],
    "Philosophical Papers": [
        "We propose a new conceptual framework and a taxonomy for structuring the field.",
        "This paper offers a new way of looking at existing concepts and their relationships.",
        "We present an ontology and a conceptual model of the domain.",
    ],
    "Opinion Papers": [
        "The authors express a personal opinion about what is good or bad practice.",
        "This position paper argues a viewpoint without using a research methodology.",
        "We reflect critically on current practice and argue how it ought to change.",
    ],
}

CONTRIBUTION_TYPE = {                                # Petersen et al. (2008)
    "Process": [
        "We describe a development process and its lifecycle phases.",
        "The contribution is a process model with defined roles and activities.",
        "A framework organises the techniques into a coherent workflow for the team.",
    ],
    "Tool": [
        "We implemented a software tool that automates the approach.",
        "The contribution is a prototype tool with a user interface and its architecture.",
        "A plugin and its implementation are made available to practitioners.",
    ],
    "Empirical Insights": [
        "We report empirical findings about how the practice is used and what it costs.",
        "The study contributes evidence, benefits and challenges observed in practice.",
        "We distil guidelines and recommendations from the observed data.",
    ],
    "Method": [
        "We contribute a method for carrying out the task.",
        "A new technique and its algorithm are described step by step.",
        "The paper presents an approach consisting of a sequence of analysis steps.",
    ],
    "Model": [
        "We contribute a model that represents the structure of the domain.",
        "A formal model and its notation are defined.",
        "The paper presents a metamodel and an abstract representation of the concepts.",
    ],
    "Metric": [
        "We define a set of metrics and measures for assessing quality.",
        "The contribution is a measurement scheme with quantitative indicators.",
        "We propose an index for quantifying the property of interest.",
    ],
}

EVALUATION_METHOD = {                                # Petersen et al. (2008)
    "Example": [
        "The approach is illustrated with a running example.",
        "We demonstrate feasibility on a small illustrative scenario as a proof of concept.",
        "A toy example shows how the technique is applied in principle.",
    ],
    "Case Study": [
        "We carried out a case study in a single organisation over several months.",
        "The approach was evaluated through an industrial case study with real projects.",
        "A multiple case study examines the phenomenon in its real-life context.",
    ],
    "Experiment": [
        "We conducted a controlled experiment with participants assigned to treatments.",
        "The evaluation compares experimental and control groups and reports significance.",
        "A quasi-experiment measures the dependent variables under varying conditions.",
    ],
    "Experience Report": [
        "We surveyed practitioners using a questionnaire and analysed the responses.",
        "Semi-structured interviews were conducted with participants from several companies.",
        "The authors report their own experience of using the approach on a real project.",
    ],
    "Discussion": [
        "No empirical evaluation of the proposal is reported in this paper.",
        "The paper is conceptual and discusses the idea without measuring it.",
        "Validation is left for future work.",
    ],
    "Rigorous Analysis": [
        "We evaluate the technique on a benchmark dataset and report accuracy and performance.",
        "Simulation experiments measure execution time and scalability.",
        "The method is assessed quantitatively against baseline algorithms.",
    ],
}

FACET_TAXONOMIES = {
    "research_type":     RESEARCH_TYPE,
    "contribution_type": CONTRIBUTION_TYPE,
    "evaluation_method": EVALUATION_METHOD,
}

REVIEW_MARGIN = 0.25      # top two within this many SDs -> flag for review


class ZeroShotFacetClassifier:
    """Assign a taxonomy category by nearest anchor sentence in embedding space.

    Scores are standardised per category across the corpus, which removes the
    bias that otherwise lets one category swallow everything. `margin` is the
    gap to the runner-up, in standard deviations; small margins are coin flips.
    """

    def __init__(self, taxonomy, encoder):
        self.labels, anchors, owner = list(taxonomy), [], []
        for i, label in enumerate(self.labels):
            for a in taxonomy[label]:
                anchors.append(a)
                owner.append(i)
        self.owner = np.asarray(owner)
        self.encoder = encoder
        self.anchor_vecs = normalize(
            np.asarray(encoder.encode(anchors, show_progress_bar=False)))

    def predict(self, texts):
        vecs = normalize(np.asarray(
            self.encoder.encode([str(t) for t in texts], show_progress_bar=False)))
        sim = vecs @ self.anchor_vecs.T
        cat = np.stack([sim[:, self.owner == i].max(axis=1)
                        for i in range(len(self.labels))], axis=1)
        z = (cat - cat.mean(axis=0, keepdims=True)) / (cat.std(axis=0, keepdims=True) + 1e-9)
        order = np.argsort(-z, axis=1)
        rows = np.arange(len(texts))
        best, second = order[:, 0], order[:, 1]
        margin = z[rows, best] - z[rows, second]
        return pd.DataFrame({
            "label":      [self.labels[i] for i in best],
            "confidence": cat[rows, best].round(4),
            "margin":     margin.round(4),
            "runner_up":  [self.labels[i] for i in second],
            "needs_review": margin < REVIEW_MARGIN,
        })


def assign_facets(results, facet_texts, encoder):
    """Write the three facet columns onto `results` (a list of row dicts)."""
    print()
    print("🏷️  Assigning mapping facets (zero-shot, no labels)…")
    report = {}
    for facet, taxonomy in FACET_TAXONOMIES.items():
        texts = [ft[facet] for ft in facet_texts]
        out = ZeroShotFacetClassifier(taxonomy, encoder).predict(texts)
        for i, row in out.iterrows():
            results[i][facet] = row["label"]
            results[i][f"{facet}_confidence"] = float(row["confidence"])
            results[i][f"{facet}_margin"] = float(row["margin"])
            results[i][f"{facet}_needs_review"] = bool(row["needs_review"])
        counts = out["label"].value_counts()
        flagged = int(out["needs_review"].sum())
        report[facet] = {"counts": counts.to_dict(), "needs_review": flagged}
        print(f"   {facet}:")
        for lab, n in counts.items():
            print(f"      {lab:<22} {n:>4}  ({100 * n / len(out):>4.1f}%)")
        print(f"      → {flagged}/{len(out)} flagged for review "
              f"(margin < {REVIEW_MARGIN} SD)")
    return report

## Step 11 - Finding themes with BERTopic

The primary method. Sentence-BERT embeddings are reduced with UMAP, clustered with
HDBSCAN, and each theme is named by class-based TF-IDF refined with KeyBERT.

HDBSCAN parameters auto-scale to the corpus size, so a set of 123 papers does not
inherit defaults tuned for tens of thousands. Papers HDBSCAN leaves as outliers are
reassigned to their most similar theme when the cosine similarity clears
`outlier_threshold`; anything below stays an outlier.

In [11]:
# ============================================================================
# FIX 2 – BERTopic EXTRACTOR WITH AUTO-SCALING PARAMS
# min_cluster_size and n_neighbors now scale with corpus size so the model
# does not collapse to 3 topics on 25 docs or crash on tiny corpora.
# Embeddings are stored for reuse in evaluation (Fix 3).
# ============================================================================

class BERTopicThemeExtractor:

    def __init__(self,
                 embedding_model: str = 'all-MiniLM-L6-v2',
                 min_cluster_size: int = None,   # None = auto-scale
                 min_samples: int = None,
                 n_neighbors: int = None,
                 n_components: int = 5,
                 reduce_outliers_strategy: str = 'embeddings',  # None disables
                 outlier_threshold: float = 0.4):

        self.embedding_model_name = embedding_model
        self._user_min_cluster_size = min_cluster_size
        self._user_min_samples = min_samples
        self._user_n_neighbors = n_neighbors
        self.n_components = n_components
        self.reduce_outliers_strategy = reduce_outliers_strategy
        self.outlier_threshold = outlier_threshold

        self.embedding_model = SentenceTransformer(embedding_model)
        self.is_fitted = False
        self.topics = None
        self.probabilities = None
        self.documents = None
        self.topic_info = None
        self.embeddings_ = None      # stored for evaluation (Fix 3)
        self.umap_embeddings_ = None
        self.outliers_before_ = None  # outlier count straight from HDBSCAN
        self.outliers_after_ = None   # outlier count after reduction

    # ------------------------------------------------------------------ #
    def _resolve_params(self, n_docs: int) -> Tuple[int, int, int]:
        """
        Auto-scale HDBSCAN / UMAP hyperparameters to corpus size.
        Rules (from BERTopic best-practice guidelines):
          min_cluster_size ~ max(2, n_docs // 10)  capped at 15
          min_samples      ~ max(1, min_cluster_size // 2)
          n_neighbors      ~ min(n_docs - 1, max(5, int(n_docs ** 0.5)))
        """
        mcs = self._user_min_cluster_size or max(2, min(15, n_docs // 10))
        ms  = self._user_min_samples      or max(1, mcs // 2)
        nn  = self._user_n_neighbors      or min(n_docs - 1, max(5, int(n_docs ** 0.5)))
        return mcs, ms, nn

    # ------------------------------------------------------------------ #
    def _build_topic_model(self, mcs: int, ms: int, nn: int) -> BERTopic:
        hdbscan_model = hdbscan.HDBSCAN(
            min_cluster_size=mcs,
            min_samples=ms,
            metric='euclidean',
            cluster_selection_method='eom',
            prediction_data=True
        )
        umap_model = UMAP(
            n_neighbors=nn,
            n_components=self.n_components,
            min_dist=0.0,
            metric='cosine',
            random_state=42
        )
        vectorizer_model = CountVectorizer(
            ngram_range=(1, 3),
            stop_words="english",
            min_df=2,
            max_df=0.85
        )
        return BERTopic(
            embedding_model=self.embedding_model,
            hdbscan_model=hdbscan_model,
            umap_model=umap_model,
            vectorizer_model=vectorizer_model,
            ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
            representation_model=KeyBERTInspired(),
            verbose=True,
            calculate_probabilities=True,
            top_n_words=10,
            language="english"
        )

    # ------------------------------------------------------------------ #
    def fit(self, documents: List[str], titles: List[str] = None):
        docs = [f"{t}\n{d}" for t, d in zip(titles, documents)] if titles else documents
        self.documents = docs
        n = len(docs)

        mcs, ms, nn = self._resolve_params(n)
        self.min_cluster_size = mcs
        self.min_samples      = ms
        self.n_neighbors      = nn

        print(f"📚 Fitting BERTopic on {n} documents...")
        print(f"   Auto-scaled params → min_cluster_size={mcs}, min_samples={ms}, n_neighbors={nn}")

        self.topic_model = self._build_topic_model(mcs, ms, nn)

        # Encode once, store embeddings for evaluation
        print("   Encoding documents…")
        self.embeddings_ = self.embedding_model.encode(docs, show_progress_bar=True)

        self.topics, self.probabilities = self.topic_model.fit_transform(docs, embeddings=self.embeddings_)
        self.is_fitted = True
        self.topic_info = self.topic_model.get_topic_info()

        if isinstance(self.topics, np.ndarray):
            self.topics = self.topics.tolist()

        n_topics   = len(self.topic_info) - 1
        n_outliers = sum(1 for t in self.topics if t == -1)
        self.outliers_before_ = n_outliers
        print(f"✅ Model fitted → {n_topics} topics, {n_outliers} outliers ({n_outliers/n*100:.1f}%)")

        # ---- OUTLIER REDUCTION (best practice: keep HDBSCAN honest, then ----
        # ---- reassign each outlier to its most similar topic by embedding ---
        # ---- cosine similarity; below-threshold papers STAY outliers) -------
        if self.reduce_outliers_strategy and n_outliers > 0 and n_topics > 0:
            new_topics = None
            try:  # official BERTopic API first
                new_topics = self.topic_model.reduce_outliers(
                    docs, self.topics,
                    strategy=self.reduce_outliers_strategy,
                    embeddings=self.embeddings_,
                    threshold=self.outlier_threshold)
            except Exception as e:
                print(f"   (BERTopic reduce_outliers unavailable: {e}; using manual fallback)")
                try:  # manual fallback — identical idea, guaranteed to work
                    from sklearn.preprocessing import normalize as _nrm
                    X = _nrm(np.asarray(self.embeddings_))
                    tp = np.array(self.topics)
                    tids = sorted(t for t in set(self.topics) if t >= 0)
                    temb = _nrm(np.vstack([X[tp == t].mean(axis=0) for t in tids]))
                    out_idx = np.where(tp == -1)[0]
                    sims = cosine_similarity(X[out_idx], temb)
                    best = sims.max(axis=1); which = np.array(tids)[sims.argmax(axis=1)]
                    tp2 = tp.copy(); m = best >= self.outlier_threshold
                    tp2[out_idx[m]] = which[m]
                    new_topics = tp2.tolist()
                except Exception as e2:
                    print(f"   (outlier reduction skipped: {e2})")
            if new_topics is not None:
                try:  # refresh topic keyword representations for the new members
                    self.topic_model.update_topics(docs, topics=new_topics)
                    self.topic_info = self.topic_model.get_topic_info()
                except Exception:
                    pass
                self.topics = list(new_topics)
                self.outliers_after_ = sum(1 for t in self.topics if t == -1)
                print(f"   🔻 Outlier reduction ({self.reduce_outliers_strategy}, "
                      f"threshold={self.outlier_threshold}): "
                      f"{self.outliers_before_} → {self.outliers_after_} outliers "
                      f"({self.outliers_after_/n*100:.1f}% of corpus)")
        return self

    # ------------------------------------------------------------------ #
    def predict(self, documents: List[str], titles: List[str] = None):
        if not self.is_fitted:
            raise ValueError("Call fit() first.")
        docs = [f"{t}\n{d}" for t, d in zip(titles, documents)] if titles else documents
        return self.topic_model.transform(docs)

    # ------------------------------------------------------------------ #
    def get_document_info(self, documents: List[str], titles: List[str] = None) -> pd.DataFrame:
        topics, probs = self.predict(documents, titles)
        rows = []
        for idx, (topic, prob) in enumerate(zip(topics, probs)):
            rows.append({
                'document_id':       idx,
                'topic_id':          int(topic),
                'topic_name':        self._get_topic_name(topic),
                'topic_confidence':  float(np.max(prob)) if isinstance(prob, np.ndarray) else float(prob),
                'is_outlier':        topic == -1,
                'topic_size':        self._get_topic_size(topic),
            })
        return pd.DataFrame(rows)

    def _get_topic_name(self, topic_id: int) -> str:
        if topic_id == -1:
            return "Outlier / No clear topic"
        words = self.topic_model.get_topic(topic_id)
        return ", ".join(w for w, _ in words[:5]) if words else "Unknown topic"

    def _get_topic_size(self, topic_id: int) -> int:
        if not self.topics:
            return 0
        return sum(1 for t in self.topics if t == topic_id)

    # ------------------------------------------------------------------ #
    def save_model(self, save_path: str, overwrite: bool = True):
        if not self.is_fitted:
            raise ValueError("Model not fitted.")
        os.makedirs(save_path, exist_ok=True)
        model_file = os.path.join(save_path, "bertopic_model")
        # RE-RUN SAFETY: clear any previous serialization (dir or file) so the
        # save overwrites cleanly instead of erroring or leaving stale files.
        if overwrite:
            if os.path.isdir(model_file):
                shutil.rmtree(model_file, ignore_errors=True)
            elif os.path.exists(model_file):
                try: os.remove(model_file)
                except OSError: pass
        try:
            self.topic_model.save(model_file, save_embedding_model=True)
        except Exception as e:
            # some BERTopic versions dislike an existing target; retry clean
            shutil.rmtree(model_file, ignore_errors=True)
            if os.path.exists(model_file):
                try: os.remove(model_file)
                except OSError: pass
            self.topic_model.save(model_file, save_embedding_model=True)
        joblib.dump({
            'embedding_model_name': self.embedding_model_name,
            'min_cluster_size':     self.min_cluster_size,
            'min_samples':          self.min_samples,
            'n_neighbors':          self.n_neighbors,
            'n_components':         self.n_components,
            'is_fitted':            self.is_fitted,
            'topics':               self.topics,
            'probabilities':        self.probabilities,
            'documents':            self.documents,
            'embeddings':           self.embeddings_,
            'topic_info':           self.topic_info.to_dict() if self.topic_info is not None else None,
            'timestamp':            datetime.now().isoformat(),
        }, os.path.join(save_path, "model_data.joblib"))
        with open(os.path.join(save_path, "config.json"), 'w') as f:
            json.dump({
                'embedding_model': self.embedding_model_name,
                'min_cluster_size': self.min_cluster_size,
                'n_topics': len(self.topic_info) - 1 if self.topic_info is not None else 0,
                'n_documents': len(self.documents) if self.documents else 0,
                'save_date': datetime.now().isoformat(),
            }, f, indent=2)
        print(f"✅ Model saved to {save_path}")

    @classmethod
    def load_model(cls, load_path: str):
        data = joblib.load(os.path.join(load_path, "model_data.joblib"))
        inst = cls(embedding_model=data['embedding_model_name'],
                   min_cluster_size=data['min_cluster_size'],
                   min_samples=data['min_samples'],
                   n_neighbors=data['n_neighbors'],
                   n_components=data['n_components'])
        inst.topic_model   = BERTopic.load(os.path.join(load_path, "bertopic_model"))
        inst.is_fitted     = data['is_fitted']
        inst.topics        = data['topics']
        inst.probabilities = data['probabilities']
        inst.documents     = data['documents']
        inst.embeddings_   = data.get('embeddings')
        if data['topic_info']:
            inst.topic_info = pd.DataFrame.from_dict(data['topic_info'])
        print(f"✅ Model loaded — {len(inst.topic_info)-1} topics, {len(inst.documents)} docs")
        return inst

## Step 12 - Scoring the model

Reports the numbers that answer "is this clustering any good": silhouette, topic
diversity and inter-topic similarity, C_v coherence through gensim, the spread of
assignment confidences, and a Gini coefficient for how evenly papers are distributed.

Read the silhouette with care. It is a property of the space it is measured in, not of
the corpus alone, and the same partition scores very differently before and after
dimensionality reduction. The cluster-figures step shows this directly.

In [12]:
# ============================================================================
# FIX 3 – MODEL PERFORMANCE EVALUATOR
# Silhouette now uses stored UMAP-space embeddings (accurate, fast, no re-encode).
# Added per-topic breakdown table and confidence stats.
# ============================================================================

class ModelPerformanceEvaluator:

    def __init__(self, extractor: BERTopicThemeExtractor,
                 documents: List[str],
                 true_labels: Optional[List] = None):
        self.extractor   = extractor
        self.documents   = documents
        self.true_labels = true_labels
        self.topics      = (extractor.topics.tolist()
                            if isinstance(extractor.topics, np.ndarray)
                            else (extractor.topics or []))
        self.probabilities = extractor.probabilities

    # ------------------------------------------------------------------ #
    def evaluate_all(self) -> Dict:
        return {
            'internal':     self._internal_metrics(),
            'topic_quality': self._topic_quality(),
            'coherence':    self._coherence(),
            'distribution': self._distribution(),
            'per_topic':    self._per_topic_stats(),
            'confidence':   self._confidence_stats(),
            'external':     self._external() if self.true_labels else None,
        }

    # ------------------------------------------------------------------ #
    def _internal_metrics(self) -> Dict:
        topics = self.topics
        if not topics:
            return {}
        n_outliers = sum(1 for t in topics if t == -1)
        n_clusters = len(set(topics)) - (1 if -1 in topics else 0)
        mask = np.array([t != -1 for t in topics])

        sil = 0.0
        if mask.sum() >= 4 and n_clusters >= 2:
            try:
                # FIX 3: use stored embeddings (UMAP or raw) — never re-encodes
                if self.extractor.embeddings_ is not None:
                    emb = np.array(self.extractor.embeddings_)[mask]
                else:
                    emb = self.extractor.embedding_model.encode(
                        [d for d, m in zip(self.documents, mask) if m],
                        show_progress_bar=False)
                labels = np.array(topics)[mask]
                sil = float(silhouette_score(emb, labels, metric='cosine'))
            except Exception as e:
                print(f"   ⚠️  Silhouette skipped: {e}")

        return {
            'silhouette_score':    sil,
            'n_clusters':          n_clusters,
            'n_outliers':          n_outliers,
            'outlier_pct':         n_outliers / len(topics) * 100,
            'coverage_pct':        (len(topics) - n_outliers) / len(topics) * 100,
        }

    # ------------------------------------------------------------------ #
    def _topic_quality(self) -> Dict:
        ti = self.extractor.topic_info
        if ti is None:
            return {}
        valid_ids = [t for t in ti['Topic'].values if t != -1]
        topic_words = []
        for tid in valid_ids:
            words = self.extractor.topic_model.get_topic(tid)
            if words:
                topic_words.append([w for w, _ in words[:10]])

        avg_sim = 0.0
        if len(topic_words) > 1:
            strings  = [' '.join(w) for w in topic_words]
            tfidf    = TfidfVectorizer().fit_transform(strings)
            sim_mat  = cosine_similarity(tfidf)
            np.fill_diagonal(sim_mat, 0)
            avg_sim  = sim_mat.sum() / (len(topic_words) * (len(topic_words) - 1))

        valid_counts = ti[ti['Topic'] != -1]['Count'].values
        return {
            'n_topics':              len(valid_ids),
            'topic_diversity':       round(1 - avg_sim, 4),
            'inter_topic_similarity': round(avg_sim, 4),
            'avg_topic_size':        round(float(valid_counts.mean()), 2) if len(valid_counts) else 0,
            'min_topic_size':        int(valid_counts.min()) if len(valid_counts) else 0,
            'max_topic_size':        int(valid_counts.max()) if len(valid_counts) else 0,
        }

    # ------------------------------------------------------------------ #
    def _coherence(self) -> Dict:
        if not GENSIM_AVAILABLE or not self.topics:
            return {'c_v_coherence': None, 'note': 'gensim not available'}
        try:
            tokenized = [d.lower().split() for d in self.documents]
            dictionary = Dictionary(tokenized)
            ti = self.extractor.topic_info
            topics_words = []
            for tid in ti['Topic'].values:
                if tid != -1:
                    words = self.extractor.topic_model.get_topic(tid)
                    if words:
                        topics_words.append([w for w, _ in words[:10]])
            if topics_words:
                cm = CoherenceModel(topics=topics_words, texts=tokenized,
                                    dictionary=dictionary, coherence='c_v')
                cv = cm.get_coherence()
                return {'c_v_coherence': round(cv, 4),
                        'interpretability_pct': round(cv * 100, 1)}
        except Exception as e:
            return {'c_v_coherence': None, 'error': str(e)}
        return {'c_v_coherence': None}

    # ------------------------------------------------------------------ #
    def _distribution(self) -> Dict:
        counts = [v for k, v in Counter(self.topics).items() if k != -1]
        if not counts:
            return {}
        arr = np.sort(counts)
        n   = len(arr)
        gini = float(np.sum((2 * np.arange(1, n+1) - n - 1) * arr) /
                     (n * np.sum(arr))) if np.sum(arr) > 0 else 1.0
        quality = ('excellent' if gini < 0.3 else
                   'good'      if gini < 0.5 else
                   'fair'      if gini < 0.7 else 'poor')
        return {
            'gini_coefficient':    round(gini, 4),
            'distribution_quality': quality,
            'dominant_topic_share': round(max(counts) / sum(counts), 4),
        }

    # ------------------------------------------------------------------ #
    def _per_topic_stats(self) -> pd.DataFrame:
        """Return a table: topic_id | label | size | share% | avg_confidence"""
        rows = []
        ti   = self.extractor.topic_info
        probs = self.probabilities if self.probabilities is not None else []
        topic_arr = np.array(self.topics)

        for _, row in ti.iterrows():
            tid = int(row['Topic'])
            if tid == -1:
                continue
            mask  = topic_arr == tid
            confs = []
            if len(probs) > 0:
                prob_arr = np.array(probs)
                for i, m in enumerate(mask):
                    if m and i < len(prob_arr):
                        p = prob_arr[i]
                        confs.append(float(np.max(p)) if isinstance(p, np.ndarray) else float(p))
            rows.append({
                'topic_id':       tid,
                'label':          self.extractor._get_topic_name(tid),
                'size':           int(row['Count']),
                'share_pct':      round(int(row['Count']) / len(self.topics) * 100, 1),
                'avg_confidence': round(np.mean(confs), 4) if confs else None,
            })
        return pd.DataFrame(rows).sort_values('size', ascending=False).reset_index(drop=True)

    # ------------------------------------------------------------------ #
    def _confidence_stats(self) -> Dict:
        if self.probabilities is None:
            return {}
        confs = []
        for p in self.probabilities:
            confs.append(float(np.max(p)) if isinstance(p, np.ndarray) else float(p))
        arr = np.array(confs)
        return {
            'mean_confidence':   round(float(arr.mean()), 4),
            'median_confidence': round(float(np.median(arr)), 4),
            'std_confidence':    round(float(arr.std()), 4),
            'pct_high_conf':     round(float((arr >= 0.7).mean() * 100), 1),  # ≥ 0.70
        }

    # ------------------------------------------------------------------ #
    def _external(self) -> Dict:
        mask = np.array([t != -1 for t in self.topics])
        ft   = np.array(self.topics)[mask]
        fl   = np.array(self.true_labels)[mask]
        if len(set(fl)) < 2:
            return {'error': 'Not enough unique ground-truth labels'}
        return {
            'adjusted_rand_index':      round(float(adjusted_rand_score(fl, ft)), 4),
            'normalized_mutual_info':   round(float(normalized_mutual_info_score(fl, ft)), 4),
        }

    # ------------------------------------------------------------------ #
    def generate_report(self) -> str:
        m = self.evaluate_all()
        i = m['internal']
        q = m['topic_quality']
        c = m['coherence']
        d = m['distribution']
        conf = m['confidence']

        sil_flag = "⚠️ weak" if i.get('silhouette_score', 0) < 0.1 else "✅ ok"

        lines = [
            "",
            "╔══════════════════════════════════════════════════════════════════╗",
            "║              MODEL PERFORMANCE REPORT (v2)                      ║",
            "╚══════════════════════════════════════════════════════════════════╝",
            "",
            "📊 CLUSTERING METRICS:",
            f"   • Silhouette Score  : {i.get('silhouette_score', 0):.4f}  {sil_flag}",
            f"   • Topics Found      : {i.get('n_clusters', 0)}",
            f"   • Outliers          : {i.get('n_outliers', 0)}  ({i.get('outlier_pct', 0):.1f}%)",
            f"   • Coverage          : {i.get('coverage_pct', 0):.1f}%  (docs assigned to a topic)",
            "",
            "🎯 TOPIC QUALITY:",
            f"   • Topic Diversity       : {q.get('topic_diversity', 0):.4f}  (1.0 = fully distinct)",
            f"   • Inter-topic Similarity: {q.get('inter_topic_similarity', 0):.4f}",
            f"   • Avg / Min / Max size  : {q.get('avg_topic_size', 0):.1f} / {q.get('min_topic_size', 0)} / {q.get('max_topic_size', 0)}",
            "",
            "📚 TOPIC COHERENCE:",
            f"   • C_v Coherence     : {c.get('c_v_coherence') or 'N/A'}",
            f"   • Interpretability  : {c.get('interpretability_pct') or 'N/A'}%",
            "",
            "🎲 CONFIDENCE STATS:",
            f"   • Mean / Median     : {conf.get('mean_confidence', 0):.4f} / {conf.get('median_confidence', 0):.4f}",
            f"   • Std Dev           : {conf.get('std_confidence', 0):.4f}",
            f"   • High-confidence≥0.7: {conf.get('pct_high_conf', 0):.1f}% of docs",
            "",
            "⚖️  DISTRIBUTION:",
            f"   • Gini Coefficient  : {d.get('gini_coefficient', 0):.4f}  (0 = perfectly even)",
            f"   • Quality           : {d.get('distribution_quality', 'N/A')}",
            f"   • Dominant topic    : {d.get('dominant_topic_share', 0)*100:.1f}% of docs",
        ]

        if m.get('external'):
            ext = m['external']
            lines += [
                "",
                "✅ EXTERNAL VALIDATION (vs ground truth):",
                f"   • Adjusted Rand Index  : {ext.get('adjusted_rand_index', 'N/A')}",
                f"   • Normalized Mutual Info: {ext.get('normalized_mutual_info', 'N/A')}",
            ]

        # Per-topic table
        pt = m['per_topic']
        if not pt.empty:
            lines += ["", "📋 PER-TOPIC BREAKDOWN:", pt.to_string(index=False)]

        return "\n".join(lines)

## Step 13 - The validation dashboard

Draws the scoring from step 7 as one multi-panel PNG for the dissertation.

The grid is built from whichever panels actually have data, so there are no empty boxes.
The three facet panels appear on their own once the research type, contribution type and
evaluation method columns exist; until then a line under the figure says why they are
absent.

In [13]:
# ============================================================================
# FIX 4 – VALIDATION DASHBOARD (visual)
# Saves a 6-panel figure: topic sizes, confidence histogram, topic heatmap,
# research-type bar, contribution bar, evaluation-method bar.
# ============================================================================

class ValidationDashboard:

    def __init__(self, df: pd.DataFrame, extractor: BERTopicThemeExtractor, output_dir: str):
        self.df         = df
        self.extractor  = extractor
        self.output_dir = output_dir

    def save_all(self):
        self._plot_main_dashboard()
        self._plot_topic_heatmap()
        print(f"📊 Validation plots saved to {self.output_dir}")

    # ------------------------------------------------------------------ #
    def _plot_main_dashboard(self):
        """Draw only the panels that have data, so the dashboard has no blanks.

        The facet panels (research type, contribution type, evaluation method)
        appear automatically once those columns exist. Until then the grid
        closes up around them rather than leaving empty boxes.
        """
        from functools import partial

        panels = []                                   # (column span, draw fn)
        ti = getattr(self.extractor, "topic_info", None)
        if ti is not None and not ti[ti["Topic"] != -1].empty:
            panels.append((2, self._panel_topic_sizes))
        if getattr(self.extractor, "topics", None) is not None:
            panels.append((1, self._panel_coverage))
        if getattr(self.extractor, "probabilities", None) is not None:
            panels.append((2, self._panel_confidence))
        if "cluster_label" in self.df.columns:
            panels.append((1, self._panel_clusters))
        for col, title, colour in [("research_type",     "Research Type",     "slateblue"),
                                   ("contribution_type", "Contribution Type", "teal"),
                                   ("evaluation_method", "Evaluation Method", "coral")]:
            if col in self.df.columns and self.df[col].notna().any():
                panels.append((1, partial(self._panel_counts, col, title, colour)))
        if "abstract_strategy" in self.df.columns:
            panels.append((1, self._panel_abstract_source))
        if "word_count" in self.df.columns:
            panels.append((1, self._panel_wordcount))

        NCOL = 3                                      # pack into a 3-column grid
        placed, row, col = [], 0, 0
        for span, draw in panels:
            if col + span > NCOL:
                row, col = row + 1, 0
            placed.append((row, col, span, draw))
            col += span
            if col >= NCOL:
                row, col = row + 1, 0
        nrow = max((r for r, _, _, _ in placed), default=0) + 1

        fig = plt.figure(figsize=(18, 4.7 * nrow))
        fig.suptitle("SMS Pipeline \u2014 Validation Dashboard",
                     fontsize=16, fontweight="bold", y=0.99)
        gs = gridspec.GridSpec(nrow, NCOL, figure=fig, hspace=0.5, wspace=0.35)
        for r, c, span, draw in placed:
            draw(fig.add_subplot(gs[r, c:c + span]))

        absent = [n for n, c in (("research type",     "research_type"),
                                 ("contribution type", "contribution_type"),
                                 ("evaluation method", "evaluation_method"))
                  if c not in self.df.columns]
        if absent:
            fig.text(0.5, 0.004,
                     "Facet panels not shown: no " + ", ".join(absent) + " column yet. "
                     "Map the clusters in cluster_summary.csv to add them.",
                     ha="center", fontsize=11, color="#52514e")

        path = os.path.join(self.output_dir, "validation_dashboard.png")
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"   \u2705 Dashboard \u2192 {path}  ({len(placed)} panels)")

    # ---- individual panels ------------------------------------------- #
    def _panel_topic_sizes(self, ax):
        ti = self.extractor.topic_info
        v = ti[ti["Topic"] != -1].copy()
        v["label"] = v["Topic"].apply(lambda t: self.extractor._get_topic_name(t)[:30])
        v = v.sort_values("Count", ascending=False)
        ax.barh(v["label"], v["Count"], color="steelblue")
        ax.set_title("Topic Size Distribution")
        ax.set_xlabel("# Documents")
        ax.invert_yaxis()

    def _panel_coverage(self, ax):
        n_out = sum(1 for t in self.extractor.topics if t == -1)
        n_in = len(self.extractor.topics) - n_out
        if n_out == 0:
            ax.pie([n_in], labels=["Assigned"], autopct="%1.1f%%",
                   colors=["#4CAF50"], startangle=90)
        else:
            ax.pie([n_in, n_out], labels=["Assigned", "Outlier"], autopct="%1.1f%%",
                   colors=["#4CAF50", "#F44336"], startangle=90)
        ax.set_title("Coverage")

    def _panel_confidence(self, ax):
        confs = [float(np.max(p)) if isinstance(p, np.ndarray) else float(p)
                 for p in self.extractor.probabilities]
        ax.hist(confs, bins=20, color="darkorange", edgecolor="white", alpha=0.85)
        ax.axvline(np.mean(confs), color="red", linestyle="--",
                   label=f"Mean={np.mean(confs):.2f}")
        ax.axvline(0.7, color="green", linestyle=":", label="0.70 threshold")
        ax.legend(fontsize=9)
        ax.set_title("Topic Assignment Confidence")
        ax.set_xlabel("Confidence Score")
        ax.set_ylabel("# Documents")

    def _panel_clusters(self, ax):
        sizes = self.df["cluster_label"].value_counts()
        labels = [str(s).split(":")[0].strip()[:22] for s in sizes.index]
        ax.barh(labels, sizes.values, color="#2a78d6")
        ax.set_title("K-Means Cluster Sizes")
        ax.set_xlabel("# Papers")
        ax.invert_yaxis()

    def _panel_counts(self, col, title, colour, ax):
        counts = self.df[col].value_counts()
        ax.barh([str(i)[:24] for i in counts.index], counts.values, color=colour)
        ax.set_title(title)
        ax.set_xlabel("# Papers")
        ax.invert_yaxis()

    def _panel_abstract_source(self, ax):
        counts = self.df["abstract_strategy"].value_counts()
        ax.bar(counts.index.astype(str), counts.values, color="#1baf7a")
        ax.set_title("How the Abstract Was Found")
        ax.set_ylabel("# Papers")

    def _panel_wordcount(self, ax):
        ax.hist(self.df["word_count"].dropna(), bins=20,
                color="mediumseagreen", edgecolor="white")
        ax.set_title("Paper Word-Count Distribution")
        ax.set_xlabel("Words")

    # ------------------------------------------------------------------ #
    def _plot_topic_heatmap(self):
        """
        Heatmap: BERTopic themes × research types.
        Shows where each cluster concentrates — useful for ML validation.
        """
        if 'bertopic_theme' not in self.df.columns or 'research_type' not in self.df.columns:
            return
        ct = pd.crosstab(self.df['bertopic_theme'], self.df['research_type'])
        if ct.empty:
            return

        fig, ax = plt.subplots(figsize=(max(10, len(ct.columns) * 1.5),
                                        max(6,  len(ct.index)   * 0.6)))
        sns.heatmap(ct, annot=True, fmt='d', cmap='Blues',
                    linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.7})
        ax.set_title('BERTopic Theme × Research Type (counts)', fontsize=13)
        ax.set_xlabel('Research Type')
        ax.set_ylabel('BERTopic Theme')
        plt.tight_layout()

        path = os.path.join(self.output_dir, "topic_research_heatmap.png")
        fig.savefig(path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"   ✅ Heatmap   → {path}")

## Step 14 - Benchmarking

Compares the automated output against the manual baseline: the facet distribution
reported by Binamungu and Maro (2023), and the time a human screener would need for
the same number of studies.

In [14]:
# ============================================================================
# FIX 5 – BENCHMARKING: now covers all three dimensions
# ============================================================================

class BenchmarkingMetrics:

    PAPER_FINDINGS = {
        'research_type_distribution': {
            'Solution Proposal': 89, 'Validation Research': 62,
            'Evaluation Research': 19, 'Experience Papers': 13,
            'Philosophical Papers': 3, 'Opinion Papers': 1,
        },
        'contribution_distribution': {
            'Process': 52, 'Tool': 44, 'Empirical Insights': 41,
            'Method': 31, 'Model': 19, 'Metric': 3,
        },
        'evaluation_distribution': {
            'Example': 71, 'Case Study': 55, 'Experiment': 31,
            'Experience Report': 14, 'Discussion': 11, 'Rigorous Analysis': 3,
        },
        'total_papers_analyzed': 166,
    }

    def __init__(self, paper_findings: Dict = None):
        self.paper_findings = paper_findings or self.PAPER_FINDINGS

    # ------------------------------------------------------------------ #
    def _mae_and_corr(self, pipeline_col: pd.Series, ref_dist: Dict) -> Dict:
        pipeline_dist  = pipeline_col.value_counts().to_dict()
        pipeline_total = sum(pipeline_dist.values())
        paper_total    = sum(ref_dist.values())

        all_keys = set(pipeline_dist) | set(ref_dist)
        mae = sum(
            abs(pipeline_dist.get(k, 0) / pipeline_total -
                ref_dist.get(k, 0)    / paper_total)
            for k in all_keys
        ) / len(ref_dist)

        common = set(pipeline_dist) & set(ref_dist)
        corr   = 0.0
        if len(common) >= 2:
            pv = np.array([pipeline_dist[k] / pipeline_total for k in common])
            rv = np.array([ref_dist[k]    / paper_total    for k in common])
            denom = np.linalg.norm(pv) * np.linalg.norm(rv)
            corr  = float(np.dot(pv, rv) / denom) if denom > 1e-9 else 0.0

        return {'mae': round(mae, 4), 'correlation': round(corr, 4),
                'benchmark_score': round(1 - mae, 4)}

    # ------------------------------------------------------------------ #
    def calculate_all(self, df: pd.DataFrame) -> Dict:
        results = {}
        if 'research_type' in df.columns:
            results['research_type'] = self._mae_and_corr(
                df['research_type'], self.paper_findings['research_type_distribution'])
        if 'contribution_type' in df.columns:
            results['contribution'] = self._mae_and_corr(
                df['contribution_type'], self.paper_findings['contribution_distribution'])
        if 'evaluation_method' in df.columns:
            results['evaluation'] = self._mae_and_corr(
                df['evaluation_method'], self.paper_findings['evaluation_distribution'])
        return results

    # ------------------------------------------------------------------ #
    def generate_benchmark_report(self, df: pd.DataFrame) -> str:
        all_m = self.calculate_all(df)
        pf    = self.paper_findings

        def row(dim: str, col: str, ref_dist: Dict):
            m = all_m.get(dim, {})
            lines = [
                f"\n  [{dim.upper()}]",
                f"  {'Category':<28} {'Paper (2023)':>15} {'Pipeline':>12} {'Δ':>8}",
                "  " + "-"*65,
            ]
            pipe_dist = df[col].value_counts().to_dict() if col in df.columns else {}
            pipe_tot  = sum(pipe_dist.values()) or 1
            ref_tot   = sum(ref_dist.values())
            for k, rv in sorted(ref_dist.items(), key=lambda x: -x[1]):
                pv  = pipe_dist.get(k, 0)
                delta = pv/pipe_tot*100 - rv/ref_tot*100
                lines.append(
                    f"  {k:<28} {rv/ref_tot*100:>13.1f}%  {pv/pipe_tot*100:>10.1f}%  "
                    f"{'▲' if delta>0 else '▼'}{abs(delta):>5.1f}%")
            lines += [
                "  " + "-"*65,
                f"  MAE={m.get('mae','N/A'):.4f}  Correlation={m.get('correlation','N/A'):.4f}  "
                f"Benchmark={m.get('benchmark_score','N/A'):.4f}",
            ]
            return "\n".join(lines)

        header = (
            "\n╔══════════════════════════════════════════════════════════════════╗\n"
            "║       BENCHMARKING REPORT — Binamungu & Maro (2023) SMS         ║\n"
            "╚══════════════════════════════════════════════════════════════════╝\n"
            f"  Papers: 2023 study = {pf['total_papers_analyzed']}  │  Pipeline = {len(df)}"
        )
        return (header
                + row('research_type', 'research_type', pf['research_type_distribution'])
                + row('contribution',  'contribution_type', pf['contribution_distribution'])
                + row('evaluation',    'evaluation_method', pf['evaluation_distribution']))

## Step 15 - The orchestrator

`CompleteSMSPipeline` ties the preceding steps together: extract, cluster, assign facets,
model themes, score, benchmark, save. It owns the output directory and the results table that every
later step consumes.

Results are written as **CSV only**. The main table goes to `sms_results.csv`, and each
count table that used to be an Excel worksheet becomes its own file under
`distributions/`. Plain CSV stays readable without Excel, diffs cleanly in version
control, and loads with a single `pd.read_csv` call.

In [15]:
# ============================================================================
# MAIN SMS PIPELINE (orchestrator)
# ============================================================================

class CompleteSMSPipeline:

    def __init__(self, output_dir: str = None, use_tfidf: bool = False,
                 target_k: int = None, min_clusters: int = 2, k_max: int = 15):
        self.extractor       = RobustPDFExtractor()
        # unsupervised K-Means grouping; target_k/min_clusters control diversity
        self.classifier      = KMeansFacetClassifier(
            use_tfidf=use_tfidf, target_k=target_k,
            min_clusters=min_clusters, k_max=k_max)
        self.benchmark       = BenchmarkingMetrics()
        self.theme_extractor = None
        self.results         = []
        self.output_dir      = output_dir or create_output_directory()
        os.makedirs(self.output_dir, exist_ok=True)          # fix: ensure it exists
        os.makedirs(os.path.join(self.output_dir, "figs"), exist_ok=True)

    # ------------------------------------------------------------------ #
    def process_and_train(self, pdf_folder: str = None, dataset=None,
                          save_model: bool = True, **model_params):
        _t0 = datetime.now()          # ⏱ end-to-end timing for the efficiency target
        print("\n" + "="*70)
        print("📚 COMPLETE SMS PIPELINE WITH BERTopic + HDBSCAN  [v3 — PyMuPDF]")
        print("="*70)

        processed_docs, successful = [], 0
        if dataset is not None:
            # PART B path: the PDFs were read once in Part A and are not
            # touched again, so modelling can be re-run in seconds.
            rows = dataset.to_dict("records")
            print(f"📂 Loading {len(rows)} papers from the prepared dataset")
            for row in tqdm(rows, desc="Loading dataset"):
                result = self._doc_from_row(row)
                if result:
                    processed_docs.append(result)
                    self.results.append(result['metadata'])
                    successful += 1
            print(f"✅ Loaded {successful}/{len(rows)} papers")
        else:
            pdf_files = sorted([os.path.join(pdf_folder, f)
                                 for f in os.listdir(pdf_folder)
                                 if f.lower().endswith('.pdf')])
            print(f"📁 Found {len(pdf_files)} PDF files")
            for pdf_path in tqdm(pdf_files, desc="Extracting PDFs"):
                result = self._extract_paper_data(pdf_path)
                if result:
                    processed_docs.append(result)
                    self.results.append(result['metadata'])
                    successful += 1
            print(f"✅ Successfully processed {successful}/{len(pdf_files)} PDFs")

        # Abstract recovery stats — now sourced from the extractor's own
        # strategy report rather than guessed from string length.
        n_with_abstract = sum(1 for d in processed_docs if d['abstract'])
        strat_counts = Counter(d.get('abstract_strategy', 'none') for d in processed_docs)
        print(f"   📑 Abstracts found : {n_with_abstract}/{successful}  "
              f"({', '.join(f'{k}={v}' for k, v in strat_counts.most_common())})")
        n_typo_title = sum(1 for d in processed_docs if d.get('title_source') == 'typography')
        print(f"   🔤 Titles from layout: {n_typo_title}/{successful}")
        method_counts = Counter(self.results[i].get('extraction_method', 'native')
                                for i in range(len(processed_docs)))
        n_scanned = method_counts.get('ocr', 0) + method_counts.get('hybrid', 0)
        if n_scanned:
            print(f"   🖨️  Scanned pages OCR'd: {n_scanned}/{successful} papers needed OCR "
                  f"({', '.join(f'{k}={v}' for k, v in method_counts.most_common())})")

        if not processed_docs:
            print("❌ No documents processed.")
            return None, None

        # ---- FULLY UNSUPERVISED GROUPING (K-Means, batch over corpus) ----
        print("\n🔵 Grouping papers by K-Means clustering (unsupervised, no rules, no labels)…")
        clf_texts = [d['clf_text'] for d in processed_docs]
        self.clf_texts_ = clf_texts   # kept so the saved model can be verified
        grouping = self.classifier.fit_predict(clf_texts)
        for i in range(len(processed_docs)):
            cid = grouping['cluster_id'][i]
            self.results[i]['cluster_id']       = cid
            self.results[i]['cluster_keywords'] = self.classifier.cluster_keywords_.get(cid, '')
            self.results[i]['cluster_label']    = grouping['cluster_label'][i]
        # ---- THE THREE MAPPING FACETS (zero-shot, no labels, no keyword rules) ----
        self.facet_report_ = assign_facets(self.results,
                                           [d['facet_text'] for d in processed_docs],
                                           encoder=self.classifier._get_embedder())

        # save the cluster summary so clusters can be mapped to categories by hand
        try:
            self.classifier.cluster_summary().to_csv(
                os.path.join(self.output_dir, "cluster_summary.csv"), index=False)
            print(f"   ✅ {self.classifier.chosen_k_.get('k','?')} clusters; "
                  f"summary → {self.output_dir}/cluster_summary.csv")
        except Exception as e:
            print(f"   (cluster summary skipped: {e})")
        # save the silhouette-vs-k diagram
        try:
            self.classifier.plot_silhouette(
                os.path.join(self.output_dir, "figs", "silhouette_vs_k.png"))
        except Exception as e:
            print(f"   (silhouette diagram skipped: {e})")
        # ALGORITHM COMPARISON: K-Means vs HDBSCAN on the same reduced space
        try:
            compare_kmeans_hdbscan(self.classifier.doc_vecs_, clf_texts, self.output_dir)
        except Exception as e:
            print(f"   (K-Means vs HDBSCAN comparison skipped: {e})")

        # Use ALL docs that have any abstract (S1–S3 always provides one)
        docs_for_bert  = [d for d in processed_docs if d['abstract']]
        abstracts      = [clean_academic_text(d['abstract']) or d['abstract'] for d in docs_for_bert]
        titles         = [d['title']    for d in docs_for_bert]

        print(f"\n🎯 Training BERTopic on {len(abstracts)} documents…")
        self.theme_extractor = BERTopicThemeExtractor(**model_params)
        self.theme_extractor.fit(abstracts, titles)

        doc_info = self.theme_extractor.get_document_info(abstracts, titles)

        # Align topic info back to self.results
        bert_idx = 0
        for i, doc in enumerate(processed_docs):
            if doc['abstract'] and bert_idx < len(doc_info):
                info = doc_info.iloc[bert_idx]
                self.results[i]['bertopic_theme']     = info['topic_name']
                self.results[i]['bertopic_topic_id']  = info['topic_id']
                self.results[i]['bertopic_confidence'] = info['topic_confidence']
                self.results[i]['is_outlier']          = info['is_outlier']
                bert_idx += 1
            else:
                self.results[i]['bertopic_theme']     = 'No abstract'
                self.results[i]['bertopic_topic_id']  = -2
                self.results[i]['bertopic_confidence'] = 0.0
                self.results[i]['is_outlier']          = True

        # Evaluate
        print("\n📊 Evaluating model performance…")
        evaluator = ModelPerformanceEvaluator(self.theme_extractor, abstracts)
        print(evaluator.generate_report())

        df = self._to_dataframe()

        # Benchmark (facet distribution vs 2023) — only if named facets exist.
        # In the fully-unsupervised setup the output is anonymous clusters, so
        # this comparison does not apply until clusters are mapped to categories.
        if 'research_type' in df.columns:
            print(self.benchmark.generate_benchmark_report(df))
        else:
            print("\n(ℹ️  Facet benchmark skipped - output is unsupervised clusters. "
                  "Map clusters to SMS categories via cluster_summary.csv to enable it.)")

        # Validation dashboard (visual)
        dashboard = ValidationDashboard(df, self.theme_extractor, self.output_dir)
        dashboard.save_all()

        # Baseline (K-Means) vs primary (BERTopic) comparison — the framing the
        # methodology prescribes; saves method_comparison.json + figure
        try:
            compare_methods(df, self.output_dir)
        except Exception as e:
            print(f"(method comparison skipped: {e})")

        if save_model:
            model_path = os.path.join(self.output_dir, "bertopic_model")
            self.theme_extractor.save_model(model_path)

        self._save_results(df)
        # ⏱ EFFICIENCY TARGET (panel Comment #4): measured per-study time vs
        # the 107-minute manual single-extraction baseline (Felizardo et al. 2024)
        try:
            elapsed = (datetime.now() - _t0).total_seconds()
            n_ok = len(df)
            secs_per_study = elapsed / max(1, n_ok)
            timing = TimingBenchmark.report(secs_per_study)
            timing['total_seconds'] = round(elapsed, 1)
            timing['n_studies'] = int(n_ok)
            with open(os.path.join(self.output_dir, 'timing.json'), 'w') as fh:
                json.dump(timing, fh, indent=2)
            print(f"\n⏱  TIMING: {elapsed:.0f}s total for {n_ok} studies "
                  f"= {secs_per_study:.1f}s/study "
                  f"({timing['automated_min_per_study']:.2f} min vs 107 min manual "
                  f"→ {timing['reduction_vs_single_pct']:.1f}% reduction; "
                  f"≥50% target met: {timing['meets_50pct_target']})")
        except Exception as e:
            print(f"(timing skipped: {e})")

        self._print_final_summary(df)
        return df, self.theme_extractor

    # ------------------------------------------------------------------ #
    def load_and_analyze(self, model_path: str, pdf_folder: str):
        print("\n" + "="*70)
        print("📚 LOADING EXISTING MODEL FOR ANALYSIS")
        print("="*70)
        self.theme_extractor = BERTopicThemeExtractor.load_model(model_path)

        pdf_files = sorted([os.path.join(pdf_folder, f)
                             for f in os.listdir(pdf_folder)
                             if f.lower().endswith('.pdf')])
        processed_docs = []
        for pdf_path in tqdm(pdf_files, desc="Processing PDFs"):
            result = self._extract_paper_data(pdf_path)
            if result and result['abstract']:
                processed_docs.append(result)

        if not processed_docs:
            print("❌ No documents processed.")
            return None

        abstracts = [clean_academic_text(d['abstract']) or d['abstract'] for d in processed_docs]
        titles    = [d['title']    for d in processed_docs]
        doc_info  = self.theme_extractor.get_document_info(abstracts, titles)

        # unsupervised grouping (K-Means) over this corpus
        print("\n🔵 Grouping papers by K-Means clustering (unsupervised, no rules)…")
        grouping = self.classifier.fit_predict([d['clf_text'] for d in processed_docs])

        for idx, (doc, (_, info)) in enumerate(zip(processed_docs, doc_info.iterrows())):
            r = doc['metadata']
            cid = grouping['cluster_id'][idx]
            r['cluster_id']          = cid
            r['cluster_keywords']    = self.classifier.cluster_keywords_.get(cid, '')
            r['cluster_label']       = grouping['cluster_label'][idx]
            r['bertopic_theme']      = info['topic_name']
            r['bertopic_topic_id']   = info['topic_id']
            r['bertopic_confidence'] = info['topic_confidence']
            r['is_outlier']          = info['is_outlier']
            self.results.append(r)

        df = self._to_dataframe()
        self._save_results(df)
        return df

    # ------------------------------------------------------------------ #
    def _doc_from_row(self, row) -> Optional[Dict]:
        """Rebuild a working document from one row of the Part A dataset.

        Every derived text is recomputed here rather than stored, so changing
        how text is cleaned, or how a facet is composed, only needs Part B to
        re-run. The dataset itself never goes stale.
        """
        def _s(key):
            v = row.get(key, "")
            return "" if v is None or (isinstance(v, float) and pd.isna(v)) else str(v)

        if not _s("full_text"):
            return None
        title, abstract = _s("title"), _s("abstract")
        raw_clf = (f"{title}\n{abstract}\n"
                   + _s("introduction")[:1500] + _s("methodology")[:1000])

        return {
            'metadata': {
                'paper_id':          _s("paper_id"),
                'title':             clean_for_excel(title[:500] or "No title"),
                'abstract':          clean_for_excel(abstract[:5000]),
                'word_count':        int(row.get("word_count") or 0),
                'abstract_strategy': _s("abstract_strategy") or "none",
                'title_source':      _s("title_source") or "none",
                'extraction_method': _s("extraction_method") or "native",
                'ocr_pages':         int(row.get("ocr_pages") or 0),
                'source_file':       _s("source_file"),
                'n_sections':        int(row.get("n_sections") or 0),
                **({ 'introduction': _s("introduction"),
                     'methodology':  _s("methodology"),
                     'results':      _s("results"),
                     'discussion':   _s("discussion"),
                     'conclusion':   _s("conclusion"),
                     'full_text':    _s("full_text") } if RESULTS_INCLUDE_FULLTEXT else {}),
            },
            'clf_text': clean_academic_text(raw_clf),
            'facet_text': {
                'research_type':     f"{title}. {abstract}",
                'contribution_type': f"{title}. {abstract} " + _s("introduction")[:1500],
                'evaluation_method': f"{abstract} " + _s("methodology")[:1500]
                                     + " " + _s("results")[:1500],
            },
            'abstract':          abstract,
            'title':             title,
            'abstract_strategy': _s("abstract_strategy") or "none",
            'title_source':      _s("title_source") or "none",
        }

    # ------------------------------------------------------------------ #
    def _extract_paper_data(self, pdf_path: str) -> Optional[Dict]:
        sections = self.extractor.extract_pdf(pdf_path)
        if not sections.get('full_text'):
            return None

        # clustering text — CLEANED so PDF boilerplate can't form noise clusters
        raw_clf = (f"{sections['title']}\n{sections['abstract']}\n"
                   + sections['introduction'][:1500]
                   + sections['methodology'][:1000])
        clf_text = clean_academic_text(raw_clf)

        abstract = sections['abstract']
        # Strategy now comes straight from the extractor's own report — no
        # more inferring quality from string length after the fact.
        strategy = sections.get('abstract_strategy', 'none')

        return {
            'metadata': {
                'paper_id':          os.path.basename(pdf_path).replace('.pdf', ''),
                'title':             clean_for_excel(sections['title'][:500] or "No title"),
                'abstract':          clean_for_excel(abstract[:5000]) if abstract else "",
                'word_count':        len(sections['full_text'].split()),
                'abstract_strategy': strategy,
                'title_source':      sections.get('title_source', 'none'),
                'extraction_method': sections.get('extraction_method', 'native'),
                'ocr_pages':         sections.get('ocr_pages', 0),
                'source_file':       os.path.basename(pdf_path),
                'n_sections':        sum(1 for _c in
                                         ('introduction', 'methodology', 'results',
                                          'discussion', 'conclusion')
                                         if sections.get(_c, '').strip()),
                **({ 'introduction': sections.get('introduction', ''),
                     'methodology':  sections.get('methodology', ''),
                     'results':      sections.get('results', ''),
                     'discussion':   sections.get('discussion', ''),
                     'conclusion':   sections.get('conclusion', ''),
                     'full_text':    sections.get('full_text', '') }
                   if RESULTS_INCLUDE_FULLTEXT else {}),
                # facet columns intentionally omitted — grouping is unsupervised;
                # cluster_id / cluster_keywords are added in batch (see process_and_train)
            },
            'clf_text':          clf_text,   # text used for K-Means clustering
            'facet_text': {      # what each facet is judged on (see Step 6)
                'research_type':     f"{sections['title']}. {abstract}",
                'contribution_type': f"{sections['title']}. {abstract} "
                                     + sections['introduction'][:1500],
                'evaluation_method': f"{abstract} " + sections['methodology'][:1500]
                                     + " " + sections['results'][:1500],
            },
            'abstract':          abstract,
            'title':             sections['title'],
            'abstract_strategy': strategy,
            'title_source':      sections.get('title_source', 'none'),
        }

    # ------------------------------------------------------------------ #
    def _to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self.results)

    # ------------------------------------------------------------------ #
    def _save_results(self, df: pd.DataFrame):
        _tail = ["introduction", "methodology", "results", "discussion",
                 "conclusion", "abstract", "full_text"]
        _front = [c for c in df.columns if c not in _tail]
        df = df[_front + [c for c in _tail if c in df.columns]]

        csv_path = os.path.join(self.output_dir, "sms_results.csv")
        try:
            df.to_csv(csv_path, index=False, encoding='utf-8')
            print(f"✅ CSV  → {csv_path}")
        except Exception as e:
            # re-run safety: if the file is locked/open, write a timestamped copy
            alt = os.path.join(self.output_dir, f"sms_results_{datetime.now():%H%M%S}.csv")
            df.to_csv(alt, index=False, encoding='utf-8')
            print(f"⚠️  {csv_path} locked ({e}); wrote {alt} instead")

        # One CSV per distribution, in place of the old multi-sheet workbook.
        dist_dir = os.path.join(self.output_dir, "distributions")
        os.makedirs(dist_dir, exist_ok=True)
        written = []
        for col, name in [('cluster_label',     'cluster_distribution'),
                          ('research_type',     'research_type_distribution'),
                          ('contribution_type', 'contribution_type_distribution'),
                          ('evaluation_method', 'evaluation_method_distribution'),
                          ('bertopic_theme',    'theme_distribution'),
                          ('abstract_strategy', 'extraction_quality'),
                          ('extraction_method', 'ocr_usage')]:
            if col not in df.columns:
                continue
            counts = (df[col].value_counts()
                        .rename_axis(col).reset_index(name='count'))
            counts['share_pct'] = (100 * counts['count'] / max(len(df), 1)).round(1)
            counts.to_csv(os.path.join(dist_dir, f"{name}.csv"),
                          index=False, encoding='utf-8')
            written.append(name)
        if written:
            print(f"✅ {len(written)} distribution CSVs → {dist_dir}")

        summary = os.path.join(self.output_dir, "summary.txt")
        with open(summary, 'w') as f:
            f.write("SMS Pipeline Results Summary\n" + "="*50 + "\n")
            f.write(f"Total Papers        : {len(df)}\n")
            f.write(f"With Abstract       : {(df.get('abstract_strategy','') != 'none').sum()}\n")
            if 'abstract_strategy' in df.columns:
                for k, v in df['abstract_strategy'].value_counts().items():
                    f.write(f"  · {k:<12}: {v}\n")
            if 'extraction_method' in df.columns:
                f.write(f"Extraction Method   :\n")
                for k, v in df['extraction_method'].value_counts().items():
                    f.write(f"  · {k:<12}: {v}\n")
            f.write(f"Research Types      : {df['research_type'].nunique() if 'research_type' in df.columns else 0}\n")
            f.write(f"BERTopic Themes     : {df['bertopic_theme'].nunique() if 'bertopic_theme' in df.columns else 0}\n")
            f.write(f"Outlier Papers      : {df['is_outlier'].sum() if 'is_outlier' in df.columns else 0}\n")
        print(f"✅ Summary → {summary}")

    # ------------------------------------------------------------------ #
    def _print_final_summary(self, df: pd.DataFrame):
        print("\n" + "="*60)
        print("📊 FINAL RESULTS SUMMARY")
        print("="*60)
        print(f"✅ Processed  : {len(df)} papers")
        if 'bertopic_theme' in df.columns:
            n_topics = df.loc[df['bertopic_topic_id'] >= 0, 'bertopic_topic_id'].nunique()
            print(f"✅ BERTopics  : {n_topics} themes discovered")
        print(f"✅ Outputs    : {self.output_dir}")
        if len(df) > 0:
            print("\n📄 Sample Result:")
            r = df.iloc[0]
            print(f"   Title          : {str(r.get('title',''))[:80]}…")
            print(f"   Research Type  : {r.get('research_type','')}")
            print(f"   BERTopic Theme : {r.get('bertopic_theme','')}")
            print(f"   Confidence     : {r.get('bertopic_confidence', 0):.3f}")
            print(f"   Abstract src   : {r.get('abstract_strategy','')}")

## Step 16 - Entry points

`main_train_mode` trains from a folder of PDFs. `main_load_mode` reloads a model saved
earlier and applies it to new PDFs without refitting.

In [16]:
# ============================================================================
# ENTRY POINTS
# ============================================================================

def main_train_mode(pdf_folder: str, output_dir: str = None,
                    use_tfidf: bool = False, target_k: int = None,
                    min_clusters: int = 2, k_max: int = 15,
                    min_cluster_size: int = None,
                    reduce_outliers_strategy: str = 'embeddings',
                    outlier_threshold: float = 0.4):
    """Train new model on PDF collection.
    target_k                : force this many K-Means baseline clusters (None = silhouette).
    min_clusters            : softer floor for the silhouette search.
    min_cluster_size        : BERTopic minimum theme size (None = auto ~12 for n=123).
    reduce_outliers_strategy: 'embeddings' reassigns each outlier to its most
                              similar theme (cosine); None keeps raw HDBSCAN outliers.
    outlier_threshold       : minimum similarity to reassign; papers below it
                              STAY outliers (0.4 keeps only truly alien papers out)."""
    pipeline = CompleteSMSPipeline(output_dir, use_tfidf=use_tfidf,
                                   target_k=target_k, min_clusters=min_clusters, k_max=k_max)
    df, model = pipeline.process_and_train(
        pdf_folder=pdf_folder,
        save_model=True,
        embedding_model='all-MiniLM-L6-v2',
        min_cluster_size=min_cluster_size,
        n_components=5,
        reduce_outliers_strategy=reduce_outliers_strategy,
        outlier_threshold=outlier_threshold,
    )
    return pipeline


def main_load_mode(model_path: str, pdf_folder: str, output_dir: str = None,
                   use_tfidf: bool = False, target_k: int = None,
                   min_clusters: int = 2, k_max: int = 15):
    """Load existing model and analyze new PDFs (same clustering settings as
    train mode, so both entry points behave identically)."""
    pipeline = CompleteSMSPipeline(output_dir, use_tfidf=use_tfidf,
                                   target_k=target_k, min_clusters=min_clusters, k_max=k_max)
    df = pipeline.load_and_analyze(model_path, pdf_folder)
    if df is not None:
        print(f"\n✅ Analyzed {len(df)} papers → {pipeline.output_dir}")
    return pipeline

## Step 17 - Evaluation module

Four independent checks, each optional and each degrading quietly when its input is
missing: cross-validated F1 for a classifier trained on the pipeline's own labels, a
distribution benchmark against the published baseline, a gold-standard comparison once
you have corrected `gold_labels_template.csv`, and a timing comparison against manual
screening.

These need **facet** columns — research type, contribution type, evaluation method. The
unsupervised run produces clusters rather than facets, so until you map clusters onto
the mapping categories these checks report nothing. That is expected, not a failure.

In [17]:
# ============================================================================
# EVALUATION MODULE — supervised F1 (no manual labels) + distribution benchmark
#   + optional gold-standard F1 + timing.  Produces every number Chapter 5 needs.
# ============================================================================

def _build_text(df):
    title = df['title'].fillna('') if 'title' in df else ''
    abstract = df['abstract'].fillna('') if 'abstract' in df else ''
    return (title + '. ' + abstract).tolist()


def relabel_with_kmeans(df, use_tfidf=False, k_min=2, k_max=15):
    """Re-group an existing results CSV by unsupervised K-Means (no rules, no
    labels; k chosen by silhouette). Adds cluster_id / cluster_keywords /
    cluster_label columns from title+abstract. BERTopic themes are untouched.
    Set use_tfidf=True for the TF-IDF lexical baseline instead of Sentence-BERT."""
    clf = KMeansFacetClassifier(use_tfidf=use_tfidf, k_min=k_min, k_max=k_max)
    grouping = clf.fit_predict(_build_text(df))
    out = df.copy()
    out['cluster_id']       = grouping['cluster_id']
    out['cluster_keywords'] = [clf.cluster_keywords_.get(c, '') for c in grouping['cluster_id']]
    out['cluster_label']    = grouping['cluster_label']
    return out


# backward-compatible alias (older cells may call the old name)
relabel_with_improved_classifier = relabel_with_kmeans


class SupervisedFacetEvaluator:
    """Train supervised classifiers on the pipeline's facet labels and report
    stratified-CV macro/micro-F1, per-class P/R/F1, confusion matrix, ablation
    and CV robustness. NB: this measures how learnable / internally consistent
    the labels are (weak supervision) - report it as such. For human-accuracy
    F1, use a corrected gold file via FacetBenchmark.gold_eval."""

    FACETS = ['research_type', 'contribution_type', 'evaluation_method']

    def __init__(self, df, use_embeddings=True, min_per_class=3, max_splits=5, random_state=42):
        self.df = df.reset_index(drop=True)
        self.texts = _build_text(self.df)
        self.use_embeddings = use_embeddings
        self.min_per_class = min_per_class
        self.max_splits = max_splits
        self.rs = random_state
        self._tfidf = None
        self._emb = None

    def _tfidf_feats(self):
        if self._tfidf is None:
            from sklearn.feature_extraction.text import TfidfVectorizer
            self._tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2,
                                          max_features=5000, stop_words='english'
                                          ).fit_transform(self.texts)
        return self._tfidf

    def _emb_feats(self):
        if self._emb is None:
            try:
                from sentence_transformers import SentenceTransformer
                model = SentenceTransformer('all-MiniLM-L6-v2')
                self._emb = np.asarray(model.encode(self.texts, show_progress_bar=False))
            except Exception as e:
                print(f"   (embeddings unavailable: {e}; using TF-IDF only)")
                self._emb = False
        return self._emb

    def _features(self, kind):
        from scipy.sparse import hstack, csr_matrix
        if kind == 'tfidf':
            return self._tfidf_feats()
        emb = self._emb_feats()
        if emb is False:
            return None
        if kind == 'embeddings':
            return csr_matrix(emb)
        if kind == 'combined':
            return hstack([self._tfidf_feats(), csr_matrix(emb)]).tocsr()

    def _cv_eval(self, X, y):
        from sklearn.svm import LinearSVC
        from sklearn.linear_model import SGDClassifier, LogisticRegression
        from sklearn.model_selection import StratifiedKFold, cross_val_predict
        from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix
        y = np.asarray(y)
        vc = pd.Series(y).value_counts()
        keep = vc[vc >= self.min_per_class].index.tolist()
        mask = np.isin(y, keep)
        Xk, yk = X[mask], y[mask]
        dropped = vc[vc < self.min_per_class].to_dict()
        if len(set(yk)) < 2:
            return None
        k = max(2, min(self.max_splits, int(pd.Series(yk).value_counts().min())))
        models = {'LinearSVC': LinearSVC(class_weight='balanced'),
                  'SGD': SGDClassifier(loss='hinge', class_weight='balanced', random_state=self.rs),
                  'LogReg': LogisticRegression(max_iter=2000, class_weight='balanced')}
        best = None
        for name, mdl in models.items():
            skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=self.rs)
            try:
                yp = cross_val_predict(mdl, Xk, yk, cv=skf)
            except Exception:
                continue
            macro = f1_score(yk, yp, average='macro', zero_division=0)
            if best is None or macro > best['macro_f1']:
                labels = sorted(set(yk))
                p, r, f, s = precision_recall_fscore_support(yk, yp, labels=labels, average=None, zero_division=0)
                best = {'model': name, 'k_folds': k, 'n_evaluated': int(mask.sum()),
                        'classes_dropped_sparse': dropped,
                        'macro_f1': round(float(macro), 4),
                        'micro_f1': round(float(f1_score(yk, yp, average='micro', zero_division=0)), 4),
                        'weighted_f1': round(float(f1_score(yk, yp, average='weighted', zero_division=0)), 4),
                        'per_class': {lab: {'precision': round(float(pp), 4), 'recall': round(float(rr), 4),
                                            'f1': round(float(ff), 4), 'support': int(ss)}
                                      for lab, pp, rr, ff, ss in zip(labels, p, r, f, s)},
                        'confusion': {'labels': labels,
                                      'matrix': confusion_matrix(yk, yp, labels=labels).tolist()}}
        return best

    def _cv_robustness(self, X, y, model_name):
        from sklearn.svm import LinearSVC
        from sklearn.linear_model import SGDClassifier, LogisticRegression
        from sklearn.model_selection import StratifiedKFold
        from sklearn.metrics import f1_score
        y = np.asarray(y)
        vc = pd.Series(y).value_counts()
        keep = vc[vc >= self.min_per_class].index.tolist()
        mask = np.isin(y, keep)
        Xk, yk = X[mask], y[mask]
        k = max(2, min(self.max_splits, int(pd.Series(yk).value_counts().min())))
        mdl = {'LinearSVC': LinearSVC(class_weight='balanced'),
               'SGD': SGDClassifier(loss='hinge', class_weight='balanced', random_state=self.rs),
               'LogReg': LogisticRegression(max_iter=2000, class_weight='balanced')}[model_name]
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=self.rs)
        fold = []
        for tr, te in skf.split(Xk, yk):
            mdl.fit(Xk[tr], yk[tr])
            fold.append(f1_score(yk[te], mdl.predict(Xk[te]), average='macro', zero_division=0))
        return {'mean_macro_f1': round(float(np.mean(fold)), 4),
                'std_macro_f1': round(float(np.std(fold)), 4),
                'folds': [round(float(x), 4) for x in fold]}

    def run(self):
        out = {}
        Xtfidf = self._features('tfidf')
        emb_ok = self.use_embeddings and self._emb_feats() is not False
        for facet in self.FACETS:
            if facet not in self.df.columns:
                continue
            y = self.df[facet].values
            Xmain = self._features('combined') if emb_ok else Xtfidf
            best = self._cv_eval(Xmain, y)
            if best is None:
                out[facet] = {'error': 'insufficient data'}
                continue
            best['cv_robustness'] = self._cv_robustness(Xmain, y, best['model'])
            ablation = {}
            for kind in (['tfidf', 'embeddings', 'combined'] if emb_ok else ['tfidf']):
                Xk = self._features(kind)
                if Xk is None:
                    continue
                b = self._cv_eval(Xk, y)
                if b:
                    ablation[kind] = b['macro_f1']
            best['ablation_macro_f1'] = ablation
            best['features'] = 'tfidf+embeddings' if emb_ok else 'tfidf'
            out[facet] = best
        return out


class FacetBenchmark:
    PAPER_2023 = {
        'research_type': {'Solution Proposal':89,'Validation Research':62,'Evaluation Research':19,
                          'Experience Papers':13,'Philosophical Papers':3,'Opinion Papers':1},
        'contribution_type': {'Process':52,'Tool':44,'Empirical Insights':41,'Method':31,'Model':19,'Metric':3},
        'evaluation_method': {'Example':71,'Case Study':55,'Experiment':31,'Experience Report':14,
                              'Discussion':11,'Rigorous Analysis':3},
    }

    @classmethod
    def distribution(cls, df):
        out = {}
        for col, ref in cls.PAPER_2023.items():
            if col not in df.columns:
                continue
            pred = df[col].value_counts().to_dict()
            pt, rt = sum(pred.values()) or 1, sum(ref.values())
            mae = sum(abs(pred.get(k,0)/pt - ref.get(k,0)/rt) for k in (set(pred)|set(ref)))/len(ref)
            common = set(pred) & set(ref); corr = 0.0
            if len(common) >= 2:
                pv = np.array([pred[k]/pt for k in common]); rv = np.array([ref[k]/rt for k in common])
                d = np.linalg.norm(pv)*np.linalg.norm(rv)
                corr = float(np.dot(pv, rv)/d) if d > 1e-9 else 0.0
            out[col] = {'mae': round(mae,4), 'correlation': round(corr,4),
                        'benchmark_score': round(1-mae,4),
                        'pipeline_pct': {k: round(100*pred.get(k,0)/pt,1) for k in ref},
                        'paper_2023_pct': {k: round(100*ref[k]/rt,1) for k in ref}}
        return out

    @staticmethod
    def gold_eval(df, gold):
        from sklearn.metrics import (precision_recall_fscore_support, f1_score,
                                      confusion_matrix, classification_report)
        merged = df.merge(gold, on='paper_id', suffixes=('_pred', '_true'))
        res = {}
        for facet in ['research_type','contribution_type','evaluation_method']:
            yp, yt = merged[f'{facet}_pred'], merged[f'{facet}_true']
            labels = sorted(set(yt) | set(yp))
            p, r, f, s = precision_recall_fscore_support(yt, yp, labels=labels, average=None, zero_division=0)
            res[facet] = {'n': int(len(merged)),
                          'macro_f1': round(float(f1_score(yt, yp, average='macro', zero_division=0)), 4),
                          'micro_f1': round(float(f1_score(yt, yp, average='micro', zero_division=0)), 4),
                          'per_class': {l:{'precision':round(float(pp),4),'recall':round(float(rr),4),
                                           'f1':round(float(ff),4),'support':int(ss)}
                                        for l,pp,rr,ff,ss in zip(labels,p,r,f,s)},
                          'confusion': {'labels':labels,
                                        'matrix':confusion_matrix(yt,yp,labels=labels).tolist()},
                          'report': classification_report(yt, yp, labels=labels, zero_division=0)}
        return res


class TimingBenchmark:
    MANUAL_SINGLE_MIN = 107.0
    MANUAL_DUAL_MIN = 172.0
    @classmethod
    def report(cls, seconds_per_study):
        auto = seconds_per_study/60.0
        red = 100.0*(cls.MANUAL_SINGLE_MIN-auto)/cls.MANUAL_SINGLE_MIN
        return {'automated_min_per_study':round(auto,3),'manual_single_min':cls.MANUAL_SINGLE_MIN,
                'manual_dual_min':cls.MANUAL_DUAL_MIN,'reduction_vs_single_pct':round(red,1),
                'meets_50pct_target':bool(red>=50.0)}


def make_gold_template(df, path):
    cols = ['paper_id','title','research_type','contribution_type','evaluation_method']
    df[[c for c in cols if c in df.columns]].to_csv(path, index=False)
    print(f"📝 Gold template (pre-filled with predictions) -> {path}")
    print("   Review each row, CORRECT wrong labels, then pass it as gold_csv=.")


def evaluate_results(df, output_dir='.', use_embeddings=True, gold_csv=None, seconds_per_study=None):
    """Run supervised F1 + distribution benchmark (+ optional gold F1 + timing)
    and write evaluation_metrics.json and per-facet CSVs for the dissertation."""
    os.makedirs(output_dir, exist_ok=True)
    metrics = {'n_papers': int(len(df))}

    print("\n📊 Running supervised cross-validated F1 evaluation …")
    sup = SupervisedFacetEvaluator(df, use_embeddings=use_embeddings).run()
    metrics['supervised_cv'] = sup
    for facet, m in sup.items():
        if 'per_class' in m:
            pd.DataFrame([{'class':c, **v} for c,v in m['per_class'].items()]
                         ).to_csv(os.path.join(output_dir, f'supervised_{facet}.csv'), index=False)
            cm = m['confusion']
            pd.DataFrame(cm['matrix'], index=cm['labels'], columns=cm['labels']
                         ).to_csv(os.path.join(output_dir, f'confusion_{facet}.csv'))
            if m.get('ablation_macro_f1'):
                pd.DataFrame([m['ablation_macro_f1']]
                             ).to_csv(os.path.join(output_dir, f'ablation_{facet}.csv'), index=False)

    dist = FacetBenchmark.distribution(df)
    metrics['distribution_benchmark'] = dist
    for facet, m in dist.items():
        pd.DataFrame([{'category':c,'paper_2023_pct':m['paper_2023_pct'][c],
                       'pipeline_pct':m['pipeline_pct'][c]} for c in m['paper_2023_pct']]
                     ).to_csv(os.path.join(output_dir, f'benchmark_{facet}.csv'), index=False)

    if gold_csv and os.path.exists(gold_csv):
        ge = FacetBenchmark.gold_eval(df, pd.read_csv(gold_csv))
        metrics['gold_evaluation'] = {f:{k:v for k,v in m.items() if k!='report'} for f,m in ge.items()}
    else:
        make_gold_template(df, os.path.join(output_dir, 'gold_labels_template.csv'))
        metrics['gold_evaluation'] = 'Not run. Correct gold_labels_template.csv and pass gold_csv=.'

    if seconds_per_study is not None:
        metrics['timing'] = TimingBenchmark.report(seconds_per_study)

    with open(os.path.join(output_dir, 'evaluation_metrics.json'), 'w') as fh:
        json.dump(metrics, fh, indent=2)

    print("\n" + "="*70)
    print("SUPERVISED CROSS-VALIDATED F1 (trained on pipeline labels)")
    print("="*70)
    for facet, m in sup.items():
        if 'macro_f1' in m:
            print(f"  {facet:18s} macroF1={m['macro_f1']:.3f}  microF1={m['micro_f1']:.3f}  "
                  f"weightedF1={m['weighted_f1']:.3f}  [{m['model']}, {m['features']}, {m['k_folds']}-fold]")
    print("\n" + "="*70)
    print("DISTRIBUTION BENCHMARK vs Binamungu & Maro (2023)")
    print("="*70)
    for facet, m in dist.items():
        print(f"  {facet:18s} benchmark={m['benchmark_score']:.3f}  corr={m['correlation']:.3f}")
    print(f"\n💾 All evaluation artefacts saved to {output_dir}")
    return metrics

## Step 18 - Figure generation

Every figure the dissertation uses. Figures whose inputs are missing are skipped with a
printed note rather than raising.

In [18]:
# ============================================================================
# FIGURE GENERATION — every figure used in the dissertation
# ----------------------------------------------------------------------------
# Produces all 14 figures, written as 300-dpi PNGs to <output_dir>/figs:
#
#   Conceptual diagrams (Chapters 2-4) — schematic, no data needed:
#       fig2_1_sms_process   five-step systematic-mapping process
#       fig2_2_conceptual    conceptual framework of the approach
#       fig3_1_lifecycle     data-science lifecycle followed by the study
#       fig4_1_pipeline      six-stage system architecture
#       fig4_2_ner           named-entity-recognition example sentence
#
#   Data figures (Chapter 5) — driven by the pipeline output + metrics:
#       fig5_1_dist          distribution across the three facets
#       fig5_2_f1            macro vs micro F1 against published automation baselines
#       fig5_3_cm            confusion matrix (research-type facet)
#       fig5_4_topics        BERTopic theme sizes + outliers
#       fig5_5_bubble        systematic map (contribution x research type)
#       fig5_6_time          per-study time (real if timing supplied, else placeholder)
#       fig5_7_years         primary-study length distribution
#       fig5_8_prf           macro/micro/weighted F1 by facet
#       fig5_9_ablation      TF-IDF vs embeddings vs combined
#
# The data figures read the in-memory results DataFrame and the metrics dict
# returned by evaluate_results(), so they always match the numbers in the
# document. fig5_6_time draws a real chart when metrics['timing'] is present
# (set seconds_per_study in evaluate_results), otherwise a labelled placeholder.
# ============================================================================

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# ---- shared palette ---------------------------------------------------------
_BLUE="#2E5A88"; _LBLUE="#7FA8D0"; _FILL="#D9E4F0"; _GREY="#888"; _GREYF="#E2E2E2"
_ACC="#C0504D"; _GREEN="#4F8A5B"; _GREENF="#D7E8DA"; _ORANGE="#D08A3E"; _ORANGEF="#F2DEC2"
_FACETS=["research_type","contribution_type","evaluation_method"]
_LBL=["Research\ntype","Contribution\ntype","Evaluation\nmethod"]


def _fig_style():
    plt.rcParams.update({"font.family":"serif","font.serif":["DejaVu Serif"],
                         "font.size":9,"axes.edgecolor":"#333","savefig.dpi":300})

def _box(ax,x,y,w,h,text,fc=_FILL,ec=_BLUE,bold=False,fs=11,tc="#1a1a1a"):
    """Rounded box with centred text — used by the conceptual diagrams."""
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=0.02,rounding_size=0.06",
                                linewidth=1.6,edgecolor=ec,facecolor=fc,zorder=2))
    ax.text(x+w/2,y+h/2,text,ha="center",va="center",fontsize=fs,zorder=3,
            weight="bold" if bold else "normal",color=tc)

def _arrow(ax,x1,y1,x2,y2):
    ax.add_patch(FancyArrowPatch((x1,y1),(x2,y2),arrowstyle="-|>",
                                 mutation_scale=16,linewidth=1.8,color=_BLUE,zorder=1))

def _save(fig,name,out_dir):
    path=os.path.join(out_dir,name)
    fig.savefig(path,bbox_inches="tight"); plt.close(fig)
    print(f"   🖼  {name}")


# ----------------------------------------------------------------------------
# Conceptual diagrams (Chapters 2-4)
# ----------------------------------------------------------------------------
def _fig2_1_sms_process(out_dir):
    fig,ax=plt.subplots(figsize=(11,3.1)); ax.axis("off"); ax.set_xlim(0,11); ax.set_ylim(0,3)
    steps=[("Definition of\nResearch\nQuestions","Review\nScope"),("Conduct\nSearch","All\nPapers"),
           ("Screening\nof Papers","Relevant\nPapers"),("Keywording\nof Abstracts","Classification\nScheme"),
           ("Data Extraction\n& Mapping","Systematic\nMap")]
    w,h,gap=1.8,1.5,0.35; x=0.15
    for i,(t,sub) in enumerate(steps):
        last=(i==len(steps)-1)
        _box(ax,x,1.0,w,h,t,fc=_ORANGEF if last else _FILL,ec=_ORANGE if last else _BLUE,bold=last,fs=10.5)
        ax.text(x+w/2,0.6,sub,ha="center",va="center",fontsize=9,style="italic",color="#555")
        if i<len(steps)-1: _arrow(ax,x+w+0.04,1.75,x+w+gap-0.04,1.75)
        x+=w+gap
    _save(fig,"fig2_1_sms_process.png",out_dir)

def _fig2_2_conceptual(out_dir):
    fig,ax=plt.subplots(figsize=(10,6.4)); ax.axis("off"); ax.set_xlim(0,10); ax.set_ylim(0,9.4)
    _box(ax,3.1,8.3,3.8,0.95,"Primary studies (PDF)\nfrom an existing SMS baseline",fc=_GREYF,ec=_GREY)
    _box(ax,3.1,6.9,3.8,0.85,"Text extraction & preprocessing")
    _box(ax,3.1,5.4,3.8,0.95,"Document representation\n(TF-IDF + sentence embeddings)")
    _box(ax,0.3,3.6,4.2,1.0,"Supervised facet classification\n(research / contribution / evaluation)",fs=10)
    _box(ax,5.5,3.6,4.2,1.0,"Transformer topic modelling\n(thematic categories + outliers)",fs=10)
    _box(ax,3.0,2.0,4.0,0.95,"Classified portfolio &\nvisual systematic map",fc=_ORANGEF,ec=_ORANGE,bold=True)
    _box(ax,3.0,0.5,4.0,0.9,"Evaluation vs. manual baseline\n(P, R, macro/micro F1, time)",fc=_GREYF,ec=_GREY,fs=10)
    _arrow(ax,5.0,8.25,5.0,7.78); _arrow(ax,5.0,6.85,5.0,6.38)
    _arrow(ax,4.6,5.35,2.6,4.62); _arrow(ax,5.4,5.35,7.4,4.62)
    _arrow(ax,2.6,3.55,4.4,2.98); _arrow(ax,7.4,3.55,5.6,2.98); _arrow(ax,5.0,1.95,5.0,1.42)
    _save(fig,"fig2_2_conceptual.png",out_dir)

def _fig3_1_lifecycle(out_dir):
    fig,ax=plt.subplots(figsize=(11,3.0)); ax.axis("off"); ax.set_xlim(0,11); ax.set_ylim(0,3)
    steps=["Business &\nprocess\nunderstanding","Data\nacquisition","Data\npreprocessing",
           "Data\nanalysis","Modelling","Deployment &\nevaluation"]
    w,h,gap=1.55,1.4,0.18; x=0.1
    for i,t in enumerate(steps):
        _box(ax,x,0.9,w,h,t,fs=10)
        if i<len(steps)-1: _arrow(ax,x+w,1.6,x+w+gap,1.6)
        x+=w+gap
    _save(fig,"fig3_1_lifecycle.png",out_dir)

def _fig4_1_pipeline(out_dir):
    fig,ax=plt.subplots(figsize=(11,7.6)); ax.axis("off"); ax.set_xlim(0,11); ax.set_ylim(0,9.6)
    _box(ax,3.6,8.6,3.8,0.8,"PDF primary studies",fc=_GREYF,ec=_GREY)
    _box(ax,3.6,7.4,3.8,0.85,"1. Document ingestion &\ntext extraction (PyMuPDF)")
    _box(ax,3.6,6.1,3.8,0.85,"2. Preprocessing &\nstructure handling")
    _box(ax,3.6,4.8,3.8,0.85,"3. Representation\n(TF-IDF + SBERT)")
    _box(ax,0.2,2.9,3.3,1.05,"4a. Metadata extraction\n& Named Entity\nRecognition",fs=9.5)
    _box(ax,3.85,2.9,3.3,1.05,"4b. Facet classifiers\n(SVM / SGD /\nensembles)",fs=9.5)
    _box(ax,7.5,2.9,3.3,1.05,"4c. BERTopic\nthematic modelling\n+ outliers",fs=9.5)
    _box(ax,3.6,1.4,3.8,0.9,"5. Portfolio assembly\n(classified studies)",fc=_GREENF,ec=_GREEN)
    _box(ax,3.6,0.1,3.8,0.9,"6. Map generation\n(bubble plot, bar charts)",fc=_ORANGEF,ec=_ORANGE,bold=True)
    _box(ax,7.7,0.1,3.2,0.9,"Evaluation vs. manual\nbaseline (P,R,F1,time)",fc=_GREYF,ec=_GREY,fs=9.5)
    for y1,y2 in [(8.55,8.29),(7.35,7.0),(6.05,5.7)]: _arrow(ax,5.5,y1,5.5,y2)
    _arrow(ax,4.6,4.75,1.9,3.99); _arrow(ax,5.5,4.75,5.5,3.99); _arrow(ax,6.4,4.75,9.1,3.99)
    _arrow(ax,1.9,2.85,4.6,2.32); _arrow(ax,5.5,2.85,5.5,2.32); _arrow(ax,9.1,2.85,6.4,2.32)
    _arrow(ax,5.5,1.35,5.5,1.02); _arrow(ax,7.4,0.55,7.65,0.55)
    _save(fig,"fig4_1_pipeline.png",out_dir)

def _fig4_2_ner(out_dir):
    fig,ax=plt.subplots(figsize=(11,2.9)); ax.axis("off"); ax.set_xlim(0,11); ax.set_ylim(0,3)
    tokens=[("This",None),("paper",None),("proposes",None),("a",None),("tool",("CONTRIB",_ORANGE)),
            ("evaluated",None),("through",None),("a",None),("case study",("EVAL",_GREEN)),
            ("on",None),("BDD",("THEME",_BLUE)),(".",None)]
    x=0.3
    for tok,ent in tokens:
        wd=0.42+0.11*len(tok)
        if ent:
            lab,col=ent
            ax.add_patch(FancyBboxPatch((x,1.35),wd,0.55,boxstyle="round,pad=0.02",
                        facecolor=col,edgecolor=col,zorder=2))
            ax.text(x+wd/2,1.625,tok,ha="center",va="center",color="white",weight="bold",fontsize=11,zorder=3)
            ax.text(x+wd/2,1.05,lab,ha="center",va="center",color=col,weight="bold",fontsize=8.5)
        else:
            ax.text(x+wd/2,1.625,tok,ha="center",va="center",fontsize=11,color="#222")
        x+=wd+0.25
    _save(fig,"fig4_2_ner.png",out_dir)


# ----------------------------------------------------------------------------
# Data figures (Chapter 5) — df = results DataFrame, M = metrics dict
# ----------------------------------------------------------------------------
def _fig5_1_dist(df,out_dir):
    fig,ax=plt.subplots(1,3,figsize=(7.2,2.5))
    def barh(a,s,t):
        vc=s.value_counts(); y=np.arange(len(vc))
        a.barh(y,vc.values,color=_BLUE,height=0.62); a.set_yticks(y); a.set_yticklabels(vc.index,fontsize=7)
        a.invert_yaxis(); a.set_title(t,fontsize=8.5,weight="bold")
        for i,v in enumerate(vc.values): a.text(v+0.5,i,str(v),va="center",fontsize=7)
        a.set_xlim(0,max(vc.values)*1.18); [a.spines[k].set_visible(False) for k in["top","right"]]; a.tick_params(length=0)
    barh(ax[0],df["research_type"],"Research type"); barh(ax[1],df["contribution_type"],"Contribution type")
    barh(ax[2],df["evaluation_method"],"Evaluation method"); fig.tight_layout(pad=0.6); _save(fig,"fig5_1_dist.png",out_dir)

def _fig5_2_f1(M,out_dir):
    sup=M["supervised_cv"]; macro=[sup[f]["macro_f1"] for f in _FACETS]; micro=[sup[f]["micro_f1"] for f in _FACETS]
    x=np.arange(3); w=0.36; fig,a=plt.subplots(figsize=(6.2,3.2))
    a.bar(x-w/2,macro,w,label="Macro-F1",color=_BLUE); a.bar(x+w/2,micro,w,label="Micro-F1",color=_LBLUE)
    # directional reference (no fixed pass/fail target): the range published
    # automated SLR/SMS data-extraction studies report (F1 ~0.49-0.73)
    a.axhspan(0.49,0.73,color=_GREY,alpha=0.12,zorder=0)
    a.text(2.44,0.61,"published\nautomation\nbaselines",color="#666",fontsize=6.2,ha="center",va="center")
    for i,v in enumerate(macro): a.text(i-w/2,v+0.01,f"{v:.2f}",ha="center",fontsize=7.5)
    for i,v in enumerate(micro): a.text(i+w/2,v+0.01,f"{v:.2f}",ha="center",fontsize=7.5)
    a.set_xticks(x); a.set_xticklabels(_LBL,fontsize=8); a.set_ylim(0,1.0); a.set_ylabel("F1-score")
    a.set_title("F1 by facet vs published automation baselines",fontsize=9,weight="bold")
    a.legend(fontsize=8,frameon=False); [a.spines[s].set_visible(False) for s in["top","right"]]; a.tick_params(length=0)
    fig.tight_layout(); _save(fig,"fig5_2_f1.png",out_dir)

def _fig5_3_cm(M,out_dir):
    cm=M["supervised_cv"]["research_type"]["confusion"]; mat=np.array(cm["matrix"]); labs=[l.replace(" ","\n") for l in cm["labels"]]
    fig,a=plt.subplots(figsize=(4.6,4.0)); a.imshow(mat,cmap="Blues")
    a.set_xticks(range(len(labs))); a.set_xticklabels(labs,fontsize=7); a.set_yticks(range(len(labs))); a.set_yticklabels(labs,fontsize=7)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            a.text(j,i,mat[i,j],ha="center",va="center",fontsize=9,color="white" if mat[i,j]>mat.max()*0.5 else "#222")
    a.set_xlabel("Predicted",fontsize=8); a.set_ylabel("Actual (pipeline label)",fontsize=8)
    a.set_title("Confusion matrix — research type",fontsize=9,weight="bold"); fig.tight_layout(); _save(fig,"fig5_3_cm.png",out_dir)

def _fig5_4_topics(df,out_dir):
    if "bertopic_topic_id" not in df.columns: return
    vc=df["bertopic_topic_id"].value_counts().sort_index()
    sizes=[]; names=[]; cols=[]
    nm={-1:"Outliers\n(no clear topic)"}
    for tid in vc.index:
        if tid==-2: continue
        sizes.append(int(vc[tid])); names.append(nm.get(int(tid),f"Topic {int(tid)}")); cols.append(_GREY if tid<0 else _BLUE)
    fig,a=plt.subplots(figsize=(6.0,3.3)); a.bar(range(len(sizes)),sizes,color=cols,width=0.6)
    for i,v in enumerate(sizes): a.text(i,v+0.6,str(v),ha="center",fontsize=9)
    a.set_xticks(range(len(sizes))); a.set_xticklabels(names,fontsize=8); a.set_ylabel("Number of papers")
    a.set_title(f"Thematic clusters discovered by BERTopic (n = {len(df)})",fontsize=9,weight="bold")
    a.set_ylim(0,max(sizes)*1.2 if sizes else 1); [a.spines[s].set_visible(False) for s in["top","right"]]; a.tick_params(length=0)
    fig.tight_layout(); _save(fig,"fig5_4_topics.png",out_dir)

def _fig5_5_bubble(df,out_dir):
    ct=pd.crosstab(df["contribution_type"],df["research_type"])
    order=["Solution Proposal","Evaluation Research","Validation Research","Experience Papers","Philosophical Papers","Opinion Papers"]
    ct=ct.reindex(columns=[c for c in order if c in ct.columns])
    rows=ct.index.tolist(); cols2=ct.columns.tolist(); fig,a=plt.subplots(figsize=(6.6,3.9))
    for i,r in enumerate(rows):
        for j,c in enumerate(cols2):
            n=ct.loc[r,c]
            if n>0:
                a.scatter(j,i,s=n*60,color=_LBLUE,edgecolor=_BLUE,alpha=0.85,zorder=3)
                a.text(j,i,str(int(n)),ha="center",va="center",fontsize=7.5,zorder=4)
    a.set_xticks(range(len(cols2))); a.set_xticklabels([c.replace(" ","\n") for c in cols2],fontsize=7.5)
    a.set_yticks(range(len(rows))); a.set_yticklabels(rows,fontsize=7.5); a.set_xlim(-0.6,len(cols2)-0.4); a.set_ylim(len(rows)-0.4,-0.6)
    a.set_xlabel("Research type",fontsize=8.5,weight="bold"); a.set_ylabel("Contribution type",fontsize=8.5,weight="bold")
    a.set_title(f"Systematic map: contribution \u00D7 research type (n = {len(df)})",fontsize=9,weight="bold")
    a.grid(True,ls=":",color="#ccc",zorder=0); fig.tight_layout(); _save(fig,"fig5_5_bubble.png",out_dir)


def _fig_cluster_sizes(df, out_dir):
    """K-Means baseline cluster sizes with their distinctive keyword names."""
    if 'cluster_id' not in df.columns:
        return
    vc = df['cluster_id'].value_counts().sort_index()
    names = []
    for c in vc.index:
        kw = df.loc[df.cluster_id == c, 'cluster_keywords'].iloc[0] if 'cluster_keywords' in df.columns else ''
        kw_short = ', '.join(str(kw).split(', ')[:3])
        names.append(f"C{int(c)}:\n{kw_short}")
    fig, a = plt.subplots(figsize=(6.4, 3.2))
    a.bar(range(len(vc)), vc.values, color=_BLUE, width=0.6)
    for i, v in enumerate(vc.values):
        a.text(i, v + 0.6, str(int(v)), ha="center", fontsize=9)
    a.set_xticks(range(len(vc))); a.set_xticklabels(names, fontsize=7)
    a.set_ylabel("Number of papers")
    a.set_title(f"K-Means baseline clusters (n = {len(df)})", fontsize=9, weight="bold")
    a.set_ylim(0, vc.max() * 1.2)
    [a.spines[sp].set_visible(False) for sp in ("top", "right")]; a.tick_params(length=0)
    fig.tight_layout(); _save(fig, "fig_cluster_sizes.png", out_dir)


def compare_methods(df, out_dir='.'):
    """BASELINE (K-Means clusters) vs PRIMARY (BERTopic themes) — the comparison
    the methodology prescribes (TF-IDF/K-Means baseline, SBERT/BERTopic primary).
    Reports group counts, sizes, outliers, and the Adjusted Rand Index agreement
    between the two groupings; saves method_comparison.json and a figure."""
    from sklearn.metrics import adjusted_rand_score
    res = {}
    if 'cluster_id' in df.columns:
        vc = df['cluster_id'].value_counts().sort_index()
        res['kmeans_baseline'] = {'n_groups': int(vc.size),
                                  'sizes': {int(k): int(v) for k, v in vc.items()}}
    if 'bertopic_topic_id' in df.columns:
        t = df['bertopic_topic_id']
        res['bertopic_primary'] = {'n_themes': int(t[t >= 0].nunique()),
                                   'outliers': int((t == -1).sum()),
                                   'outlier_pct': round(100.0 * (t == -1).mean(), 1),
                                   'sizes': {int(k): int(v) for k, v in
                                             t.value_counts().sort_index().items()}}
    if 'cluster_id' in df.columns and 'bertopic_topic_id' in df.columns:
        m = df[df['bertopic_topic_id'] >= 0]
        if len(m) > 1:
            res['agreement_ARI'] = round(float(
                adjusted_rand_score(m['cluster_id'], m['bertopic_topic_id'])), 4)
    with open(os.path.join(out_dir, 'method_comparison.json'), 'w') as fh:
        json.dump(res, fh, indent=2)

    # side-by-side sizes figure
    if 'kmeans_baseline' in res and 'bertopic_primary' in res:
        fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.0))
        kb = res['kmeans_baseline']['sizes']
        ax[0].bar(range(len(kb)), list(kb.values()), color=_GREY, width=0.6)
        ax[0].set_xticks(range(len(kb))); ax[0].set_xticklabels([f"C{k}" for k in kb], fontsize=8)
        ax[0].set_title("Baseline: K-Means clusters", fontsize=9, weight="bold")
        bp = res['bertopic_primary']['sizes']
        cols = [_GREY if k == -1 else _BLUE for k in bp]
        ax[1].bar(range(len(bp)), list(bp.values()), color=cols, width=0.6)
        ax[1].set_xticks(range(len(bp)))
        ax[1].set_xticklabels(["Outl." if k == -1 else f"T{k}" for k in bp], fontsize=8)
        ax[1].set_title("Primary: BERTopic themes", fontsize=9, weight="bold")
        for a in ax:
            a.set_ylabel("Papers"); [a.spines[sp].set_visible(False) for sp in ("top", "right")]
            a.tick_params(length=0)
        fig.suptitle("")
        fig.tight_layout(); _save(fig, "fig_method_comparison.png", out_dir)

    print("\n" + "=" * 60)
    print("METHOD COMPARISON — baseline vs primary")
    print("=" * 60)
    if 'kmeans_baseline' in res:
        print(f"  K-Means baseline : {res['kmeans_baseline']['n_groups']} groups "
              f"{list(res['kmeans_baseline']['sizes'].values())}")
    if 'bertopic_primary' in res:
        bp = res['bertopic_primary']
        print(f"  BERTopic primary : {bp['n_themes']} themes "
              f"+ {bp['outliers']} outliers ({bp['outlier_pct']}%)")
    if 'agreement_ARI' in res:
        print(f"  Agreement (ARI, non-outliers): {res['agreement_ARI']}"
              f"   [0=independent groupings, 1=identical]")
    return res



def compare_kmeans_hdbscan(doc_vecs, texts, out_dir='.',
                           min_cluster_size=8, min_samples=1,
                           n_components=5, random_state=42):
    """ALGORITHM COMPARISON: K-Means vs HDBSCAN on the SAME UMAP-reduced
    vectors (mirroring BERTopic's pipeline), so only the algorithm differs.
    K-Means: silhouette-chosen k, assigns every paper.
    HDBSCAN: density-based, chooses its own k, flags outliers.
    Reports n_clusters, outliers, silhouette, sizes, distinctive keyword names,
    and ARI agreement; saves kmeans_vs_hdbscan.json + fig_kmeans_vs_hdbscan.png.
    NB: HDBSCAN silhouette is computed on non-outliers only (flattering) — the
    caveat is recorded in the JSON."""
    try:
        import umap, hdbscan
        from sklearn.cluster import KMeans
        from sklearn.metrics import silhouette_score, adjusted_rand_score
        from sklearn.feature_extraction.text import TfidfVectorizer
    except Exception as e:
        print(f"(algorithm comparison skipped — missing lib: {e})"); return None

    X = np.asarray(doc_vecs)
    U = umap.UMAP(n_neighbors=15, n_components=n_components, min_dist=0.0,
                  metric='cosine', random_state=random_state).fit_transform(X)

    # ---- K-Means with silhouette-chosen k in the reduced space ----
    best = None
    for k in range(2, min(16, len(texts) - 1)):
        km = KMeans(n_clusters=k, random_state=random_state, n_init=10).fit(U)
        try:
            sil = silhouette_score(U, km.labels_)
        except Exception:
            continue
        if best is None or sil > best[1]:
            best = (k, float(sil), km.labels_)
    k_km, sil_km, lab_km = best

    # ---- HDBSCAN on the same reduced space ----
    h = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size,
                        min_samples=min_samples).fit(U)
    lab_h = h.labels_
    n_h = len(set(lab_h)) - (1 if -1 in lab_h else 0)
    n_out = int((lab_h == -1).sum())
    m = lab_h != -1
    sil_h = float(silhouette_score(U[m], lab_h[m])) if n_h >= 2 and m.sum() > n_h else None
    ari = float(adjusted_rand_score(np.asarray(lab_km)[m], lab_h[m])) if m.sum() > 1 else None

    # ---- distinctive keyword names for both groupings ----
    def _names(labels):
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=5000,
                              stop_words='english')
        M2 = vec.fit_transform(texts); terms = np.array(vec.get_feature_names_out())
        overall = np.asarray(M2.mean(axis=0)).ravel(); out = {}
        for c in sorted(set(labels)):
            if c == -1:
                continue
            cl = np.asarray(M2[np.asarray(labels) == c].mean(axis=0)).ravel()
            out[int(c)] = ", ".join(terms[(cl - overall).argsort()[::-1][:4]])
        return out

    res = {
        'space': f'UMAP-{n_components}D (cosine), shared by both algorithms',
        'kmeans':  {'k': int(k_km), 'silhouette': round(sil_km, 4),
                    'sizes': {int(c): int(v) for c, v in
                              pd.Series(lab_km).value_counts().sort_index().items()},
                    'keywords': _names(lab_km)},
        'hdbscan': {'params': {'min_cluster_size': min_cluster_size,
                               'min_samples': min_samples},
                    'n_clusters': int(n_h), 'outliers': n_out,
                    'outlier_pct': round(100.0 * n_out / len(texts), 1),
                    'silhouette_non_outliers': round(sil_h, 4) if sil_h else None,
                    'silhouette_caveat': 'computed on non-outliers only',
                    'sizes': {int(c): int(v) for c, v in
                              pd.Series(lab_h).value_counts().sort_index().items()},
                    'keywords': _names(lab_h)},
        'agreement_ARI_non_outliers': round(ari, 4) if ari is not None else None,
    }
    with open(os.path.join(out_dir, 'kmeans_vs_hdbscan.json'), 'w') as fh:
        json.dump(res, fh, indent=2)

    # ---- comparison figure ----
    fig, ax = plt.subplots(1, 2, figsize=(7.6, 3.1))
    ks = res['kmeans']['sizes']
    ax[0].bar(range(len(ks)), list(ks.values()), color=_BLUE, width=0.6)
    ax[0].set_xticks(range(len(ks))); ax[0].set_xticklabels([f"C{c}" for c in ks], fontsize=8)
    ax[0].set_title(f"K-Means (k={k_km}, sil={sil_km:.2f})", fontsize=9, weight="bold")
    hs = res['hdbscan']['sizes']
    cols = [_GREY if c == -1 else _GREEN for c in hs]
    ax[1].bar(range(len(hs)), list(hs.values()), color=cols, width=0.6)
    ax[1].set_xticks(range(len(hs)))
    ax[1].set_xticklabels(["Outl." if c == -1 else f"H{c}" for c in hs], fontsize=8)
    st = f"sil*={res['hdbscan']['silhouette_non_outliers']}" if sil_h else ""
    ax[1].set_title(f"HDBSCAN ({n_h} clusters, {n_out} outl., {st})", fontsize=9, weight="bold")
    for a in ax:
        a.set_ylabel("Papers"); [a.spines[sp].set_visible(False) for sp in ("top", "right")]
        a.tick_params(length=0)
    fig.suptitle(f"Same UMAP space — agreement ARI = {res['agreement_ARI_non_outliers']}",
                 fontsize=9, y=1.02)
    fig.tight_layout(); _save(fig, "fig_kmeans_vs_hdbscan.png", out_dir)

    print("\n" + "=" * 60)
    print("ALGORITHM COMPARISON — K-Means vs HDBSCAN (same UMAP space)")
    print("=" * 60)
    print(f"  K-Means : k={k_km}, silhouette={sil_km:.3f}, sizes={list(ks.values())}")
    print(f"  HDBSCAN : {n_h} clusters + {n_out} outliers ({res['hdbscan']['outlier_pct']}%), "
          f"silhouette(non-outliers)={res['hdbscan']['silhouette_non_outliers']}")
    print(f"  Agreement (ARI, non-outliers): {res['agreement_ARI_non_outliers']}")
    return res


def _fig5_6_time(M,out_dir):
    t=M.get("timing"); fig,a=plt.subplots(figsize=(5.0,3.0))
    if t:
        vals=[t["manual_dual_min"],t["manual_single_min"],t["automated_min_per_study"]]
        names=["Manual\n(dual)","Manual\n(single)","Automated\npipeline"]
        a.bar(range(3),vals,color=[_GREY,_LBLUE,_BLUE],width=0.6)
        for i,v in enumerate(vals): a.text(i,v+1,f"{v:.0f}",ha="center",fontsize=9)
        a.set_ylabel("Minutes per study"); a.set_xticks(range(3)); a.set_xticklabels(names,fontsize=8)
        a.set_title(f"Per-study extraction time (\u2212{t['reduction_vs_single_pct']:.0f}% vs single)",fontsize=9,weight="bold")
        [a.spines[s].set_visible(False) for s in["top","right"]]; a.tick_params(length=0)
    else:
        a.axis("off")
        a.add_patch(plt.Rectangle((0.02,0.02),0.96,0.96,fill=False,edgecolor=_GREY,ls="--",lw=1.2,transform=a.transAxes))
        a.text(0.5,0.6,"[ Figure to be completed ]",ha="center",fontsize=12,weight="bold",color=_BLUE,transform=a.transAxes)
        a.text(0.5,0.4,"Per-study extraction time:\nset seconds_per_study in evaluate_results()",ha="center",fontsize=8.5,color="#444",transform=a.transAxes)
    fig.tight_layout(); _save(fig,"fig5_6_time.png",out_dir)

def _fig5_7_years(df,out_dir):
    if "word_count" not in df.columns: return
    wc=df["word_count"].clip(upper=20000); fig,a=plt.subplots(figsize=(6.2,2.9))
    a.hist(wc,bins=20,color=_BLUE,edgecolor="white"); med=df["word_count"].median()
    a.axvline(med,color=_ACC,ls="--",lw=1); a.text(med+300,a.get_ylim()[1]*0.85,f"median \u2248 {int(med)}",color=_ACC,fontsize=7.5)
    a.set_xlabel("Paper length (words, capped at 20k)",fontsize=8.5); a.set_ylabel("Number of papers",fontsize=8.5)
    a.set_title(f"Distribution of primary-study length (n = {len(df)})",fontsize=9,weight="bold")
    [a.spines[s].set_visible(False) for s in["top","right"]]; a.tick_params(length=0); fig.tight_layout(); _save(fig,"fig5_7_years.png",out_dir)

def _fig5_8_prf(M,out_dir):
    sup=M["supervised_cv"]; mac=[sup[f]["macro_f1"] for f in _FACETS]; mic=[sup[f]["micro_f1"] for f in _FACETS]; wei=[sup[f]["weighted_f1"] for f in _FACETS]
    x=np.arange(3); w=0.26; fig,a=plt.subplots(figsize=(6.2,3.1))
    a.bar(x-w,mac,w,label="Macro-F1",color=_BLUE); a.bar(x,mic,w,label="Micro-F1",color=_LBLUE); a.bar(x+w,wei,w,label="Weighted-F1",color=_GREEN)
    a.set_xticks(x); a.set_xticklabels(_LBL,fontsize=8); a.set_ylim(0,1.0); a.set_ylabel("F1-score")
    a.set_title("Macro, micro and weighted F1 by facet",fontsize=9,weight="bold")
    a.legend(fontsize=7.5,frameon=False,ncol=3); [a.spines[s].set_visible(False) for s in["top","right"]]; a.tick_params(length=0)
    fig.tight_layout(); _save(fig,"fig5_8_prf.png",out_dir)

def _fig5_9_ablation(M,out_dir):
    sup=M["supervised_cv"]; kinds=["tfidf","embeddings","combined"]; klab=["TF\u2013IDF","Embeddings","Combined"]
    data={k:[sup[f]["ablation_macro_f1"].get(k,0) for f in _FACETS] for k in kinds}
    x=np.arange(3); w=0.26; fig,a=plt.subplots(figsize=(6.2,3.1))
    for idx,k in enumerate(kinds): a.bar(x+(idx-1)*w,data[k],w,label=klab[idx],color=[_GREY,_LBLUE,_BLUE][idx])
    a.set_xticks(x); a.set_xticklabels(_LBL,fontsize=8); a.set_ylim(0,0.8); a.set_ylabel("Macro-F1")
    a.set_title("Ablation: feature representation vs macro-F1",fontsize=9,weight="bold")
    a.legend(fontsize=7.5,frameon=False,ncol=3); [a.spines[s].set_visible(False) for s in["top","right"]]; a.tick_params(length=0)
    fig.tight_layout(); _save(fig,"fig5_9_ablation.png",out_dir)


def generate_all_figures(df=None, metrics=None, out_dir="figs",
                         results_csv=None, metrics_json=None, relabel=False):
    """Generate every dissertation figure as 300-dpi PNGs in <out_dir>.

    In-pipeline use (after evaluate_results):
        generate_all_figures(df=results_df, metrics=metrics, out_dir=f"{OUTPUT_DIR}/figs")

    Standalone use (from saved files, no PDF re-processing):
        generate_all_figures(results_csv="sms_results.csv",
                             metrics_json="evaluation_metrics.json",
                             out_dir="figs", relabel=True)

    Args:
        df          results DataFrame (facets, themes, word_count). If None, read results_csv.
        metrics     dict from evaluate_results(). If None, read metrics_json (if given).
        out_dir     destination folder for the PNGs (created if absent).
        results_csv path to sms_results.csv, used only when df is None.
        metrics_json path to evaluation_metrics.json, used only when metrics is None.
        relabel     if True, re-apply the improved classifier before plotting the
                    distribution/bubble figures (use when reading a raw CSV that
                    still holds the old skewed labels). In-pipeline df is already
                    improved, so leave False.
    """
    _fig_style()
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n🎨 Generating figures -> {out_dir}")

    # ---- conceptual diagrams (no data dependency) ----
    _fig2_1_sms_process(out_dir); _fig2_2_conceptual(out_dir); _fig3_1_lifecycle(out_dir)
    _fig4_1_pipeline(out_dir); _fig4_2_ner(out_dir)

    # ---- load data if not provided ----
    if df is None and results_csv:
        df = pd.read_csv(results_csv)
    if metrics is None and metrics_json and os.path.exists(metrics_json):
        with open(metrics_json) as fh:
            metrics = json.load(fh)
    if relabel and df is not None:
        df = relabel_with_improved_classifier(df)

    # ---- data figures ----
    if df is not None:
        has_facets = all(f in df.columns for f in _FACETS)
        if has_facets:
            _fig5_1_dist(df, out_dir); _fig5_5_bubble(df, out_dir)
        else:
            print("   (no facet columns — skipping distribution/bubble figures only)")
        # these do NOT need facets — always generate when their columns exist
        _fig5_7_years(df, out_dir)      # word-count distribution
        _fig5_4_topics(df, out_dir)     # BERTopic theme sizes
        _fig_cluster_sizes(df, out_dir) # K-Means baseline cluster sizes
    if metrics and metrics.get("supervised_cv"):
        _fig5_2_f1(metrics,out_dir); _fig5_3_cm(metrics,out_dir)
        _fig5_8_prf(metrics,out_dir); _fig5_9_ablation(metrics,out_dir)
    else:
        print("   (no supervised_cv metrics — skipping F1/CM/ablation figures)")
    _fig5_6_time(metrics or {}, out_dir)
    print("✅ Figures complete.")

## Step 19 - Run the pipeline

Clusters the papers, assigns the facets, models the themes and writes the results table.

**No PDF is opened here.** Everything comes from the dataset built in Part A, so this
step and everything after it can be re-run as often as you like while tuning. Expect
under a minute for ~120 papers, plus a one-off download of the sentence-transformer
weights the first time.

Results go to `RUN_DIR`, a fresh timestamped folder under `output/runs/`, so a new run
never overwrites an old one. `output/latest` always points at the most recent.

**Run every step above this one first.** The orchestrator also calls three helpers that
are defined further down for readability: the timing benchmark, and the two comparison
figures. Executing this cell on a fresh kernel without them raises a `NameError`.
Running the notebook top to bottom always works.

In [19]:
print("TRAINING")
pipeline = CompleteSMSPipeline(RUN_DIR, target_k=TARGET_K)
df_model, topic_model = pipeline.process_and_train(
    dataset=dataset,              # Part A output - no PDF is opened here
    save_model=True,
    embedding_model=EMBEDDING_MODEL,
    min_cluster_size=MIN_CLUSTER_SIZE,
    n_components=5,
    reduce_outliers_strategy=OUTLIER_STRATEGY,
    outlier_threshold=OUTLIER_THRESHOLD,
)
results_df = pipeline._to_dataframe()
print(f"\n{len(results_df)} papers in the results table")
results_df.head(3)

TRAINING

📚 COMPLETE SMS PIPELINE WITH BERTopic + HDBSCAN  [v3 — PyMuPDF]
📂 Loading 123 papers from the prepared dataset


Loading dataset: 100%|██████████████████████| 123/123 [00:00<00:00, 594.99it/s]


✅ Loaded 123/123 papers
   📑 Abstracts found : 123/123  (heading=105, pre_intro=16, fallback=2)
   🔤 Titles from layout: 115/123

🔵 Grouping papers by K-Means clustering (unsupervised, no rules, no labels)…


Loading weights: 100%|████████████████████| 103/103 [00:00<00:00, 10949.52it/s]


      · silhouette chose k=2 clusters (score=0.067)
        cluster  0 (n= 73): bdd, agile, development, software, tdd
        cluster  1 (n= 50): systems, mobile, microservice, integration, apps

🏷️  Assigning mapping facets (zero-shot, no labels)…
   research_type:
      Validation Research      30  (24.4%)
      Philosophical Papers     24  (19.5%)
      Solution Proposal        22  (17.9%)
      Opinion Papers           18  (14.6%)
      Experience Papers        15  (12.2%)
      Evaluation Research      14  (11.4%)
      → 44/123 flagged for review (margin < 0.25 SD)
   contribution_type:
      Tool                     26  (21.1%)
      Method                   23  (18.7%)
      Metric                   21  (17.1%)
      Process                  20  (16.3%)
      Empirical Insights       18  (14.6%)
      Model                    15  (12.2%)
      → 34/123 flagged for review (margin < 0.25 SD)
   evaluation_method:
      Case Study               27  (22.0%)
      Discussion       

...

Batches: 100%|███████████████████████████████████| 4/4 [00:03<00:00,  1.20it/s]
2026-09-20 17:16:33,291 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-09-20 17:16:33,292 - BERTopic - Dimensionality - Completed ✓
2026-09-20 17:16:33,292 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-09-20 17:16:33,296 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-09-20 17:16:33,305 - BERTopic - Probabilities - Completed ✓
2026-09-20 17:16:33,306 - BERTopic - Cluster - Completed ✓



📊 Evaluating model performance…

╔══════════════════════════════════════════════════════════════════╗
║              MODEL PERFORMANCE REPORT (v2)                      ║
╚══════════════════════════════════════════════════════════════════╝

📊 CLUSTERING METRICS:
   • Silhouette Score  : 0.0732  ⚠️ weak
   • Topics Found      : 2
   • Outliers          : 1  (0.8%)
   • Coverage          : 99.2%  (docs assigned to a topic)

🎯 TOPIC QUALITY:
   • Topic Diversity       : 0.5684  (1.0 = fully distinct)
   • Inter-topic Similarity: 0.4316
   • Avg / Min / Max size  : 61.0 / 55 / 67

📚 TOPIC COHERENCE:
   • C_v Coherence     : 0.4553
   • Interpretability  : 45.5%

🎲 CONFIDENCE STATS:
   • Mean / Median     : 0.6673 / 0.5948
   • Std Dev           : 0.2318
   • High-confidence≥0.7: 41.5% of docs

⚖️  DISTRIBUTION:
   • Gini Coefficient  : 0.0492  (0 = perfectly even)
   • Quality           : excellent
   • Dominant topic    : 54.9% of docs

📋 PER-TOPIC BREAKDOWN:
 topic_id                    

2026-09-20 17:16:35,220 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


   ✅ Heatmap   → /home/naedatatz/Desktop/Notebook/output/runs/run_20260920_171507/topic_research_heatmap.png
📊 Validation plots saved to /home/naedatatz/Desktop/Notebook/output/runs/run_20260920_171507
   🖼  fig_method_comparison.png

METHOD COMPARISON — baseline vs primary
  K-Means baseline : 2 groups [73, 50]
  BERTopic primary : 2 themes + 30 outliers (24.4%)
  Agreement (ARI, non-outliers): 0.1315   [0=independent groupings, 1=identical]
✅ Model saved to /home/naedatatz/Desktop/Notebook/output/runs/run_20260920_171507/bertopic_model
✅ CSV  → /home/naedatatz/Desktop/Notebook/output/runs/run_20260920_171507/sms_results.csv
✅ 7 distribution CSVs → /home/naedatatz/Desktop/Notebook/output/runs/run_20260920_171507/distributions
✅ Summary → /home/naedatatz/Desktop/Notebook/output/runs/run_20260920_171507/summary.txt

⏱  TIMING: 47s total for 123 studies = 0.4s/study (0.01 min vs 107 min manual → 100.0% reduction; ≥50% target met: True)

📊 FINAL RESULTS SUMMARY
✅ Processed  : 123 papers
✅

,paper_id,title,abstract,word_count,abstract_strategy,title_source,extraction_method,ocr_pages,source_file,n_sections,...,contribution_type_margin,contribution_type_needs_review,evaluation_method,evaluation_method_confidence,evaluation_method_margin,evaluation_method_needs_review,bertopic_theme,bertopic_topic_id,bertopic_confidence,is_outlier
0,P0001,A Behavior-Based Ontology for Supporting Autom...,Nowadays many software development frameworks ...,5697,heading,typography,native,0,A Behavior-Based Ontology for Supporting Autom...,5,...,0.0622,True,Rigorous Analysis,0.2754,0.2476,True,"requirements, test, user, development, software",1,0.940899,False
1,P0002,A Behavior-Driven Approach to Intent Specifica...,One of the goals of Software-Defined Networkin...,5169,heading,typography,native,0,A Behavior-Driven Approach to Intent Specifica...,2,...,0.1040,True,Discussion,0.1487,0.2363,True,"requirements, test, user, development, software",1,0.430693,False
2,P0003,A Conceptual Metamodel to Bridging Requirement...,critical vulnerability of software projects wh...,4082,heading,typography,native,0,A Conceptual Metamodel to Bridging Requirement...,2,...,1.9325,False,Case Study,0.3096,0.0435,True,"requirements, test, user, development, software",1,1.000000,False


## Step 20 - Hyperparameter tuning by grid search

Everything so far used defaults or a single-parameter sweep. This step searches the
parameter space properly, in two grids, because the pipeline has two very different
kinds of parameter.

**A. The clustering grid.** UMAP dimensions, neighbourhood size, cluster count and
algorithm, scored by silhouette in the space the clustering happens in and by C_v
coherence of the resulting cluster terms. These are unsupervised, so there is no
cross-validation to do; the grid is exhaustive and every combination is reported.

**B. The classifier grid.** A real `GridSearchCV` with stratified 5-fold
cross-validation and macro F1 as the scorer, over three model families and their
regularisation. This tunes the facet classifier used in the evaluation step.

A warning carried from earlier: the facet labels come from the zero-shot assigner, so
grid search here finds the model that best reproduces those labels. It cannot make wrong
labels right, and a large jump in F1 would mean the classifier is fitting the assigner's
quirks rather than the papers.

In [20]:
import itertools

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.svm import LinearSVC
from umap import UMAP

_FACETS = ["research_type", "contribution_type", "evaluation_method"]

CLUSTER_GRID = {
    "n_components": [2, 5],
    "n_neighbors":  [5, 11, 20],
    "k":            [3, 4, 5, 6, 7, 8,15],
}

CLASSIFIER_GRID = [
    (LogisticRegression(max_iter=3000),
     {"C": [0.1, 1.0, 10.0], "class_weight": [None, "balanced"]}),
    (LinearSVC(max_iter=5000),
     {"C": [0.1, 1.0, 10.0], "class_weight": [None, "balanced"]}),
    (RandomForestClassifier(random_state=RANDOM_STATE),
     {"n_estimators": [200, 400], "max_depth": [None, 12],
      "min_samples_leaf": [1, 3]}),
]


def grid_search_clustering(embeddings, texts, grid=CLUSTER_GRID, out_dir=None):
    """Exhaustive grid over the reduction and the cluster count."""
    out_dir = out_dir or RUN_DIR
    rows = []
    combos = list(itertools.product(grid["n_components"], grid["n_neighbors"]))
    for nd, nn in combos:
        X = UMAP(n_components=nd, n_neighbors=nn, min_dist=0.0, metric="cosine",
                 random_state=RANDOM_STATE).fit_transform(embeddings)
        for k in grid["k"]:
            lab = KMeans(k, random_state=RANDOM_STATE, n_init=10).fit_predict(X)
            if len(set(lab)) < 2:
                continue
            rows.append({
                "n_components": nd, "n_neighbors": nn, "k": k,
                "silhouette": round(float(silhouette_score(X, lab)), 4),
                "coherence_cv": round(cluster_coherence(texts, lab), 4),
                "smallest_cluster": int(np.bincount(lab).min()),
            })
    tab = pd.DataFrame(rows)
    # coherence leads, silhouette supports, and a cluster of 2 papers is useless
    usable = tab[tab.smallest_cluster >= 5].copy()
    pool = usable if len(usable) else tab.copy()
    for col in ("coherence_cv", "silhouette"):
        v = pool[col].astype(float)
        pool[col + "_n"] = (v - v.min()) / (v.max() - v.min() + 1e-9)
    pool["score"] = (0.6 * pool.coherence_cv_n + 0.4 * pool.silhouette_n).round(4)
    pool = pool.sort_values("score", ascending=False)
    tab.to_csv(os.path.join(out_dir, "gridsearch_clustering.csv"), index=False)

    print(f"A. CLUSTERING GRID - {len(tab)} combinations "
          f"({len(usable)} with every cluster >= 5 papers)")
    print(pool[["n_components", "n_neighbors", "k", "silhouette",
                "coherence_cv", "smallest_cluster", "score"]].head(6).to_string(index=False))
    best = pool.iloc[0]
    print(f"   \U0001f3c6 best: {int(best.n_components)}-D UMAP, "
          f"n_neighbors={int(best.n_neighbors)}, k={int(best.k)}")
    return tab, best


def grid_search_classifier(X, df, grid=CLASSIFIER_GRID, out_dir=None, k_folds=5):
    """A real GridSearchCV per facet, scored by cross-validated macro F1."""
    out_dir = out_dir or RUN_DIR
    cv = StratifiedKFold(k_folds, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    print(f"\nB. CLASSIFIER GRID - GridSearchCV, {k_folds}-fold, scoring macro F1")
    for facet in _FACETS:
        if facet not in df.columns:
            continue
        y = df[facet].to_numpy()
        best_overall = None
        for estimator, params in grid:
            gs = GridSearchCV(estimator, params, scoring="f1_macro", cv=cv,
                              n_jobs=1, error_score=0.0)
            gs.fit(X, y)
            rows.append({"facet": facet,
                         "model": type(estimator).__name__,
                         "best_params": str(gs.best_params_),
                         "macro_f1": round(float(gs.best_score_), 4)})
            if best_overall is None or gs.best_score_ > best_overall["macro_f1"]:
                best_overall = rows[-1]
        print(f"   {facet}:")
        for r in [r for r in rows if r["facet"] == facet]:
            mark = " <-" if r is best_overall else ""
            print(f"      {r['model']:<24} F1={r['macro_f1']:.4f}  "
                  f"{r['best_params']}{mark}")
    tab = pd.DataFrame(rows)
    tab.to_csv(os.path.join(out_dir, "gridsearch_classifier.csv"), index=False)
    return tab


_grid_clusters, _best_cluster_cfg = grid_search_clustering(
    pipeline.classifier.doc_vecs_, _model_texts)
_grid_models = grid_search_classifier(pipeline.classifier.doc_vecs_, results_df)
_grid_models

A. CLUSTERING GRID - 42 combinations (37 with every cluster >= 5 papers)
 n_components  n_neighbors  k  silhouette  coherence_cv  smallest_cluster  score
            2           11  6      0.4018        0.5228                12 0.8460
            2           11  5      0.4313        0.4975                12 0.8049
            2           11  7      0.4161        0.4877                12 0.7298
            5           11  7      0.3326        0.5254                13 0.7030
            5           20  3      0.3068        0.5246                31 0.6422
            2            5  6      0.4616        0.4398                16 0.6291
   🏆 best: 2-D UMAP, n_neighbors=11, k=6

B. CLASSIFIER GRID - GridSearchCV, 5-fold, scoring macro F1
   research_type:
      LogisticRegression       F1=0.4658  {'C': 10.0, 'class_weight': None}
      LinearSVC                F1=0.4707  {'C': 10.0, 'class_weight': None} <-
      RandomForestClassifier   F1=0.3566  {'max_depth': None, 'min_samples_leaf': 1, 

,facet,model,best_params,macro_f1
0,research_type,LogisticRegression,"{'C': 10.0, 'class_weight': None}",0.4658
1,research_type,LinearSVC,"{'C': 10.0, 'class_weight': None}",0.4707
2,research_type,RandomForestClassifier,"{'max_depth': None, 'min_samples_leaf': 1, 'n_...",0.3566
3,contribution_type,LogisticRegression,"{'C': 10.0, 'class_weight': 'balanced'}",0.4466
4,contribution_type,LinearSVC,"{'C': 1.0, 'class_weight': 'balanced'}",0.4262
5,contribution_type,RandomForestClassifier,"{'max_depth': None, 'min_samples_leaf': 1, 'n_...",0.4042
6,evaluation_method,LogisticRegression,"{'C': 1.0, 'class_weight': 'balanced'}",0.3833
7,evaluation_method,LinearSVC,"{'C': 0.1, 'class_weight': 'balanced'}",0.4058
8,evaluation_method,RandomForestClassifier,"{'max_depth': None, 'min_samples_leaf': 1, 'n_...",0.3483


## Step 21 - Save the model with pickle

Writes one `.pkl` file holding everything needed to reproduce the run or classify new
papers: the fitted K-Means model and its keywords, the BERTopic model, the results
table, and the settings used.

The sentence-transformer weights are deliberately **left out**. They are about 90 MB,
they are fetched by name from the hub, and pickling a live torch module ties the file
to one exact torch version. The encoder's *name* is stored instead and step 16 reattaches
it. This keeps the pickle small and portable.

In [21]:
def save_pipeline(pipeline, path=MODEL_PKL):
    """Pickle everything needed to reload this trained pipeline."""
    clf = pipeline.classifier
    te  = pipeline.theme_extractor
    tm  = getattr(te, "topic_model", None)

    # detach the heavy encoder from both objects before pickling, then restore
    detached = []
    for obj in (te, tm):
        if obj is not None and getattr(obj, "embedding_model", None) is not None:
            detached.append((obj, obj.embedding_model))
            obj.embedding_model = None

    bundle = {
        "format_version": 1,
        "created": datetime.now().isoformat(timespec="seconds"),
        "embedding_model_name": EMBEDDING_MODEL,
        "settings": {
            "target_k": TARGET_K, "min_cluster_size": MIN_CLUSTER_SIZE,
            "outlier_strategy": OUTLIER_STRATEGY,
            "outlier_threshold": OUTLIER_THRESHOLD,
            "random_state": RANDOM_STATE,
        },
        "kmeans_model":     getattr(clf, "kmeans_", None),
        "kmeans_k":         getattr(clf, "chosen_k_", {}),
        "kmeans_keywords":  getattr(clf, "cluster_keywords_", {}),
        "kmeans_labels":    getattr(clf, "labels_", None),
        "doc_vectors":      getattr(clf, "doc_vecs_", None),
        "clf_texts":        getattr(pipeline, "clf_texts_", None),
        "silhouette_scores": getattr(clf, "silhouette_scores_", {}),
        "topic_model":      tm,
        "topic_info":       getattr(te, "topic_info", None),
        "results":          pipeline._to_dataframe(),
        "output_dir":       pipeline.output_dir,
    }
    with open(path, "wb") as fh:
        pickle.dump(bundle, fh, protocol=pickle.HIGHEST_PROTOCOL)

    for obj, enc in detached:                  # put the encoder back
        obj.embedding_model = enc

    mb = os.path.getsize(path) / 1e6
    print(f"saved -> {path}  ({mb:.1f} MB)")
    return path


save_pipeline(pipeline)

saved -> /home/naedatatz/Desktop/Notebook/output/runs/run_20260920_171507/sms_model.pkl  (10.1 MB)


'/home/naedatatz/Desktop/Notebook/output/runs/run_20260920_171507/sms_model.pkl'

## Step 22 - Load the model back

`load_pipeline` reads the pickle and reattaches the sentence encoder by name, so the
returned object is ready to use. Run this cell on its own in a fresh session to work
with a trained model without re-reading a single PDF.

`build_clf_text` composes a paper's text the way training composes it, and
`assign_clusters` places it into the clusters the model learned. Together they are the
payoff: a trained model that classifies papers it has never seen.

Two checks run at the bottom. The first re-assigns the stored training vectors, which
tests the pickled K-Means itself. The second re-encodes the training text from scratch,
which tests the whole reload path including the encoder. Both must be 100%, and the cell
raises if either is not. The last lines then classify an invented abstract to show the
loaded model working on new input.

In [22]:
def load_pipeline(path=MODEL_PKL):
    """Read the pickle back and reattach the sentence encoder."""
    with open(path, "rb") as fh:
        bundle = pickle.load(fh)

    encoder = SentenceTransformer(bundle["embedding_model_name"])
    tm = bundle.get("topic_model")
    if tm is not None and getattr(tm, "embedding_model", None) is None:
        tm.embedding_model = encoder
    bundle["encoder"] = encoder

    print(f"loaded {path}")
    print(f"  trained   : {bundle['created']}")
    print(f"  encoder   : {bundle['embedding_model_name']}")
    print(f"  K-Means   : k={bundle['kmeans_k'].get('k')} "
          f"(silhouette {bundle['kmeans_k'].get('silhouette')})")
    print(f"  papers    : {len(bundle['results'])}")
    return bundle


def build_clf_text(title, abstract, introduction="", methodology=""):
    """Compose a paper's text exactly the way training composes it.

    The pipeline clusters on title + abstract + the first 1500 characters of the
    introduction + the first 1000 of the methodology, cleaned. Passing only a
    title and abstract still works; it is simply less context than training saw.
    """
    raw = (f"{title}\n{abstract}\n"
           + str(introduction)[:1500] + str(methodology)[:1000])
    return clean_academic_text(raw) or raw


def assign_clusters(bundle, texts, already_clean=False):
    """Place new papers into the clusters learned during training.

    `texts` should come from `build_clf_text`. Pass `already_clean=True` only for
    text that has been through `clean_academic_text` once already - cleaning twice
    changes the wording and moves the embedding.
    """
    from sklearn.preprocessing import normalize
    prepared = ([str(t) for t in texts] if already_clean
                else [clean_academic_text(str(t)) or str(t) for t in texts])
    vecs = normalize(np.asarray(bundle["encoder"].encode(prepared, show_progress_bar=False)))
    ids  = bundle["kmeans_model"].predict(vecs)
    kw   = bundle["kmeans_keywords"]
    return pd.DataFrame({
        "cluster_id": [int(c) for c in ids],
        "cluster_keywords": [kw.get(int(c), "") for c in ids],
    })


# ---- load it and prove the round trip is faithful ---------------------------
model  = load_pipeline()
_saved = np.asarray(model["kmeans_labels"])

# check 1: the pickled K-Means reproduces its own assignment from the stored vectors
_from_vecs = model["kmeans_model"].predict(model["doc_vectors"])
_ok1 = (_from_vecs == _saved).mean()
print(f"\ncheck 1  stored vectors re-assigned : "
      f"{(_from_vecs == _saved).sum()}/{len(_saved)}  ({_ok1:.1%})")

# check 2: re-encoding the training text from scratch lands on the same clusters
_again = assign_clusters(model, model["clf_texts"], already_clean=True)["cluster_id"].to_numpy()
_ok2 = (_again == _saved).mean()
print(f"check 2  text re-encoded end to end : "
      f"{(_again == _saved).sum()}/{len(_saved)}  ({_ok2:.1%})")

assert _ok1 == 1.0, "the pickled K-Means does not reproduce its own labels"
assert _ok2 == 1.0, "re-encoding the training text does not reproduce the labels"
print("\nthe pickle reproduces the trained model exactly")

# ---- and this is how you classify a paper it has never seen ----------------
_new = build_clf_text(
    "Behaviour-driven development for microservice integration testing",
    "We present an approach that generates integration tests for microservice "
    "architectures directly from Gherkin scenarios, and evaluate it on three "
    "industrial systems.")
assign_clusters(model, [_new])

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 6917.64it/s]


loaded /home/naedatatz/Desktop/Notebook/output/runs/run_20260920_171507/sms_model.pkl
  trained   : 2026-09-20T17:17:55
  encoder   : all-MiniLM-L6-v2
  K-Means   : k=2 (silhouette 0.0669)
  papers    : 123

check 1  stored vectors re-assigned : 123/123  (100.0%)
check 2  text re-encoded end to end : 123/123  (100.0%)

the pickle reproduces the trained model exactly


,cluster_id,cluster_keywords
0,1,"systems, mobile, microservice, integration, apps"


## Step 23 - Evaluate and draw the dissertation figures

Runs the evaluation checks and then the figure set, both defined above.

Anything needing facet labels stays empty on an unsupervised run. To switch those on,
open `cluster_summary.csv`, decide which mapping category each cluster corresponds to,
and feed the corrected `gold_labels_template.csv` back as `gold_csv`.

In [23]:
def _measured_seconds_per_study():
    """Read the timing this training run actually recorded.

    Without a measured value the evaluator skips the timing comparison and the
    figure is drawn as an empty placeholder, which is what used to happen.
    """
    path = os.path.join(pipeline.output_dir, "timing.json")
    try:
        with open(path) as fh:
            rec = json.load(fh)
        secs = rec["total_seconds"] / max(1, rec["n_studies"])
        print(f"   measured {secs:.1f} s/study from this run")
        return secs
    except Exception as e:
        print(f"   (no timing.json: {e}; timing figure stays a placeholder)")
        return None


print("EVALUATING")
metrics = evaluate_results(
    results_df,
    output_dir=pipeline.output_dir,
    use_embeddings=True,
    gold_csv=None,            # point at your corrected gold file for a true F1
    seconds_per_study=_measured_seconds_per_study(),
)

print("\nFIGURES")
generate_all_figures(df=results_df, metrics=metrics, out_dir=FIGS_DIR)

EVALUATING
   measured 0.4 s/study from this run

📊 Running supervised cross-validated F1 evaluation …


Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 6617.75it/s]


📝 Gold template (pre-filled with predictions) -> /home/naedatatz/Desktop/Notebook/output/runs/run_20260920_171507/gold_labels_template.csv
   Review each row, CORRECT wrong labels, then pass it as gold_csv=.

SUPERVISED CROSS-VALIDATED F1 (trained on pipeline labels)
  research_type      macroF1=0.513  microF1=0.569  weightedF1=0.553  [LinearSVC, tfidf+embeddings, 5-fold]
  contribution_type  macroF1=0.484  microF1=0.488  weightedF1=0.486  [LogReg, tfidf+embeddings, 5-fold]
  evaluation_method  macroF1=0.490  microF1=0.496  weightedF1=0.494  [LogReg, tfidf+embeddings, 5-fold]

DISTRIBUTION BENCHMARK vs Binamungu & Maro (2023)
  research_type      benchmark=0.872  corr=0.757
  contribution_type  benchmark=0.933  corr=0.894
  evaluation_method  benchmark=0.874  corr=0.770

💾 All evaluation artefacts saved to /home/naedatatz/Desktop/Notebook/output/runs/run_20260920_171507

FIGURES

🎨 Generating figures -> /home/naedatatz/Desktop/Notebook/output/runs/run_20260920_171507/figs
   🖼  fig2_1_

## Step 24 - Classification reports

The pipeline stores per-class numbers as CSV and JSON but never produces a readable
classification report. This step writes one, for two different questions.

**A. Supervised.** Can a classifier relearn each facet from the text? Stratified 5-fold
cross-validation, logistic regression over the sentence embeddings, scored against the
pipeline's own facet labels. Read this as a measure of how **self-consistent** the label
set is, not how correct it is. The labels came from the zero-shot assigner, so a class it
applies coherently scores well and a class it applies at random scores near zero. An F1
of 0.000 means that category cannot be relearned from its own labels, which is the
clearest evidence available that the category is not real.

**B. Cluster agreement.** Do the unsupervised clusters recover the facets at all? Cluster
ids are arbitrary, so each cluster is matched one-to-one to the facet class it overlaps
most using the Hungarian algorithm, after which the report reads like any other.
Adjusted Rand Index and normalised mutual information are reported alongside, because
they need no matching and cannot be flattered by it.

Everything is written to `reports/` inside the run folder, as both text and CSV.

In [24]:
import os

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (adjusted_rand_score, classification_report,
                             confusion_matrix, normalized_mutual_info_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict

_FACETS = ["research_type", "contribution_type", "evaluation_method"]


def match_clusters(clusters, truth):
    """Map each cluster id to a facet class, one-to-one, by maximum overlap."""
    cl, cls = sorted(set(clusters)), sorted(set(truth))
    M = np.zeros((len(cl), len(cls)), dtype=int)
    for i, c in enumerate(cl):
        for j, t in enumerate(cls):
            M[i, j] = int(((clusters == c) & (truth == t)).sum())
    rows, cols = linear_sum_assignment(-M)
    mapping = {cl[r]: cls[c] for r, c in zip(rows, cols)}
    return np.array([mapping.get(c, cls[0]) for c in clusters]), mapping


def classification_reports(df, embeddings, clusters=None, out_dir=None, k_folds=5):
    out_dir = out_dir or os.path.join(RUN_DIR, "reports")
    os.makedirs(out_dir, exist_ok=True)
    summary = []

    for facet in _FACETS:
        if facet not in df.columns:
            continue
        y = df[facet].to_numpy()

        # ---- A. supervised cross-validation --------------------------------
        cv = StratifiedKFold(k_folds, shuffle=True, random_state=RANDOM_STATE)
        pred = cross_val_predict(LogisticRegression(max_iter=2000),
                                 embeddings, y, cv=cv)
        labels = sorted(set(y) | set(pred))
        text = classification_report(y, pred, labels=labels,
                                     zero_division=0, digits=3)
        header = (f"SUPERVISED {k_folds}-FOLD CROSS-VALIDATION - {facet}\n"
                  "logistic regression on sentence-BERT embeddings\n"
                  "scored against the pipeline's own zero-shot labels, so this\n"
                  "measures self-consistency, not correctness\n" + "=" * 70)
        with open(os.path.join(out_dir, f"classification_report_{facet}.txt"), "w") as fh:
            fh.write(header + "\n" + text + "\n")
        tbl = pd.DataFrame(classification_report(
            y, pred, labels=labels, zero_division=0, output_dict=True)).T
        tbl.round(4).to_csv(os.path.join(out_dir, f"classification_report_{facet}.csv"))
        pd.DataFrame(confusion_matrix(y, pred, labels=labels),
                     index=labels, columns=labels).to_csv(
            os.path.join(out_dir, f"confusion_{facet}.csv"))

        print("=" * 72)
        print(header)
        print(text)

        dead = [c for c in labels
                if tbl.loc[c, "f1-score"] == 0 and tbl.loc[c, "support"] > 0]
        if dead:
            print(f"  \u26a0\ufe0f  cannot be relearned at all: {', '.join(dead)}")

        row = {"facet": facet,
               "macro_f1": round(float(tbl.loc["macro avg", "f1-score"]), 4),
               "accuracy": round(float(tbl.loc["accuracy", "f1-score"]), 4),
               "classes_with_zero_f1": len(dead)}

        # ---- B. cluster agreement -----------------------------------------
        if clusters is not None:
            mapped, mapping = match_clusters(np.asarray(clusters), y)
            clabels = sorted(set(y) | set(mapped))
            ctext = classification_report(y, mapped, labels=clabels,
                                          zero_division=0, digits=3)
            ari = adjusted_rand_score(y, clusters)
            nmi = normalized_mutual_info_score(y, clusters)
            cheader = (f"CLUSTER AGREEMENT - clusters vs {facet}\n"
                       "clusters matched one-to-one to classes by maximum overlap\n"
                       f"ARI {ari:.4f}   NMI {nmi:.4f}\n"
                       f"mapping: {mapping}\n" + "=" * 70)
            with open(os.path.join(out_dir, f"cluster_agreement_{facet}.txt"), "w") as fh:
                fh.write(cheader + "\n" + ctext + "\n")
            print(cheader)
            print(ctext)
            row.update({"cluster_ari": round(float(ari), 4),
                        "cluster_nmi": round(float(nmi), 4)})

        summary.append(row)

    out = pd.DataFrame(summary)
    out.to_csv(os.path.join(out_dir, "classification_summary.csv"), index=False)
    print(f"\n\u2705 reports \u2192 {out_dir}")
    return out


_reports = classification_reports(
    results_df,
    pipeline.classifier.doc_vecs_,
    clusters=results_df["cluster_id"].to_numpy() if "cluster_id" in results_df else None,
)
_reports

SUPERVISED 5-FOLD CROSS-VALIDATION - research_type
logistic regression on sentence-BERT embeddings
scored against the pipeline's own zero-shot labels, so this
measures self-consistency, not correctness
                      precision    recall  f1-score   support

 Evaluation Research      0.000     0.000     0.000        14
   Experience Papers      0.500     0.133     0.211        15
      Opinion Papers      0.250     0.056     0.091        18
Philosophical Papers      0.515     0.708     0.596        24
   Solution Proposal      0.417     0.227     0.294        22
 Validation Research      0.343     0.800     0.480        30

            accuracy                          0.398       123
           macro avg      0.337     0.321     0.279       123
        weighted avg      0.356     0.398     0.325       123

  ⚠️  cannot be relearned at all: Evaluation Research
CLUSTER AGREEMENT - clusters vs research_type
clusters matched one-to-one to classes by maximum overlap
ARI 0.0136   NMI 

,facet,macro_f1,accuracy,classes_with_zero_f1,cluster_ari,cluster_nmi
0,research_type,0.2787,0.3984,1,0.0136,0.0525
1,contribution_type,0.3851,0.4390,1,0.0169,0.0371
2,evaluation_method,0.2908,0.3821,1,0.0460,0.0629


## Step 25 - Cluster figures

Three figures about the clustering itself.

**The map** puts every paper in a 2-D projection coloured by its K-Means cluster, with
each cluster's centre marked and its keywords in the legend.

**The silhouette profile** gives one bar per paper. Bars left of zero are papers sitting
closer to the other cluster than to their own, which is the honest way to see whether a
split is real.

**The space comparison** shows that the silhouette is not a fixed property of the corpus.
It sweeps the UMAP neighbourhood size and plots the same clusters scored two ways: in the
projected space, and back in the real embedding space. The two lines diverge completely.
Squeezing the projection inflates the score while the clusters themselves get worse, so
the number only means something alongside the space it was measured in. There is no
threshold to hit here; the figure exists to make the dependence explicit.

In [25]:
# This cell imports everything it uses, so it runs on its own even after a
# kernel restart. The names are also imported in Step 1; repeating them is free.
import os

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker
from matplotlib.lines import Line2D
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
from umap import UMAP

SURFACE, INK, INK_2 = "#fcfcfb", "#0b0b0b", "#52514e"
MUTED, GRID = "#8a8984", "#e6e5e1"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]     # validated categorical slots

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "font.size": 11, "text.color": INK,
    "axes.labelcolor": INK_2, "axes.edgecolor": GRID,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "grid.color": GRID, "grid.linewidth": 0.8,
})


def _header(fig, title, *subs, top=0.955, gap=0.049):
    fig.text(0.055, top, title, fontsize=17, fontweight="bold", color=INK, va="top")
    for i, line in enumerate(subs):
        fig.text(0.055, top - gap - i * 0.036, line, fontsize=10.5,
                 color=INK_2, va="top")


def cluster_figures(df, embeddings, keywords, out_dir=FIGS_DIR):
    labels = df["cluster_id"].to_numpy()
    ids = sorted(set(labels))
    sil = silhouette_score(embeddings, labels)

    # --- 1. the map ---------------------------------------------------------
    XY = UMAP(n_components=2, n_neighbors=15, min_dist=0.12, metric="cosine",
              random_state=RANDOM_STATE).fit_transform(embeddings)
    fig, ax = plt.subplots(figsize=(10.5, 7.6))
    fig.subplots_adjust(top=0.84, bottom=0.17)
    handles = []
    for i, cid in enumerate(ids):
        m = labels == cid
        ax.scatter(XY[m, 0], XY[m, 1], s=76, c=SERIES[i % 3], edgecolors=SURFACE,
                   linewidths=1.6, alpha=0.9, zorder=3)
        ax.scatter([XY[m, 0].mean()], [XY[m, 1].mean()], marker="X", s=260,
                   c=SERIES[i % 3], edgecolors=SURFACE, linewidths=2.6, zorder=6)
        words = ", ".join(str(keywords.get(cid, "")).split(", ")[:4])
        handles.append(Line2D([], [], marker="o", ls="", ms=10, mfc=SERIES[i % 3],
                              mec=SURFACE, mew=1.4,
                              label=f"Cluster {cid}  ({int(m.sum())} papers)   {words}"))
    _header(fig, f"How the {len(df)} papers cluster",
            "K-Means on sentence-BERT embeddings, drawn in a 2-D UMAP projection. "
            "X marks each cluster centre.",
            f"Mean silhouette in the embedding space is {sil:.3f}.")
    ax.set_xlabel("UMAP dimension 1"); ax.set_ylabel("UMAP dimension 2")
    ax.set_xticklabels([]); ax.set_yticklabels([])
    ax.legend(handles=handles, frameon=False, fontsize=10.5, labelcolor=INK_2,
              loc="upper center", bbox_to_anchor=(0.5, -0.075))
    fig.savefig(os.path.join(out_dir, "fig_kmeans_map.png"), dpi=200)
    plt.show()

    # --- 2. the silhouette profile -----------------------------------------
    vals = silhouette_samples(embeddings, labels)
    fig, ax = plt.subplots(figsize=(10.5, 6.8))
    fig.subplots_adjust(top=0.84, bottom=0.11, left=0.22)
    y = 0
    for i, cid in enumerate(ids):
        v = np.sort(vals[labels == cid])
        ax.barh(np.arange(y, y + len(v)), v, height=1.0, color=SERIES[i % 3])
        words = ", ".join(str(keywords.get(cid, "")).split(", ")[:3])
        ax.text(-0.052, y + len(v) / 2, f"Cluster {cid}\n{words}", ha="right",
                va="center", fontsize=10, color=INK_2)
        y += len(v) + 8
    ax.axvline(sil, color=INK, lw=2, ls="--", zorder=4)
    ax.annotate(f"mean {sil:.3f}", (sil, y * 0.985), fontsize=11, fontweight="bold",
                color=INK, ha="left", va="top", xytext=(7, 0),
                textcoords="offset points")
    ax.axvline(0, color=MUTED, lw=1)
    _header(fig, "Silhouette of every individual paper",
            "Bars left of zero are papers sitting closer to the other cluster than to their own.")
    ax.set_xlabel("Silhouette coefficient"); ax.set_yticks([])
    ax.grid(axis="y", visible=False)
    fig.savefig(os.path.join(out_dir, "fig_kmeans_silhouette.png"), dpi=200)
    plt.show()

    # --- 3. the score depends on the space ---------------------------------
    rows = []
    for nn in (2, 3, 4, 5, 8, 11, 15, 25, 40, 60):
        X = UMAP(n_components=3, n_neighbors=nn, min_dist=0.0, metric="cosine",
                 random_state=RANDOM_STATE).fit_transform(embeddings)
        best = None
        for k in range(2, 13):
            lab = KMeans(k, random_state=RANDOM_STATE, n_init=10).fit_predict(X)
            s = float(silhouette_score(X, lab))
            if best is None or s > best[1]:
                best = (k, s, lab)
        rows.append(dict(n_neighbors=nn, in_umap=best[1],
                         in_embedding=float(silhouette_score(embeddings, best[2]))))
    d = pd.DataFrame(rows)
    d.to_csv(os.path.join(out_dir, "silhouette_by_space.csv"), index=False)

    fig, ax = plt.subplots(figsize=(10.5, 6.8))
    fig.subplots_adjust(top=0.80, bottom=0.21)
    ax.plot(d.n_neighbors, d.in_umap, "-o", color=SERIES[0], lw=2, ms=9,
            mec=SURFACE, mew=1.6, label="measured in the UMAP projection")
    ax.plot(d.n_neighbors, d.in_embedding, "-o", color=SERIES[1], lw=2, ms=9,
            mec=SURFACE, mew=1.6,
            label="the very same clusters, measured in the real embedding space")
    ax.set_xscale("log"); ax.set_xticks(d.n_neighbors); ax.set_xticks([], minor=True)
    ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
    _header(fig, "The silhouette is set by the projection, not by the papers",
            "Shrinking the UMAP neighbourhood inflates the score while the clusters themselves get worse.")
    ax.set_xlabel("UMAP n_neighbors  (smaller = more aggressive projection)")
    ax.set_ylabel("Silhouette coefficient")
    ax.legend(frameon=False, fontsize=10.5, labelcolor=INK_2,
              loc="upper center", bbox_to_anchor=(0.5, -0.115))
    fig.savefig(os.path.join(out_dir, "fig_silhouette_spaces.png"), dpi=200)
    plt.show()
    return d


_emb = pipeline.classifier.doc_vecs_
_space_table = cluster_figures(results_df, _emb, pipeline.classifier.cluster_keywords_)
_space_table

,n_neighbors,in_umap,in_embedding
0,2,0.785127,0.013610
1,3,0.574778,0.023988
2,4,0.829311,0.046256
3,5,0.471309,0.023230
4,8,0.407484,0.014131
5,11,0.397770,0.047546
6,15,0.390175,0.046467
7,25,0.349575,0.039057
8,40,0.346882,0.046982
9,60,0.328208,0.048525


## Step 26 - Training history

There are no epochs in this pipeline, so there is no loss curve to plot. K-Means,
HDBSCAN and the zero-shot facet assignment all fit in one pass. What the pipeline does
have is a fitting process with a genuine history, and this figure shows the four parts
of it that are worth reporting.

**A. Convergence.** Lloyd's algorithm reassigns points and recomputes centres until
nothing moves. The curve is the within-cluster sum of squares after each iteration.

**B. Model selection.** The silhouette at every candidate k, with the chosen value
marked. This is the search that picked k, so it belongs in the methods chapter.

**C. The elbow.** Within-cluster sum of squares against k. A sharp bend would indicate
a natural number of clusters; a smooth decline says the corpus has no obvious break.

**D. Learning curve.** Cross-validated macro F1 against the number of labelled papers,
one line per facet, with the shaded band showing variation across folds. This is the
panel that answers "would more labelled data help", which matters because hand-labelling
is the obvious next step.

In [26]:
import os

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import silhouette_score
from sklearn.model_selection import StratifiedKFold, learning_curve

_FACETS = ["research_type", "contribution_type", "evaluation_method"]


def kmeans_convergence(X, k, max_iter=25, seed=RANDOM_STATE):
    """Within-cluster sum of squares after each Lloyd iteration.

    scikit-learn does not expose the per-iteration history, so the fit is
    repeated with a rising max_iter from one fixed starting position. The
    starting centres are shared, which makes the curve a true trajectory
    rather than a set of unrelated fits.
    """
    start = KMeans(k, n_init=10, random_state=seed, max_iter=1).fit(X).cluster_centers_
    return [KMeans(k, init=start, n_init=1, max_iter=i, tol=0.0,
                   algorithm="lloyd").fit(X).inertia_
            for i in range(1, max_iter + 1)]


def training_history(df, embeddings, out_dir=FIGS_DIR, k_max=15):
    k_range = range(2, k_max + 1)
    inertia, sil = [], []
    for k in k_range:
        km = KMeans(k, random_state=RANDOM_STATE, n_init=10).fit(embeddings)
        inertia.append(km.inertia_)
        sil.append(silhouette_score(embeddings, km.labels_))
    chosen_k = list(k_range)[int(np.argmax(sil))]
    conv = kmeans_convergence(embeddings, chosen_k)
    settled = next((i + 1 for i in range(1, len(conv))
                    if abs(conv[i] - conv[i - 1]) < 1e-6), len(conv))

    fig, axes = plt.subplots(2, 2, figsize=(13, 10.2))
    fig.subplots_adjust(top=0.80, bottom=0.07, hspace=0.40, wspace=0.26)

    ax = axes[0, 0]
    ax.plot(range(1, len(conv) + 1), conv, "-o", color=SERIES[0], lw=2, ms=7,
            mec=SURFACE, mew=1.4)
    ax.axvline(settled, color=INK, ls="--", lw=1.4)
    ax.annotate(f"settled at iteration {settled}", (settled, max(conv)),
                xytext=(8, -4), textcoords="offset points", fontsize=10,
                fontweight="bold", color=INK)
    ax.set_title(f"A. K-Means convergence (k={chosen_k})", loc="left", fontweight="bold")
    ax.set_xlabel("Lloyd iteration"); ax.set_ylabel("Within-cluster sum of squares")

    ax = axes[0, 1]
    ax.plot(list(k_range), sil, "-o", color=SERIES[1], lw=2, ms=7, mec=SURFACE, mew=1.4)
    ax.plot([chosen_k], [max(sil)], "o", ms=13, color=SERIES[1], mec=INK, mew=2)
    ax.annotate(f"chosen k={chosen_k}", (chosen_k, max(sil)), xytext=(8, 6),
                textcoords="offset points", fontsize=10, fontweight="bold", color=INK)
    ax.set_title("B. Model selection by silhouette", loc="left", fontweight="bold")
    ax.set_xlabel("Number of clusters (k)"); ax.set_ylabel("Silhouette")

    ax = axes[1, 0]
    ax.plot(list(k_range), inertia, "-o", color=SERIES[2], lw=2, ms=7,
            mec=SURFACE, mew=1.4)
    ax.set_title("C. Elbow curve", loc="left", fontweight="bold")
    ax.set_xlabel("Number of clusters (k)"); ax.set_ylabel("Within-cluster sum of squares")

    ax = axes[1, 1]
    rows = []
    for i, facet in enumerate(_FACETS):
        if facet not in df.columns:
            continue
        n_tr, _tr, te = learning_curve(
            LogisticRegression(max_iter=2000), embeddings, df[facet].to_numpy(),
            train_sizes=np.linspace(0.2, 1.0, 6),
            cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
            scoring="f1_macro", n_jobs=1)
        ax.plot(n_tr, te.mean(1), "-o", color=SERIES[i], lw=2, ms=7, mec=SURFACE,
                mew=1.4, label=facet.replace("_", " "))
        ax.fill_between(n_tr, te.mean(1) - te.std(1), te.mean(1) + te.std(1),
                        color=SERIES[i], alpha=0.12, lw=0)
        rows.append(pd.DataFrame({"facet": facet, "train_size": n_tr,
                                  "macro_f1": te.mean(1).round(4),
                                  "std": te.std(1).round(4)}))
    ax.set_title("D. Learning curve (5-fold macro F1)", loc="left", fontweight="bold")
    ax.set_xlabel("Labelled papers used for training"); ax.set_ylabel("Macro F1")
    ax.legend(frameon=False, fontsize=9.5, labelcolor=INK_2)

    fig.text(0.055, 0.972, "Training history", fontsize=17, fontweight="bold",
             color=INK, va="top")
    fig.text(0.055, 0.932,
             "This pipeline has no epochs, so there is no loss curve. These are the four "
             "histories it does have:", fontsize=10.5, color=INK_2, va="top")
    fig.text(0.055, 0.905,
             "how the clustering settles, how k was chosen, where the elbow sits, and how "
             "performance responds to more labelled data.",
             fontsize=10.5, color=INK_2, va="top")

    fig.savefig(os.path.join(out_dir, "fig_training_history.png"), dpi=200)
    plt.show()

    hist = pd.DataFrame({"k": list(k_range), "silhouette": np.round(sil, 4),
                         "inertia": np.round(inertia, 4)})
    hist.to_csv(os.path.join(out_dir, "training_history_k.csv"), index=False)
    if rows:
        pd.concat(rows).to_csv(os.path.join(out_dir, "training_history_learning.csv"),
                               index=False)
    print(f"   converged at iteration {settled}; silhouette chose k={chosen_k}")
    return hist


_history = training_history(results_df, pipeline.classifier.doc_vecs_)
_history

   converged at iteration 3; silhouette chose k=2


,k,silhouette,inertia
0,2,0.0669,54.2207
1,3,0.0383,51.7517
2,4,0.0375,50.1245
3,5,0.0356,49.2910
4,6,0.0391,47.8601
5,7,0.0228,46.8017
6,8,0.0487,46.2307
7,9,0.0502,45.0525
8,10,0.0385,44.7838
9,11,0.0377,43.0399


## Step 27 - Where everything ended up

Lists what this run produced and where it sits.

The layout separates the two phases. `output/dataset.csv` is the corpus, written once by
Part A and shared by every run. Each modelling run writes to its own
`output/runs/run_<timestamp>/` folder, so nothing is ever overwritten and two runs can be
compared side by side. `output/latest` is a symlink to the most recent one, so the
write-up can cite a stable path.

To go back to overwriting a single folder, set `NEW_RUN_FOLDER = False` in step 2.

In [27]:
print(f"DATASET (shared)   : {DATASET_CSV}"
      f"  ({os.path.getsize(DATASET_CSV)/1e6:.1f} MB)")
print(f"THIS RUN           : {RUN_DIR}")
print(f"LATEST points here : {os.path.join(OUTPUT_DIR, 'latest')}\n")

for root, _dirs, files in sorted(os.walk(RUN_DIR)):
    rel = os.path.relpath(root, RUN_DIR)
    if files:
        print(f"  {'.' if rel == '.' else rel}/")
        for f in sorted(files):
            size = os.path.getsize(os.path.join(root, f)) / 1e3
            print(f"      {f:<40} {size:>10.1f} KB")

_runs = sorted(os.listdir(os.path.join(OUTPUT_DIR, "runs"))) \
    if os.path.isdir(os.path.join(OUTPUT_DIR, "runs")) else []
print(f"\n{len(_runs)} run(s) kept: " + ", ".join(_runs[-5:]))

print("\nReload the trained model in any session with:")
print(f"    model = load_pipeline({MODEL_PKL!r})")
print("    assign_clusters(model, [build_clf_text(title, abstract)])")
print("\nRe-run only the modelling (Part B) without re-reading any PDF:")
print("    dataset = pd.read_csv(DATASET_CSV)")

DATASET (shared)   : /home/naedatatz/Desktop/Notebook/output/dataset.csv  (9.4 MB)
THIS RUN           : /home/naedatatz/Desktop/Notebook/output/runs/run_20260920_171507
LATEST points here : /home/naedatatz/Desktop/Notebook/output/latest

  ./
      ablation_contribution_type.csv                  0.0 KB
      ablation_evaluation_method.csv                  0.0 KB
      ablation_research_type.csv                      0.0 KB
      benchmark_contribution_type.csv                 0.1 KB
      benchmark_evaluation_method.csv                 0.2 KB
      benchmark_research_type.csv                     0.2 KB
      cluster_summary.csv                             0.1 KB
      confusion_contribution_type.csv                 0.2 KB
      confusion_evaluation_method.csv                 0.2 KB
      confusion_research_type.csv                     0.3 KB
      evaluation_metrics.json                         9.2 KB
      fig_kmeans_vs_hdbscan.png                      24.6 KB
      fig_method_comparis

## Step 28 - Inspect one paper

Everything above works on 123 papers at once. This step opens a single one and prints it
exactly as the pipeline sees it, which is the fastest way to check whether an odd result
is a modelling problem or an extraction problem.

It shows the header fields, then each body section with its length and opening lines, then
whatever the model decided about the paper. Empty sections are called out, because a
missing methodology is the usual reason an evaluation-method facet looks wrong.

Call it with any identifier: `show_paper("P0007")`, a file name fragment like
`show_paper("Cucumber")`, or nothing at all for the first paper in the dataset.

In [28]:
def show_paper(which=None, data=None, results=None, chars=600):
    """Print one paper as the pipeline sees it: fields, sections, model output."""
    data = data if data is not None else dataset
    results = results if results is not None else globals().get("results_df")

    if which is None:
        row = data.iloc[0]
    elif isinstance(which, int):
        row = data.iloc[which]
    else:
        key = str(which).lower()
        hit = data[data["paper_id"].str.lower() == key]
        if hit.empty:
            hit = data[data["source_file"].str.lower().str.contains(key, regex=False)
                       | data["title"].fillna("").str.lower().str.contains(key, regex=False)]
        if hit.empty:
            print(f"no paper matches {which!r}")
            return None
        row = hit.iloc[0]

    def _s(k):
        v = row.get(k, "")
        return "" if v is None or (isinstance(v, float) and pd.isna(v)) else str(v)

    line = "=" * 78
    print(line)
    print(f"{_s('paper_id')}   {_s('source_file')}")
    print(line)
    print(f"  title      : {_s('title')[:200] or '(none)'}")
    print(f"  year       : {_s('year') or '(not found)'}")
    print(f"  keywords   : {_s('keywords')[:200] or '(none found)'}")
    print(f"  length     : {int(row.get('word_count') or 0):,} words, "
          f"{int(row.get('n_sections') or 0)}/5 body sections")
    print(f"  extracted  : title by {_s('title_source')}, "
          f"abstract by {_s('abstract_strategy')}, text by {_s('extraction_method')}")

    print("\n  ABSTRACT")
    abstract = _s("abstract")
    print("    " + (abstract[:chars].replace(chr(10), " ") + ("\u2026" if len(abstract) > chars else "")
                    if abstract else "(empty)"))

    print("\n  BODY SECTIONS")
    empty = []
    for sec in SECTION_COLUMNS:
        body = _s(sec).strip()
        if not body:
            empty.append(sec)
            print(f"    {sec:<14} (empty)")
            continue
        preview = body[:chars].replace(chr(10), " ")
        print(f"    {sec:<14} {len(body.split()):>6,} words")
        print(f"        {preview}{'\u2026' if len(body) > chars else ''}")
    if empty:
        print(f"\n    \u26a0\ufe0f  no heading was detected for: {', '.join(empty)}")

    full = _s("full_text")
    print(f"\n  FULL TEXT  {len(full):,} characters, first 300:")
    print("    " + full[:300].replace(chr(10), " ") + "\u2026")

    if results is not None and "paper_id" in results.columns:
        r = results[results.paper_id == row["paper_id"]]
        if not r.empty:
            r = r.iloc[0]
            print("\n  WHAT THE MODEL DECIDED")
            if "cluster_label" in r:
                print(f"    cluster    : {str(r['cluster_label'])[:70]}")
            if "bertopic_theme" in r:
                print(f"    theme      : {str(r['bertopic_theme'])[:70]} "
                      f"(confidence {float(r.get('bertopic_confidence', 0)):.2f})")
            for facet in ("research_type", "contribution_type", "evaluation_method"):
                if facet in r:
                    flag = " \u26a0 low margin" if r.get(f"{facet}_needs_review") else ""
                    print(f"    {facet:<18}: {r[facet]} "
                          f"(margin {float(r.get(f'{facet}_margin', 0)):.2f}){flag}")
    print(line)
    return row


# change the argument to inspect any other paper
_example = show_paper()

P0001   A Behavior-Based Ontology for Supporting Automated Assessment of Interactive Systems.pdf
  title      : A Behavior-Based Ontology for Supporting Automated Assessment of Interactive Systems
  year       : 2017.0
  keywords   : Automated Requirements Assessment; Behavior-Driven Development; Ontological Modeling; Testing of Interactive Systems. I
  length     : 5,697 words, 5/5 body sections
  extracted  : title by typography, abstract by heading, text by native

  ABSTRACT
    Nowadays many software development frameworks implement Behavior-Driven Development (BDD) as a mean of automating the test of interactive systems under construction. Automated testing helps to simulate user's action on the User Interface and therefore check if the system behaves properly and in accordance to Scenarios that describe functional requirements. However, most of tools supporting BDD requires that tests should be written using low-level events and components that only exist when the system is alre